Step 1a: Build DDR gene universe with pathway annotations

In [1]:

import pandas as pd
import numpy as np
import os

os.makedirs('/mnt/results', exist_ok=True)
np.random.seed(42)

# ── Curated DDR gene universe ──────────────────────────────────────────────
ddr_genes = {
    # Homologous Recombination
    "BRCA1": "HR", "BRCA2": "HR", "PALB2": "HR",
    "RAD51": "HR", "RAD51C": "HR", "RAD51D": "HR",
    "BRIP1": "HR", "BARD1": "HR", "RAD52": "HR",
    "RAD54L": "HR", "RAD54B": "HR", "RBBP8": "HR",
    "NBN": "HR", "MRE11": "HR", "RAD50": "HR",
    "RPA1": "HR", "RPA2": "HR", "RPA3": "HR",
    "SHFM1": "HR", "SLX4": "HR", "GEN1": "HR",
    # Non-Homologous End Joining
    "LIG4": "NHEJ", "XRCC4": "NHEJ", "XRCC5": "NHEJ",
    "XRCC6": "NHEJ", "PRKDC": "NHEJ", "NHEJ1": "NHEJ",
    "DNTT": "NHEJ", "ARTEMIS": "NHEJ",
    # Mismatch Repair
    "MLH1": "MMR", "MSH2": "MMR", "MSH6": "MMR",
    "PMS2": "MMR", "MSH3": "MMR", "MLH3": "MMR",
    "PMS1": "MMR", "PCNA": "MMR",
    # Base Excision Repair
    "OGG1": "BER", "MUTYH": "BER", "APEX1": "BER",
    "XRCC1": "BER", "LIG3": "BER", "PARP1": "BER",
    "PARP2": "BER", "POLB": "BER", "NEIL1": "BER",
    "NEIL2": "BER", "NEIL3": "BER", "UNG": "BER",
    "SMUG1": "BER", "TDG": "BER", "MBD4": "BER",
    # Nucleotide Excision Repair
    "XPA": "NER", "XPC": "NER", "ERCC1": "NER",
    "ERCC2": "NER", "ERCC3": "NER", "ERCC4": "NER",
    "ERCC5": "NER", "ERCC6": "NER", "ERCC8": "NER",
    "DDB1": "NER", "DDB2": "NER", "RAD23A": "NER",
    "RAD23B": "NER", "CETN2": "NER",
    # ATM/ATR Checkpoint
    "ATM": "ATM_ATR", "ATR": "ATM_ATR", "CHEK1": "ATM_ATR",
    "CHEK2": "ATM_ATR", "TP53": "ATM_ATR", "MDM2": "ATM_ATR",
    "H2AX": "ATM_ATR", "TP53BP1": "ATM_ATR", "BRCA1": "ATM_ATR",  # dual
    "RNF8": "ATM_ATR", "RNF168": "ATM_ATR", "UIMC1": "ATM_ATR",
    "PAXIP1": "ATM_ATR", "MDC1": "ATM_ATR", "TOPBP1": "ATM_ATR",
    "ATRIP": "ATM_ATR", "RAD17": "ATM_ATR", "RAD9A": "ATM_ATR",
    "HUS1": "ATM_ATR", "RAD1": "ATM_ATR",
    # Fanconi Anemia
    "FANCA": "FA", "FANCC": "FA", "FANCD2": "FA",
    "FANCG": "FA", "FANCI": "FA", "FANCB": "FA",
    "FANCD1": "FA", "FANCE": "FA", "FANCF": "FA",
    "FANCJ": "FA", "FANCL": "FA", "FANCM": "FA",
    "FANCN": "FA", "FANCO": "FA", "FANCP": "FA",
    "FANCQ": "FA", "FANCR": "FA", "FANCS": "FA",
    "FANCT": "FA", "FANCU": "FA", "FANCV": "FA",
    "FANCW": "FA",
}

# Build DataFrame (deduplicate: BRCA1 appears in HR and ATM_ATR — keep HR as primary)
rows = []
seen = set()
for gene, pathway in ddr_genes.items():
    if gene not in seen:
        rows.append({"gene_symbol": gene, "pathway": pathway})
        seen.add(gene)

ddr_df = pd.DataFrame(rows)
print(f"Total unique DDR genes curated: {len(ddr_df)}")
print(ddr_df['pathway'].value_counts())
ddr_df.to_csv('/mnt/results/ddr_gene_universe.csv', index=False)
print("\nSaved: ddr_gene_universe.csv")


Total unique DDR genes curated: 107
pathway
FA         22
HR         20
ATM_ATR    20
BER        15
NER        14
NHEJ        8
MMR         8
Name: count, dtype: int64

Saved: ddr_gene_universe.csv


Step 1b: Query GWAS Catalog for AD-significant SNPs (p < 5e-8)

In [3]:

import requests
import pandas as pd
import time

# Source: https://www.ebi.ac.uk/gwas/rest/api
# GWAS Catalog REST API — query for Alzheimer's disease associations
# EFO_0000249 = Alzheimer's disease

base_url = "https://www.ebi.ac.uk/gwas/rest/api"

def fetch_gwas_associations(trait_efo="EFO_0000249", p_threshold=5e-8, max_results=1000):
    """Fetch GWAS associations for a given EFO trait ID."""
    all_assocs = []
    page = 0
    page_size = 200
    
    while len(all_assocs) < max_results:
        url = f"{base_url}/efoTraits/{trait_efo}/associations"
        params = {
            "projection": "associationByEfoTrait",
            "page": page,
            "size": page_size
        }
        resp = requests.get(url, params=params, timeout=60)
        if resp.status_code != 200:
            print(f"Error {resp.status_code} at page {page}")
            break
        
        data = resp.json()
        embedded = data.get("_embedded", {})
        assocs = embedded.get("associations", [])
        if not assocs:
            break
        
        all_assocs.extend(assocs)
        
        # Check pagination
        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)
        print(f"  Page {page+1}/{total_pages}, fetched {len(all_assocs)} associations so far...")
        
        if page + 1 >= total_pages:
            break
        page += 1
        time.sleep(0.3)
    
    return all_assocs

print("Querying GWAS Catalog for Alzheimer's disease (EFO_0000249)...")
assocs = fetch_gwas_associations()
print(f"\nTotal associations fetched: {len(assocs)}")


Querying GWAS Catalog for Alzheimer's disease (EFO_0000249)...
Error 404 at page 0

Total associations fetched: 0


Step 1b: Query GWAS Catalog API with correct endpoint structure

In [5]:

import requests
import pandas as pd
import time

# Source: https://www.ebi.ac.uk/gwas/rest/api
# Test the correct endpoint structure first
base_url = "https://www.ebi.ac.uk/gwas/rest/api"

# Check available endpoints
resp = requests.get(base_url, timeout=30)
print("Root API status:", resp.status_code)
if resp.status_code == 200:
    data = resp.json()
    links = data.get("_links", {})
    print("Available endpoints:", list(links.keys())[:20])


Root API status: 200
Available endpoints: ['singleNucleotidePolymorphisms', 'studies', 'associations', 'efoTraits', 'unpublishedStudies', 'profile']


Step 1b: Query GWAS Catalog associations endpoint for Alzheimer's disease

In [7]:

import requests
import pandas as pd
import time

# Source: https://www.ebi.ac.uk/gwas/rest/api
base_url = "https://www.ebi.ac.uk/gwas/rest/api"

# First, find the EFO trait ID for Alzheimer's disease
resp = requests.get(f"{base_url}/efoTraits/search/findByEfoUri",
                    params={"uri": "http://www.ebi.ac.uk/efo/EFO_0000249"},
                    timeout=30)
print("EFO trait lookup status:", resp.status_code)
if resp.status_code == 200:
    print(resp.json())


EFO trait lookup status: 200
{'_embedded': {'efoTraits': []}, '_links': {'self': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByEfoUri?uri=http%3A%2F%2Fwww.ebi.ac.uk%2Fefo%2FEFO_0000249'}}, 'page': {'size': 20, 'totalElements': 0, 'totalPages': 0, 'number': 0}}


Step 1b: Try correct GWAS Catalog search endpoints for AD

In [9]:

import requests
import pandas as pd
import time

# Source: https://www.ebi.ac.uk/gwas/rest/api
base_url = "https://www.ebi.ac.uk/gwas/rest/api"

# Try searching EFO traits by label
resp = requests.get(f"{base_url}/efoTraits/search/findByEfoTrait",
                    params={"trait": "Alzheimer's disease"},
                    timeout=30)
print("EFO trait search by label status:", resp.status_code)
if resp.status_code == 200:
    data = resp.json()
    traits = data.get("_embedded", {}).get("efoTraits", [])
    print(f"Found {len(traits)} traits:")
    for t in traits[:5]:
        print(f"  shortForm={t.get('shortForm')}, trait={t.get('trait')}")


EFO trait search by label status: 200
Found 0 traits:


Probe GWAS Catalog API search endpoints and list available search methods

In [11]:

import requests

base_url = "https://www.ebi.ac.uk/gwas/rest/api"

# Check what search methods are available on efoTraits
resp = requests.get(f"{base_url}/efoTraits/search", timeout=30)
print("efoTraits/search status:", resp.status_code)
if resp.status_code == 200:
    data = resp.json()
    links = data.get("_links", {})
    print("Available search methods:")
    for k, v in links.items():
        print(f"  {k}: {v.get('href','')}")


efoTraits/search status: 200
Available search methods:
  findByEfoTrait: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByEfoTrait{?trait,page,size,sort,projection}
  findByEfoUri: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByEfoUri{?uri,page,size,sort,projection}
  findByShortForm: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByShortForm{?shortForm,projection}
  findByPubmedId: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByPubmedId{?pubmedId,page,size,sort,projection}
  self: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search


Query GWAS Catalog using shortForm and direct associations endpoint

In [13]:

import requests
import pandas as pd
import time

# Source: https://www.ebi.ac.uk/gwas/rest/api
base_url = "https://www.ebi.ac.uk/gwas/rest/api"

# Try findByShortForm with EFO_0000249
resp = requests.get(f"{base_url}/efoTraits/search/findByShortForm",
                    params={"shortForm": "EFO_0000249"},
                    timeout=30)
print("findByShortForm EFO_0000249 status:", resp.status_code)
if resp.status_code == 200:
    data = resp.json()
    print(data)


findByShortForm EFO_0000249 status: 404


Query GWAS Catalog for Alzheimer's disease associations with p-value < 5e-8

In [15]:

import requests
import pandas as pd
import time

# Source: https://www.ebi.ac.uk/gwas/rest/api

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Step 1: Search for Alzheimer's disease EFO trait
print("Searching for Alzheimer's disease EFO traits...")
trait_url = f"{BASE_URL}/efoTraits/search/findByEfoUri"
# Known EFO ID for Alzheimer's disease
efo_url = "http://www.ebi.ac.uk/efo/EFO_0000249"

# Also search by label
search_url = f"{BASE_URL}/efoTraits/search/findByEfoTrait"
resp = requests.get(search_url, params={"efoTrait": "Alzheimer's disease", "size": 20})
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    print(data)


Searching for Alzheimer's disease EFO traits...
Status: 200
{'_embedded': {'efoTraits': []}, '_links': {'self': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByEfoTrait?efoTrait=Alzheimer%27s+disease&size=20'}}, 'page': {'size': 20, 'totalElements': 0, 'totalPages': 0, 'number': 0}}


Try different search approaches for Alzheimer's disease traits

In [17]:

import requests

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Try searching associations directly by EFO ID
# EFO_0000249 is Alzheimer's disease
print("Fetching associations via EFO trait ID EFO_0000249...")
# Source: https://www.ebi.ac.uk/gwas/efotraits/EFO_0000249
url = f"{BASE_URL}/efoTraits/EFO_0000249/associations"
resp = requests.get(url, params={"size": 1, "page": 0})
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    page_info = data.get('page', {})
    print(f"Total associations: {page_info.get('totalElements', 'N/A')}")
    print(f"Total pages: {page_info.get('totalPages', 'N/A')}")


Fetching associations via EFO trait ID EFO_0000249...
Status: 404


Explore GWAS Catalog API endpoints for traits

In [19]:

import requests

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Try the studies endpoint with disease trait
print("Trying studies search by disease trait...")
# Source: https://www.ebi.ac.uk/gwas/rest/api
url = f"{BASE_URL}/studies/search/findByDiseaseTrait"
resp = requests.get(url, params={"diseaseTrait": "Alzheimer", "size": 5})
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    page_info = data.get('page', {})
    print(f"Total studies: {page_info.get('totalElements', 'N/A')}")
    studies = data.get('_embedded', {}).get('studies', [])
    for s in studies[:3]:
        print(s.get('accessionId'), s.get('diseaseTrait', {}).get('trait'), s.get('publicationInfo', {}).get('pubmedId'))


Trying studies search by disease trait...
Status: 200
Total studies: 0


Try GWAS Catalog REST API with correct endpoint structure

In [21]:

import requests

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Check available endpoints
print("Checking API root...")
resp = requests.get(BASE_URL)
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    links = data.get('_links', {})
    for k, v in links.items():
        print(f"  {k}: {v.get('href', '')}")


Checking API root...
Status: 200
  singleNucleotidePolymorphisms: https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms{?page,size,sort,projection}
  studies: https://www.ebi.ac.uk/gwas/rest/api/studies{?page,size,sort,projection}
  associations: https://www.ebi.ac.uk/gwas/rest/api/associations{?page,size,sort,projection}
  efoTraits: https://www.ebi.ac.uk/gwas/rest/api/efoTraits{?page,size,sort,projection}
  unpublishedStudies: https://www.ebi.ac.uk/gwas/rest/api/unpublished-studies{?page,size,sort}
  profile: https://www.ebi.ac.uk/gwas/rest/api/profile


Search efoTraits and studies endpoints for Alzheimer's disease

In [23]:

import requests

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Check efoTraits search options
print("Checking efoTraits search options...")
resp = requests.get(f"{BASE_URL}/efoTraits/search")
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    links = data.get('_links', {})
    for k, v in links.items():
        print(f"  {k}: {v.get('href', '')}")

print()
# Check studies search options
print("Checking studies search options...")
resp2 = requests.get(f"{BASE_URL}/studies/search")
print(f"Status: {resp2.status_code}")
if resp2.status_code == 200:
    data2 = resp2.json()
    links2 = data2.get('_links', {})
    for k, v in links2.items():
        print(f"  {k}: {v.get('href', '')}")


Checking efoTraits search options...
Status: 200
  findByEfoTrait: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByEfoTrait{?trait,page,size,sort,projection}
  findByEfoUri: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByEfoUri{?uri,page,size,sort,projection}
  findByShortForm: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByShortForm{?shortForm,projection}
  findByPubmedId: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByPubmedId{?pubmedId,page,size,sort,projection}
  self: https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search

Checking studies search options...
Status: 200
  findByEfoTrait: https://www.ebi.ac.uk/gwas/rest/api/studies/search/findByEfoTrait{?efoTrait,page,size,sort,projection}
  findByFullPvalueSet: https://www.ebi.ac.uk/gwas/rest/api/studies/search/findByFullPvalueSet{?fullPvalueSet,page,size,sort,projection}
  findByAccessionId: https://www.ebi.ac.uk/gwas/rest/api/studies/search/findByAccessionId{?accessionId,projection}
  

Find Alzheimer's disease EFO trait and studies by shortForm and URI

In [25]:

import requests

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Try findByShortForm with EFO_0000249
print("Finding EFO trait by shortForm EFO_0000249...")
# Source: https://www.ebi.ac.uk/gwas/efotraits/EFO_0000249
resp = requests.get(f"{BASE_URL}/efoTraits/search/findByShortForm", params={"shortForm": "EFO_0000249"})
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    print(data)

print()
# Try findByEfoUri
print("Finding studies by EFO URI...")
efo_uri = "http://www.ebi.ac.uk/efo/EFO_0000249"
resp2 = requests.get(f"{BASE_URL}/studies/search/findByEfoUri", params={"uri": efo_uri, "size": 5})
print(f"Status: {resp2.status_code}")
if resp2.status_code == 200:
    data2 = resp2.json()
    page_info = data2.get('page', {})
    print(f"Total studies: {page_info.get('totalElements', 'N/A')}")
    studies = data2.get('_embedded', {}).get('studies', [])
    for s in studies[:3]:
        print(s.get('accessionId'), s.get('diseaseTrait', {}).get('trait'), s.get('publicationInfo', {}).get('pubmedId'))


Finding EFO trait by shortForm EFO_0000249...
Status: 404

Finding studies by EFO URI...
Status: 200
Total studies: 0


Query GWAS Catalog by known PMIDs for major Alzheimer's meta-analyses

In [27]:

import requests

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Known PMIDs for major AD GWAS meta-analyses:
# Lambert 2013: 24162737
# Kunkle 2019: 30820047
# Bellenguez 2022: 35379992
# Jansen 2019: 30617256
# Wightman 2021: 34385711

pmids = {
    "Lambert_2013": "24162737",
    "Kunkle_2019": "30820047",
    "Bellenguez_2022": "35379992",
    "Jansen_2019": "30617256",
    "Wightman_2021": "34385711"
}

print("Fetching studies by PMID...")
for name, pmid in pmids.items():
    # Source: https://www.ebi.ac.uk/gwas/rest/api/studies/search/findByPublicationIdPubmedId
    resp = requests.get(f"{BASE_URL}/studies/search/findByPublicationIdPubmedId",
                        params={"pubmedId": pmid, "size": 20})
    if resp.status_code == 200:
        data = resp.json()
        studies = data.get('_embedded', {}).get('studies', [])
        total = data.get('page', {}).get('totalElements', 0)
        print(f"\n{name} (PMID {pmid}): {total} studies")
        for s in studies:
            print(f"  Accession: {s.get('accessionId')}, Trait: {s.get('diseaseTrait', {}).get('trait')}, InitialSampleSize: {s.get('initialSampleSize', '')[:80] if s.get('initialSampleSize') else ''}")
    else:
        print(f"{name}: HTTP {resp.status_code}")


Fetching studies by PMID...

Lambert_2013 (PMID 24162737): 1 studies
  Accession: GCST002245, Trait: Alzheimer's disease (late onset), InitialSampleSize: 17,008 European ancestry cases, 37,154 European ancestry controls

Kunkle_2019 (PMID 30820047): 1 studies
  Accession: GCST007511, Trait: Alzheimer's disease (late onset), InitialSampleSize: 21,982 European ancestry cases, 41,944 European ancestry controls

Bellenguez_2022 (PMID 35379992): 1 studies
  Accession: GCST90027158, Trait: Alzheimer's disease, InitialSampleSize: 39,106 European ancestry clinically diagnosed cases, 46,828 European ancestry pr

Jansen_2019 (PMID 30617256): 3 studies
  Accession: GCST007319, Trait: Alzheimer's disease (late onset), InitialSampleSize: 24,087 European ancestry cases, 55,058 European ancestry controls
  Accession: GCST007321, Trait: Family history of Alzheimer's disease, InitialSampleSize: 47,793 European ancestry cases, 328,320 European ancestry controls
  Accession: GCST007320, Trait: Alzheimer'

Fetch all associations for the key AD GWAS studies

In [29]:

import requests
import pandas as pd
import time

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Focus on the 4 key European ancestry AD meta-analyses
target_studies = {
    "GCST002245": {"name": "Lambert_2013", "pmid": "24162737"},
    "GCST007511": {"name": "Kunkle_2019", "pmid": "30820047"},
    "GCST90027158": {"name": "Bellenguez_2022", "pmid": "35379992"},
    "GCST007319": {"name": "Jansen_2019", "pmid": "30617256"},
}

def fetch_associations_for_study(accession, page_size=500):
    """Fetch all associations for a given study accession."""
    # Source: https://www.ebi.ac.uk/gwas/rest/api/studies/{accessionId}/associations
    url = f"{BASE_URL}/studies/{accession}/associations"
    all_assocs = []
    page = 0
    
    while True:
        resp = requests.get(url, params={"size": page_size, "page": page})
        if resp.status_code != 200:
            print(f"  Error {resp.status_code} on page {page}")
            break
        data = resp.json()
        assocs = data.get('_embedded', {}).get('associations', [])
        all_assocs.extend(assocs)
        
        page_info = data.get('page', {})
        total_pages = page_info.get('totalPages', 1)
        total_elements = page_info.get('totalElements', 0)
        
        if page == 0:
            print(f"  Total associations: {total_elements}, pages: {total_pages}")
        
        if page >= total_pages - 1:
            break
        page += 1
        time.sleep(0.3)
    
    return all_assocs

all_records = []

for accession, meta in target_studies.items():
    print(f"\nFetching {meta['name']} ({accession})...")
    assocs = fetch_associations_for_study(accession)
    print(f"  Retrieved {len(assocs)} associations")
    
    for a in assocs:
        # Extract p-value
        pval_mant = a.get('pvalueMantissa')
        pval_exp = a.get('pvalueExponent')
        if pval_mant is not None and pval_exp is not None:
            pvalue = float(pval_mant) * (10 ** int(pval_exp))
        else:
            pvalue = None
        
        # Filter p < 5e-8
        if pvalue is None or pvalue >= 5e-8:
            continue
        
        # Extract loci / SNP info
        loci = a.get('loci', [])
        for locus in loci:
            # Strongest risk alleles
            strongest_alleles = locus.get('strongestRiskAlleles', [])
            # Author reported genes
            author_genes = locus.get('authorReportedGenes', [])
            gene_names = [g.get('geneName', '') for g in author_genes]
            
            for allele in strongest_alleles:
                snp_link = allele.get('_links', {}).get('snp', {}).get('href', '')
                rsid = allele.get('riskAlleleName', '').split('-')[0] if allele.get('riskAlleleName') else ''
                
                record = {
                    'study_accession': accession,
                    'study_name': meta['name'],
                    'pmid': meta['pmid'],
                    'rsID': rsid,
                    'risk_allele': allele.get('riskAlleleName', ''),
                    'pvalue': pvalue,
                    'pvalue_mantissa': pval_mant,
                    'pvalue_exponent': pval_exp,
                    'odds_ratio': a.get('orPerCopyNum'),
                    'beta': a.get('betaNum'),
                    'beta_unit': a.get('betaUnit'),
                    'mapped_gene': ', '.join(gene_names) if gene_names else '',
                    'snp_link': snp_link,
                }
                all_records.append(record)

print(f"\nTotal significant associations (p < 5e-8): {len(all_records)}")



Fetching Lambert_2013 (GCST002245)...
  Total associations: 0, pages: 1
  Retrieved 33 associations

Fetching Kunkle_2019 (GCST007511)...
  Total associations: 0, pages: 1
  Retrieved 24 associations

Fetching Bellenguez_2022 (GCST90027158)...
  Total associations: 0, pages: 1
  Retrieved 89 associations

Fetching Jansen_2019 (GCST007319)...
  Total associations: 0, pages: 1
  Retrieved 33 associations

Total significant associations (p < 5e-8): 134


Enrich associations with chromosome, position, and ancestry from SNP and study endpoints

In [31]:

import requests
import pandas as pd
import time

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Build initial dataframe
df = pd.DataFrame(all_records)
print(f"Shape: {df.shape}")
print(df[['study_name', 'rsID', 'pvalue', 'odds_ratio', 'beta', 'mapped_gene']].head(10))


Shape: (134, 13)
     study_name        rsID        pvalue  odds_ratio  beta mapped_gene
0  Lambert_2013   rs2718058  5.000000e-09      1.0753   NaN        NME8
1  Lambert_2013   rs7274581  3.000000e-08      1.1364   NaN       CASS4
2  Lambert_2013  rs11218343  1.000000e-14      1.2987   NaN       SORL1
3  Lambert_2013   rs6656401  6.000000e-24      1.1800   NaN         CR1
4  Lambert_2013   rs6733839  7.000000e-44      1.2200   NaN        BIN1
5  Lambert_2013  rs10948363  5.000000e-11      1.1000   NaN       CD2AP
6  Lambert_2013  rs11771145  1.000000e-13      1.1111   NaN       EPHA1
7  Lambert_2013   rs9331896  3.000000e-25      1.1628   NaN         CLU
8  Lambert_2013    rs983392  6.000000e-16      1.1111   NaN      MS4A6A
9  Lambert_2013  rs10792832  9.000000e-26      1.1494   NaN      PICALM


Fetch SNP details (chromosome, position) and study ancestry for all associations

In [33]:

import requests
import pandas as pd
import time

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Get unique rsIDs to fetch SNP details
unique_rsids = df['rsID'].dropna().unique()
print(f"Unique rsIDs to look up: {len(unique_rsids)}")

snp_details = {}

for rsid in unique_rsids:
    if not rsid or not rsid.startswith('rs'):
        continue
    # Source: https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/{rsId}
    url = f"{BASE_URL}/singleNucleotidePolymorphisms/{rsid}"
    resp = requests.get(url)
    if resp.status_code == 200:
        data = resp.json()
        locations = data.get('locations', [])
        chrom, pos = None, None
        for loc in locations:
            if loc.get('chromosomeName') and loc.get('chromosomePosition'):
                chrom = loc['chromosomeName']
                pos = loc['chromosomePosition']
                break
        # Mapped genes from genomic context
        genomic_contexts = data.get('genomicContexts', [])
        mapped_genes_snp = []
        for gc in genomic_contexts:
            gene = gc.get('gene', {})
            if gene.get('geneName'):
                mapped_genes_snp.append(gene['geneName'])
        
        snp_details[rsid] = {
            'chromosome': chrom,
            'position': pos,
            'mapped_genes_snp': ', '.join(list(dict.fromkeys(mapped_genes_snp))[:5])  # deduplicate, top 5
        }
    else:
        snp_details[rsid] = {'chromosome': None, 'position': None, 'mapped_genes_snp': ''}
    time.sleep(0.15)

print(f"SNP details fetched: {len(snp_details)}")
# Preview
for k, v in list(snp_details.items())[:5]:
    print(f"  {k}: chr{v['chromosome']}:{v['position']} | genes: {v['mapped_genes_snp']}")


Unique rsIDs to look up: 122
SNP details fetched: 122
  rs2718058: chr7:37801932 | genes: NME8, GPR141BP, GPR141, CDCA3P1
  rs7274581: chr20:56443204 | genes: FAM209A, FAM210B, CASS4, RTF2, AURKA
  rs11218343: chr11:121564878 | genes: LOC105369535, LOC107984402, SORL1-AS1, SORL1, RNU6-256P
  rs6656401: chr1:207518704 | genes: CR1-AS1, CR1, CR2, CD46P1, LOC105372880
  rs6733839: chr2:127135234 | genes: LOC124907883, NIFKP9, BIN1, CYP27C1, LOC105373604


Fetch ancestry information for each study

In [35]:

import requests
import time

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

target_studies = {
    "GCST002245": {"name": "Lambert_2013", "pmid": "24162737"},
    "GCST007511": {"name": "Kunkle_2019", "pmid": "30820047"},
    "GCST90027158": {"name": "Bellenguez_2022", "pmid": "35379992"},
    "GCST007319": {"name": "Jansen_2019", "pmid": "30617256"},
}

study_ancestry = {}

for accession, meta in target_studies.items():
    # Source: https://www.ebi.ac.uk/gwas/rest/api/studies/{accessionId}/ancestries
    url = f"{BASE_URL}/studies/{accession}/ancestries"
    resp = requests.get(url)
    if resp.status_code == 200:
        data = resp.json()
        ancestries = data.get('_embedded', {}).get('ancestries', [])
        ancestry_list = []
        for anc in ancestries:
            anc_type = anc.get('type', '')
            country_of_origin = anc.get('countryOfOrigin', [])
            country_of_recruitment = anc.get('countryOfRecruitment', [])
            anc_categories = anc.get('ancestralGroups', [])
            groups = [g.get('ancestralGroup', '') for g in anc_categories]
            ancestry_list.append(f"{anc_type}: {', '.join(groups)}")
        study_ancestry[accession] = ' | '.join(ancestry_list)
        print(f"{meta['name']}: {study_ancestry[accession][:120]}")
    else:
        study_ancestry[accession] = 'European'
        print(f"{meta['name']}: HTTP {resp.status_code}")
    time.sleep(0.2)


Lambert_2013: HTTP 404
Kunkle_2019: HTTP 404
Bellenguez_2022: HTTP 404
Jansen_2019: HTTP 404


Fetch study details including initial sample size for ancestry info

In [37]:

import requests
import time

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

target_studies = {
    "GCST002245": {"name": "Lambert_2013", "pmid": "24162737"},
    "GCST007511": {"name": "Kunkle_2019", "pmid": "30820047"},
    "GCST90027158": {"name": "Bellenguez_2022", "pmid": "35379992"},
    "GCST007319": {"name": "Jansen_2019", "pmid": "30617256"},
}

study_info = {}

for accession, meta in target_studies.items():
    # Source: https://www.ebi.ac.uk/gwas/rest/api/studies/{accessionId}
    url = f"{BASE_URL}/studies/{accession}"
    resp = requests.get(url)
    if resp.status_code == 200:
        data = resp.json()
        initial_sample = data.get('initialSampleSize', '')
        replication_sample = data.get('replicationSampleSize', '')
        study_info[accession] = {
            'initial_sample': initial_sample,
            'replication_sample': replication_sample,
            'trait': data.get('diseaseTrait', {}).get('trait', ''),
        }
        print(f"{meta['name']}: {initial_sample[:100]}")
    else:
        study_info[accession] = {'initial_sample': 'European ancestry', 'replication_sample': '', 'trait': ''}
    time.sleep(0.2)


Lambert_2013: 17,008 European ancestry cases, 37,154 European ancestry controls
Kunkle_2019: 21,982 European ancestry cases, 41,944 European ancestry controls
Bellenguez_2022: 39,106 European ancestry clinically diagnosed cases, 46,828 European ancestry proxy cases, 401,577 E
Jansen_2019: 24,087 European ancestry cases, 55,058 European ancestry controls


Assemble final dataframe with all fields and save to CSV

In [39]:

import pandas as pd
import os

# Map ancestry from study info
ancestry_map = {
    "GCST002245": "European",
    "GCST007511": "European",
    "GCST90027158": "European",
    "GCST007319": "European",
}

trait_map = {
    "GCST002245": "Alzheimer's disease (late onset)",
    "GCST007511": "Alzheimer's disease (late onset)",
    "GCST90027158": "Alzheimer's disease",
    "GCST007319": "Alzheimer's disease (late onset)",
}

sample_size_map = {
    "GCST002245": "17,008 cases / 37,154 controls",
    "GCST007511": "21,982 cases / 41,944 controls",
    "GCST90027158": "39,106 cases / 46,828 proxy cases / 401,577 controls",
    "GCST007319": "24,087 cases / 55,058 controls",
}

# Enrich df with SNP details
df['chromosome'] = df['rsID'].map(lambda x: snp_details.get(x, {}).get('chromosome'))
df['position'] = df['rsID'].map(lambda x: snp_details.get(x, {}).get('position'))
df['mapped_gene_snp'] = df['rsID'].map(lambda x: snp_details.get(x, {}).get('mapped_genes_snp', ''))

# Use author-reported gene if available, else SNP-mapped gene
df['mapped_gene_final'] = df.apply(
    lambda row: row['mapped_gene'] if row['mapped_gene'] else row['mapped_gene_snp'], axis=1
)

# Add study-level fields
df['ancestry'] = df['study_accession'].map(ancestry_map)
df['trait'] = df['study_accession'].map(trait_map)
df['sample_size'] = df['study_accession'].map(sample_size_map)

# Build final clean dataframe
final_df = df[[
    'rsID',
    'risk_allele',
    'chromosome',
    'position',
    'mapped_gene_final',
    'pvalue',
    'pvalue_mantissa',
    'pvalue_exponent',
    'odds_ratio',
    'beta',
    'beta_unit',
    'ancestry',
    'trait',
    'study_name',
    'study_accession',
    'pmid',
    'sample_size',
]].copy()

final_df.rename(columns={
    'mapped_gene_final': 'mapped_gene',
    'study_name': 'study',
    'study_accession': 'gwas_catalog_accession',
}, inplace=True)

# Sort by p-value
final_df.sort_values('pvalue', inplace=True)
final_df.reset_index(drop=True, inplace=True)

print(f"Final dataset shape: {final_df.shape}")
print(f"\nStudy breakdown:")
print(final_df['study'].value_counts())
print(f"\nTop 20 most significant associations:")
print(final_df[['rsID', 'chromosome', 'position', 'mapped_gene', 'pvalue', 'odds_ratio', 'beta', 'study', 'pmid', 'ancestry']].head(20).to_string(index=False))


Final dataset shape: (134, 17)

Study breakdown:
study
Bellenguez_2022    87
Jansen_2019        21
Lambert_2013       19
Kunkle_2019         7
Name: count, dtype: int64

Top 20 most significant associations:
       rsID chromosome  position                                             mapped_gene        pvalue  odds_ratio      beta           study     pmid ancestry
 rs41289512         19  44848259                                                    APOE 3.000000e-194         NaN 29.735075     Jansen_2019 30617256 European
  rs6733839          2 127135234       LOC124907883, NIFKP9, BIN1, CYP27C1, LOC105373604 6.000000e-118      1.1700       NaN Bellenguez_2022 35379992 European
  rs3851179         11  86157598               EED, FNTAP1, LINC02695, RNU6-560P, PICALM  3.000000e-48      0.9000       NaN Bellenguez_2022 35379992 European
   rs679515          1 207577223                      CDCA4P3, CR1, CD46P1, CR1-AS1, CR2  7.000000e-46      1.1300       NaN Bellenguez_2022 35379992 Europe

Save final results to CSV

In [41]:

import os

os.makedirs("/mnt/results", exist_ok=True)
output_path = "/mnt/results/gwas_catalog_alzheimers_disease_european_significant.csv"
final_df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Total rows: {len(final_df)}")

# Summary stats
print(f"\n--- Summary ---")
print(f"Studies included:")
for study, grp in final_df.groupby('study'):
    pmid = grp['pmid'].iloc[0]
    acc = grp['gwas_catalog_accession'].iloc[0]
    print(f"  {study} (PMID: {pmid}, Accession: {acc}): {len(grp)} associations")

print(f"\nP-value range: {final_df['pvalue'].min():.2e} – {final_df['pvalue'].max():.2e}")
print(f"Unique loci (rsIDs): {final_df['rsID'].nunique()}")
print(f"Ancestry: {final_df['ancestry'].unique()}")

print(f"\nTop 10 most significant hits:")
print(final_df[['rsID', 'chromosome', 'position', 'mapped_gene', 'pvalue', 'odds_ratio', 'beta', 'study', 'ancestry']].head(10).to_string(index=False))


Saved to: /mnt/results/gwas_catalog_alzheimers_disease_european_significant.csv
Total rows: 134

--- Summary ---
Studies included:
  Bellenguez_2022 (PMID: 35379992, Accession: GCST90027158): 87 associations
  Jansen_2019 (PMID: 30617256, Accession: GCST007319): 21 associations
  Kunkle_2019 (PMID: 30820047, Accession: GCST007511): 7 associations
  Lambert_2013 (PMID: 24162737, Accession: GCST002245): 19 associations

P-value range: 3.00e-194 – 4.00e-08
Unique loci (rsIDs): 122
Ancestry: ['European']

Top 10 most significant hits:
      rsID chromosome  position                                       mapped_gene        pvalue  odds_ratio      beta           study ancestry
rs41289512         19  44848259                                              APOE 3.000000e-194         NaN 29.735075     Jansen_2019 European
 rs6733839          2 127135234 LOC124907883, NIFKP9, BIN1, CYP27C1, LOC105373604 6.000000e-118        1.17       NaN Bellenguez_2022 European
 rs3851179         11  86157598   

Try GWAS Catalog associations endpoint directly with study accession for Bellenguez 2022

In [43]:

import requests
import pandas as pd
import time

# Source: https://www.ebi.ac.uk/gwas/rest/api
base_url = "https://www.ebi.ac.uk/gwas/rest/api"

# Try fetching associations directly, filtering by p-value
# Use the associations endpoint with pvalue filter
# Known large AD GWAS study accessions:
# Bellenguez 2022: GCST90027158
# Kunkle 2019: GCST007511
# Lambert 2013: GCST001245

study_accessions = ["GCST90027158", "GCST007511", "GCST001245"]

all_rows = []
for acc in study_accessions:
    url = f"{base_url}/studies/{acc}/associations"
    params = {"size": 200, "page": 0}
    resp = requests.get(url, params=params, timeout=60)
    print(f"Study {acc}: status {resp.status_code}")
    if resp.status_code == 200:
        data = resp.json()
        assocs = data.get("_embedded", {}).get("associations", [])
        page_info = data.get("page", {})
        print(f"  Total associations: {page_info.get('totalElements', 'unknown')}, fetched: {len(assocs)}")
        for a in assocs[:3]:
            loci = a.get("loci", [])
            snps = []
            genes = []
            for loc in loci:
                for sa in loc.get("strongestRiskAlleles", []):
                    snps.append(sa.get("riskAlleleName", ""))
                for ga in loc.get("authorReportedGenes", []):
                    genes.append(ga.get("geneName", ""))
            print(f"    SNP: {snps}, genes: {genes}, pval: {a.get('pvalue')}")
    time.sleep(0.5)


Study GCST90027158: status 200
  Total associations: unknown, fetched: 89
    SNP: ['rs141749679-C'], genes: [], pval: 8e-09
    SNP: ['rs679515-T'], genes: [], pval: 7e-46
    SNP: ['rs72777026-G'], genes: [], pval: 3.0000000000000004e-08
Study GCST007511: status 200
  Total associations: unknown, fetched: 24
    SNP: ['rs7185636-T'], genes: ['IQCK'], pval: 2e-08
    SNP: ['rs593742-A'], genes: ['ADAM10'], pval: 7.000000000000001e-09
    SNP: ['rs2830500-C'], genes: ['ADAMTS1'], pval: 3.0000000000000004e-08
Study GCST001245: status 200
  Total associations: unknown, fetched: 1
    SNP: ['rs1722784-?'], genes: ['ANXA9'], pval: 2e-06


Step 1c: Process AD GWAS loci, cross-reference with DDR genes, query MSigDB

In [45]:

import pandas as pd
import numpy as np
import requests
import json

np.random.seed(42)

# ── Load data ──────────────────────────────────────────────────────────────
gwas_df = pd.read_csv('/mnt/results/gwas_catalog_alzheimers_disease_european_significant.csv')
ddr_df  = pd.read_csv('/mnt/results/ddr_gene_universe.csv')

print("GWAS columns:", gwas_df.columns.tolist())
print("GWAS shape:", gwas_df.shape)
print("\nFirst row sample:")
print(gwas_df.iloc[0])


GWAS columns: ['rsID', 'risk_allele', 'chromosome', 'position', 'mapped_gene', 'pvalue', 'pvalue_mantissa', 'pvalue_exponent', 'odds_ratio', 'beta', 'beta_unit', 'ancestry', 'trait', 'study', 'gwas_catalog_accession', 'pmid', 'sample_size']
GWAS shape: (134, 17)

First row sample:
rsID                                            rs41289512
risk_allele                                   rs41289512-G
chromosome                                              19
position                                          44848259
mapped_gene                                           APOE
pvalue                                                 0.0
pvalue_mantissa                                          3
pvalue_exponent                                       -194
odds_ratio                                             NaN
beta                                             29.735075
beta_unit                                          z-score
ancestry                                          European
trait     

Step 1c: Cross-reference AD GWAS loci with DDR genes; query MSigDB for DNA repair gene sets

In [47]:

import pandas as pd
import numpy as np
import requests
import re

np.random.seed(42)

# ── Load data ──────────────────────────────────────────────────────────────
gwas_df = pd.read_csv('/mnt/results/gwas_catalog_alzheimers_disease_european_significant.csv')
ddr_df  = pd.read_csv('/mnt/results/ddr_gene_universe.csv')

ddr_genes_set = set(ddr_df['gene_symbol'].str.upper())

# ── Clean GWAS: reconstruct proper p-value ─────────────────────────────────
gwas_df['pvalue_numeric'] = gwas_df['pvalue_mantissa'] * 10**gwas_df['pvalue_exponent']

# Filter to genome-wide significance
gwas_sig = gwas_df[gwas_df['pvalue_numeric'] <= 5e-8].copy()
print(f"Genome-wide significant associations (p ≤ 5e-8): {len(gwas_sig)}")
print(f"Unique rsIDs: {gwas_sig['rsID'].nunique()}")
print(f"Unique mapped genes: {gwas_sig['mapped_gene'].nunique()}")

# Save cleaned AD GWAS loci
gwas_sig_out = gwas_sig[['rsID','chromosome','position','mapped_gene',
                          'pvalue_numeric','odds_ratio','beta','pmid',
                          'study','gwas_catalog_accession','ancestry','sample_size']].copy()
gwas_sig_out.columns = ['rsID','chromosome','position','mapped_gene',
                         'pvalue','odds_ratio','beta','pmid',
                         'study','gwas_accession','ancestry','sample_size']
gwas_sig_out.to_csv('/mnt/results/ad_gwas_loci.csv', index=False)
print("\nSaved: ad_gwas_loci.csv")

# ── Cross-reference: GWAS mapped genes vs DDR gene universe ───────────────
# Parse mapped_gene field — can contain multiple genes separated by / or ,
def parse_genes(gene_str):
    if pd.isna(gene_str):
        return []
    # Split on common delimiters
    parts = re.split(r'[/,;\s]+', str(gene_str))
    return [g.strip().upper() for g in parts if g.strip()]

gwas_sig['gene_list'] = gwas_sig['mapped_gene'].apply(parse_genes)

overlap_rows = []
for _, row in gwas_sig.iterrows():
    for gene in row['gene_list']:
        if gene in ddr_genes_set:
            pathway = ddr_df.loc[ddr_df['gene_symbol'].str.upper() == gene, 'pathway'].values
            overlap_rows.append({
                'rsID': row['rsID'],
                'chromosome': row['chromosome'],
                'position': row['position'],
                'ad_mapped_gene': row['mapped_gene'],
                'ddr_gene': gene,
                'ddr_pathway': pathway[0] if len(pathway) > 0 else 'unknown',
                'pvalue': row['pvalue_numeric'],
                'odds_ratio': row['odds_ratio'],
                'beta': row['beta'],
                'pmid': row['pmid'],
                'study': row['study'],
            })

overlap_df = pd.DataFrame(overlap_rows)
print(f"\n── Direct DDR–AD GWAS overlaps ──")
print(f"Overlapping SNP-gene pairs: {len(overlap_df)}")
if len(overlap_df) > 0:
    print(overlap_df[['rsID','ddr_gene','ddr_pathway','pvalue','study']].to_string(index=False))
else:
    print("No direct overlaps found in mapped gene field.")

overlap_df.to_csv('/mnt/results/ddr_ad_overlap.csv', index=False)
print("\nSaved: ddr_ad_overlap.csv")


Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 16, in <module>
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/ops/common.py", line 76, in new_method
    return method(self, other)
           ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/arraylike.py", line 246, in __rpow__
    return self._arith_method(other, roperator.rpow)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/series.py", line 6146, in _arith_method
    return base.IndexOpsMixin._arith_method(self, other, op)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/base.py", line 1391, in _arith_method
    result = ops.arithmetic_op(lvalues, rvalues, op)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/ops/array_ops.py", line 283, in ari

Step 1c: Fix p-value computation and cross-reference DDR vs AD GWAS loci

In [49]:

import pandas as pd
import numpy as np
import requests
import re

np.random.seed(42)

gwas_df = pd.read_csv('/mnt/results/gwas_catalog_alzheimers_disease_european_significant.csv')
ddr_df  = pd.read_csv('/mnt/results/ddr_gene_universe.csv')
ddr_genes_set = set(ddr_df['gene_symbol'].str.upper())

# Fix: cast to float before exponentiation
gwas_df['pvalue_numeric'] = gwas_df['pvalue_mantissa'].astype(float) * (10.0 ** gwas_df['pvalue_exponent'].astype(float))

# Filter to genome-wide significance
gwas_sig = gwas_df[gwas_df['pvalue_numeric'] <= 5e-8].copy()
print(f"Genome-wide significant (p ≤ 5e-8): {len(gwas_sig)} associations")
print(f"Unique rsIDs: {gwas_sig['rsID'].nunique()}")
print(f"Unique mapped genes: {gwas_sig['mapped_gene'].nunique()}")
print(f"\nStudy breakdown:")
print(gwas_sig['study'].value_counts())

# Save cleaned AD GWAS loci
gwas_out = gwas_sig[['rsID','chromosome','position','mapped_gene',
                      'pvalue_numeric','odds_ratio','beta','pmid',
                      'study','gwas_catalog_accession','ancestry','sample_size']].copy()
gwas_out.columns = ['rsID','chromosome','position','mapped_gene',
                    'pvalue','odds_ratio','beta','pmid',
                    'study','gwas_accession','ancestry','sample_size']
gwas_out.to_csv('/mnt/results/ad_gwas_loci.csv', index=False)
print("\nSaved: ad_gwas_loci.csv")

# ── Cross-reference: GWAS mapped genes vs DDR gene universe ───────────────
def parse_genes(gene_str):
    if pd.isna(gene_str):
        return []
    parts = re.split(r'[/,;\s]+', str(gene_str))
    return [g.strip().upper() for g in parts if g.strip()]

gwas_sig['gene_list'] = gwas_sig['mapped_gene'].apply(parse_genes)

overlap_rows = []
for _, row in gwas_sig.iterrows():
    for gene in row['gene_list']:
        if gene in ddr_genes_set:
            pathway = ddr_df.loc[ddr_df['gene_symbol'].str.upper() == gene, 'pathway'].values
            overlap_rows.append({
                'rsID': row['rsID'],
                'chromosome': row['chromosome'],
                'position': row['position'],
                'ad_mapped_gene': row['mapped_gene'],
                'ddr_gene': gene,
                'ddr_pathway': pathway[0] if len(pathway) > 0 else 'unknown',
                'pvalue': row['pvalue_numeric'],
                'odds_ratio': row['odds_ratio'],
                'beta': row['beta'],
                'pmid': row['pmid'],
                'study': row['study'],
            })

overlap_df = pd.DataFrame(overlap_rows)
print(f"\n── Direct DDR–AD GWAS overlaps (mapped gene field) ──")
print(f"Overlapping SNP-gene pairs: {len(overlap_df)}")
if len(overlap_df) > 0:
    print(overlap_df[['rsID','ddr_gene','ddr_pathway','pvalue','study']].to_string(index=False))
overlap_df.to_csv('/mnt/results/ddr_ad_overlap.csv', index=False)
print("\nSaved: ddr_ad_overlap.csv")

# ── Also check all unique AD GWAS genes for any DDR proximity ─────────────
all_ad_genes = set()
for gl in gwas_sig['gene_list']:
    all_ad_genes.update(gl)
print(f"\nAll unique genes at AD GWAS loci: {len(all_ad_genes)}")
print("AD GWAS genes:", sorted(all_ad_genes)[:30], "...")


Genome-wide significant (p ≤ 5e-8): 134 associations
Unique rsIDs: 122
Unique mapped genes: 120

Study breakdown:
study
Bellenguez_2022    87
Jansen_2019        21
Lambert_2013       19
Kunkle_2019         7
Name: count, dtype: int64

Saved: ad_gwas_loci.csv

── Direct DDR–AD GWAS overlaps (mapped gene field) ──
Overlapping SNP-gene pairs: 2
     rsID ddr_gene ddr_pathway       pvalue           study
rs6584063     DNTT        NHEJ 7.000000e-11 Bellenguez_2022
rs6943429     RPA3          HR 1.000000e-10 Bellenguez_2022

Saved: ddr_ad_overlap.csv

All unique genes at AD GWAS loci: 407
AD GWAS genes: ['ABCA1', 'ABCA7', 'ABI3', 'ACE', 'ADAM10', 'ADAM17', 'ADAMTS1', 'ADCY10P1', 'ANK3', 'ANKH', 'ANXA6', 'APOBEC2', 'APOE', 'APP', 'APP-DT', 'ARMS2', 'ASPHD1', 'ATG16L1', 'ATP8B4', 'AURKA', 'BCKDK', 'BIN1', 'BLNK', 'BTBD16', 'BZRAP1-AS1', 'CA12', 'CARF', 'CASS4', 'CASTOR3P', 'CCNB2P1'] ...


Step 1d: Query MSigDB for DNA repair / DDR gene sets to expand universe

In [51]:

import pandas as pd
import numpy as np
import requests

np.random.seed(42)

# Source: https://www.gsea-msigdb.org/gsea/msigdb/
# MSigDB REST API — query C2:CGP and C5:GO for DNA repair gene sets
MSIGDB_API = "https://www.gsea-msigdb.org/gsea/msigdb/human/download_geneset.jsp"

# Use the MSigDB gene sets available locally via msigdbr
import subprocess
subprocess.run(["pip", "install", "msigdbr", "-q"])

import msigdbr

# Query C2 (curated gene sets) and C5 (GO gene sets) for DNA repair
print("Querying MSigDB C2:CGP for DNA repair gene sets...")
c2_sets = msigdbr.msigdbr(species="Homo sapiens", category="C2", subcategory="CGP")
c2_ddr = c2_sets[c2_sets['gs_name'].str.contains('DNA_REPAIR|DNA_DAMAGE|DDR|HOMOLOGOUS_RECOMB|MISMATCH_REPAIR|BASE_EXCISION|NUCLEOTIDE_EXCISION|FANCONI|NHEJ|ATM|ATR|CHECKPOINT', case=False, regex=True)]
print(f"C2:CGP DDR-related gene sets: {c2_ddr['gs_name'].nunique()}")
print(c2_ddr['gs_name'].unique()[:15])

print("\nQuerying MSigDB C5:GO for DNA repair gene sets...")
c5_sets = msigdbr.msigdbr(species="Homo sapiens", category="C5", subcategory="GO:BP")
c5_ddr = c5_sets[c5_sets['gs_name'].str.contains('DNA_REPAIR|DNA_DAMAGE|DOUBLE_STRAND_BREAK|MISMATCH_REPAIR|BASE_EXCISION|NUCLEOTIDE_EXCISION|HOMOLOGOUS_RECOMB|FANCONI|NONHOMOLOGOUS|ATM|ATR|CHECKPOINT', case=False, regex=True)]
print(f"C5:GO:BP DDR-related gene sets: {c5_ddr['gs_name'].nunique()}")
print(c5_ddr['gs_name'].unique()[:15])

# Also check Hallmark
print("\nQuerying MSigDB H (Hallmark) for DNA repair...")
h_sets = msigdbr.msigdbr(species="Homo sapiens", category="H")
h_ddr = h_sets[h_sets['gs_name'].str.contains('DNA_REPAIR|DNA_DAMAGE', case=False, regex=True)]
print(f"Hallmark DDR gene sets: {h_ddr['gs_name'].nunique()}")
print(h_ddr['gs_name'].unique())


Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 16, in <module>
ModuleNotFoundError: No module named 'msigdbr'


Step 1d: Query local MSigDB parquet files for DNA repair gene sets

In [53]:

import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Source: /mnt/datalake/msigdb/ — MSigDB human gene sets (local data lake)
# Load C2 (curated) and C5 (GO) gene sets
c2 = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_c2_curated_geneset.parquet')
c5 = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_c5_ontology_geneset.parquet')
h  = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_h_hallmark_geneset.parquet')

print("C2 columns:", c2.columns.tolist())
print("C2 shape:", c2.shape)
print("C5 columns:", c5.columns.tolist())
print("C5 shape:", c5.shape)
print("H columns:", h.columns.tolist())
print("\nC2 sample:")
print(c2.head(3))


C2 columns: ['chromosome_id', 'collection', 'systematicName', 'msigdbURL', 'exactSource', 'externalDetailsURL', 'pmid', 'geneSymbols', 'filteredBySimilarity', 'externalNamesForSimilarTerms']
C2 shape: (7411, 10)
C5 columns: ['chromosome_id', 'collection', 'systematicName', 'msigdbURL', 'exactSource', 'externalDetailsURL', 'pmid', 'geneSymbols', 'filteredBySimilarity', 'externalNamesForSimilarTerms']
C5 shape: (16107, 10)
H columns: ['chromosome_id', 'collection', 'systematicName', 'msigdbURL', 'exactSource', 'externalDetailsURL', 'pmid', 'geneSymbols', 'filteredBySimilarity', 'externalNamesForSimilarTerms']

C2 sample:
                        chromosome_id  ... externalNamesForSimilarTerms
0   FUJIWARA_PARK2_IN_LIVER_CANCER_DN  ...                           []
1  BIERIE_INFLAMMATORY_RESPONSE_TGFB1  ...                           []
2    LI_ESTROGENE_META_E2_RESPONSE_UP  ...                           []

[3 rows x 10 columns]


Step 1d: Extract DDR gene sets from MSigDB and expand DDR gene universe

In [55]:

import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Source: /mnt/datalake/msigdb/ — MSigDB human gene sets
c2 = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_c2_curated_geneset.parquet')
c5 = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_c5_ontology_geneset.parquet')
h  = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_h_hallmark_geneset.parquet')

# DDR-related keyword pattern
DDR_PATTERN = (r'DNA_REPAIR|DNA_DAMAGE|DOUBLE_STRAND_BREAK|MISMATCH_REPAIR|'
               r'BASE_EXCISION|NUCLEOTIDE_EXCISION|HOMOLOGOUS_RECOMB|FANCONI|'
               r'NONHOMOLOGOUS|ATM_PATHWAY|ATR_PATHWAY|CHECKPOINT|'
               r'BRCA|RAD51|XRCC|ERCC|MLH|MSH|PMS2|OGG1|APEX1')

def filter_ddr_sets(df, label):
    mask = df['chromosome_id'].str.contains(DDR_PATTERN, case=False, regex=True)
    sub = df[mask].copy()
    print(f"{label}: {len(sub)} DDR-related gene sets")
    print("  Gene sets:", sub['chromosome_id'].tolist()[:10])
    return sub

c2_ddr = filter_ddr_sets(c2, "C2:CGP")
c5_ddr = filter_ddr_sets(c5, "C5:GO:BP")
h_ddr  = filter_ddr_sets(h,  "Hallmark")

# Extract all unique genes from DDR gene sets
def extract_genes(df):
    genes = set()
    for gs in df['geneSymbols']:
        if isinstance(gs, list):
            genes.update([g.upper() for g in gs])
        elif isinstance(gs, str):
            genes.update([g.strip().upper() for g in gs.split(',')])
    return genes

msigdb_ddr_genes = extract_genes(c2_ddr) | extract_genes(c5_ddr) | extract_genes(h_ddr)
print(f"\nTotal unique genes in MSigDB DDR gene sets: {len(msigdb_ddr_genes)}")

# Load existing DDR universe and expand
ddr_df = pd.read_csv('/mnt/results/ddr_gene_universe.csv')
existing = set(ddr_df['gene_symbol'].str.upper())

# New genes from MSigDB not already in curated list
new_genes = msigdb_ddr_genes - existing
print(f"New genes from MSigDB not in curated list: {len(new_genes)}")
print("Sample new genes:", sorted(new_genes)[:20])

# Add new genes with pathway = "MSigDB_DDR"
new_rows = pd.DataFrame({'gene_symbol': sorted(new_genes), 'pathway': 'MSigDB_DDR'})
ddr_expanded = pd.concat([ddr_df, new_rows], ignore_index=True)
ddr_expanded.to_csv('/mnt/results/ddr_gene_universe.csv', index=False)
print(f"\nExpanded DDR gene universe: {len(ddr_expanded)} genes")
print(ddr_expanded['pathway'].value_counts())


C2:CGP: 120 DDR-related gene sets
  Gene sets: ['DACOSTA_ERCC3_ALLELE_XPCS_VS_TTD_UP', 'DACOSTA_ERCC3_ALLELE_XPCS_VS_TTD_DN', 'DACOSTA_LOW_DOSE_UV_RESPONSE_VIA_ERCC3_XPCS_UP', 'DACOSTA_LOW_DOSE_UV_RESPONSE_VIA_ERCC3_XPCS_DN', 'JAZAERI_BREAST_CANCER_BRCA1_VS_BRCA2_DN', 'MACLACHLAN_BRCA1_TARGETS_DN', 'MACLACHLAN_BRCA1_TARGETS_UP', 'WELCSH_BRCA1_TARGETS_UP', 'JAZAERI_BREAST_CANCER_BRCA1_VS_BRCA2_UP', 'KYNG_RESPONSE_TO_H2O2_VIA_ERCC6']
C5:GO:BP: 89 DDR-related gene sets
  Gene sets: ['GOBP_CELL_CYCLE_CHECKPOINT_SIGNALING', 'GOBP_DNA_REPLICATION_CHECKPOINT_SIGNALING', 'GOBP_DOUBLE_STRAND_BREAK_REPAIR_VIA_BREAK_INDUCED_REPLICATION', 'GOBP_DNA_DOUBLE_STRAND_BREAK_PROCESSING', 'GOBP_DNA_SYNTHESIS_INVOLVED_IN_DNA_REPAIR', 'GOBP_DNA_REPAIR', 'GOBP_REGULATION_OF_DNA_REPAIR', 'GOBP_TRANSCRIPTION_COUPLED_NUCLEOTIDE_EXCISION_REPAIR', 'GOBP_BASE_EXCISION_REPAIR', 'GOBP_BASE_EXCISION_REPAIR_AP_SITE_FORMATION']
Hallmark: 2 DDR-related gene sets
  Gene sets: ['HALLMARK_DNA_REPAIR', 'HALLMARK_G2M_CHECKPO

Step 1d: Fix MSigDB gene parsing, finalize DDR universe and DDR-AD overlap

In [57]:

import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Source: /mnt/datalake/msigdb/
c2 = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_c2_curated_geneset.parquet')
c5 = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_c5_ontology_geneset.parquet')
h  = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_h_hallmark_geneset.parquet')

DDR_PATTERN = (r'DNA_REPAIR|DNA_DAMAGE|DOUBLE_STRAND_BREAK|MISMATCH_REPAIR|'
               r'BASE_EXCISION|NUCLEOTIDE_EXCISION|HOMOLOGOUS_RECOMB|FANCONI|'
               r'NONHOMOLOGOUS|ATM_PATHWAY|ATR_PATHWAY|CHECKPOINT|'
               r'BRCA|RAD51|XRCC|ERCC|MLH|MSH|PMS2|OGG1|APEX1')

def filter_ddr_sets(df):
    return df[df['chromosome_id'].str.contains(DDR_PATTERN, case=False, regex=True)].copy()

c2_ddr = filter_ddr_sets(c2)
c5_ddr = filter_ddr_sets(c5)
h_ddr  = filter_ddr_sets(h)

# Fix gene parsing — strip quotes and whitespace
def extract_genes_clean(df):
    genes = set()
    for gs in df['geneSymbols']:
        if isinstance(gs, list):
            for g in gs:
                clean = str(g).strip().strip("'\"").upper()
                if re.match(r'^[A-Z][A-Z0-9\-]+$', clean):
                    genes.add(clean)
        elif isinstance(gs, str):
            for g in gs.split(','):
                clean = g.strip().strip("'\"").upper()
                if re.match(r'^[A-Z][A-Z0-9\-]+$', clean):
                    genes.add(clean)
    return genes

msigdb_ddr_genes = extract_genes_clean(c2_ddr) | extract_genes_clean(c5_ddr) | extract_genes_clean(h_ddr)
print(f"MSigDB DDR genes (cleaned): {len(msigdb_ddr_genes)}")
print("Sample:", sorted(msigdb_ddr_genes)[:15])

# ── Rebuild DDR universe: curated core (107 genes) + MSigDB additions ─────
# Reload original curated list (before the bad expansion)
ddr_core = {
    "BRCA1":"HR","BRCA2":"HR","PALB2":"HR","RAD51":"HR","RAD51C":"HR","RAD51D":"HR",
    "BRIP1":"HR","BARD1":"HR","RAD52":"HR","RAD54L":"HR","RAD54B":"HR","RBBP8":"HR",
    "NBN":"HR","MRE11":"HR","RAD50":"HR","RPA1":"HR","RPA2":"HR","RPA3":"HR",
    "SLX4":"HR","GEN1":"HR",
    "LIG4":"NHEJ","XRCC4":"NHEJ","XRCC5":"NHEJ","XRCC6":"NHEJ","PRKDC":"NHEJ",
    "NHEJ1":"NHEJ","DNTT":"NHEJ","ARTEMIS":"NHEJ",
    "MLH1":"MMR","MSH2":"MMR","MSH6":"MMR","PMS2":"MMR","MSH3":"MMR","MLH3":"MMR",
    "PMS1":"MMR","PCNA":"MMR",
    "OGG1":"BER","MUTYH":"BER","APEX1":"BER","XRCC1":"BER","LIG3":"BER",
    "PARP1":"BER","PARP2":"BER","POLB":"BER","NEIL1":"BER","NEIL2":"BER",
    "NEIL3":"BER","UNG":"BER","SMUG1":"BER","TDG":"BER","MBD4":"BER",
    "XPA":"NER","XPC":"NER","ERCC1":"NER","ERCC2":"NER","ERCC3":"NER",
    "ERCC4":"NER","ERCC5":"NER","ERCC6":"NER","ERCC8":"NER","DDB1":"NER",
    "DDB2":"NER","RAD23A":"NER","RAD23B":"NER","CETN2":"NER",
    "ATM":"ATM_ATR","ATR":"ATM_ATR","CHEK1":"ATM_ATR","CHEK2":"ATM_ATR",
    "TP53":"ATM_ATR","MDM2":"ATM_ATR","H2AX":"ATM_ATR","TP53BP1":"ATM_ATR",
    "RNF8":"ATM_ATR","RNF168":"ATM_ATR","UIMC1":"ATM_ATR","PAXIP1":"ATM_ATR",
    "MDC1":"ATM_ATR","TOPBP1":"ATM_ATR","ATRIP":"ATM_ATR","RAD17":"ATM_ATR",
    "RAD9A":"ATM_ATR","HUS1":"ATM_ATR","RAD1":"ATM_ATR",
    "FANCA":"FA","FANCC":"FA","FANCD2":"FA","FANCG":"FA","FANCI":"FA",
    "FANCB":"FA","FANCE":"FA","FANCF":"FA","FANCJ":"FA","FANCL":"FA",
    "FANCM":"FA","FANCN":"FA","FANCO":"FA","FANCP":"FA","FANCQ":"FA",
    "FANCR":"FA","FANCS":"FA","FANCT":"FA","FANCU":"FA","FANCV":"FA","FANCW":"FA",
}

# MSigDB additions: genes in MSigDB DDR sets but not in curated core
core_set = set(ddr_core.keys())
msigdb_additions = sorted(msigdb_ddr_genes - core_set)
print(f"\nMSigDB additions (not in curated core): {len(msigdb_additions)}")
print("Sample:", msigdb_additions[:20])

# Build final DDR universe
rows = [{"gene_symbol": g, "pathway": p, "source": "curated"} for g, p in ddr_core.items()]
for g in msigdb_additions:
    rows.append({"gene_symbol": g, "pathway": "MSigDB_DDR", "source": "MSigDB"})

ddr_final = pd.DataFrame(rows)
ddr_final.to_csv('/mnt/results/ddr_gene_universe.csv', index=False)
print(f"\nFinal DDR gene universe: {len(ddr_final)} genes")
print(ddr_final['pathway'].value_counts())

# ── Re-run DDR–AD GWAS overlap with expanded universe ─────────────────────
gwas_df = pd.read_csv('/mnt/results/ad_gwas_loci.csv')
ddr_all_genes = set(ddr_final['gene_symbol'].str.upper())

def parse_genes(gene_str):
    if pd.isna(gene_str): return []
    return [g.strip().upper() for g in re.split(r'[/,;\s]+', str(gene_str)) if g.strip()]

gwas_df['gene_list'] = gwas_df['mapped_gene'].apply(parse_genes)

overlap_rows = []
for _, row in gwas_df.iterrows():
    for gene in row['gene_list']:
        if gene in ddr_all_genes:
            pathway = ddr_final.loc[ddr_final['gene_symbol'].str.upper() == gene, 'pathway'].values
            source  = ddr_final.loc[ddr_final['gene_symbol'].str.upper() == gene, 'source'].values
            overlap_rows.append({
                'rsID': row['rsID'], 'chromosome': row['chromosome'],
                'position': row['position'], 'ad_mapped_gene': row['mapped_gene'],
                'ddr_gene': gene,
                'ddr_pathway': pathway[0] if len(pathway) > 0 else 'unknown',
                'ddr_source': source[0] if len(source) > 0 else 'unknown',
                'pvalue': row['pvalue'], 'odds_ratio': row['odds_ratio'],
                'beta': row['beta'], 'pmid': row['pmid'], 'study': row['study'],
            })

overlap_df = pd.DataFrame(overlap_rows)
print(f"\n── DDR–AD GWAS overlaps (expanded universe) ──")
print(f"Total overlapping SNP-gene pairs: {len(overlap_df)}")
print(overlap_df[['rsID','ddr_gene','ddr_pathway','ddr_source','pvalue','study']].to_string(index=False))
overlap_df.to_csv('/mnt/results/ddr_ad_overlap.csv', index=False)
print("\nSaved: ddr_ad_overlap.csv")


MSigDB DDR genes (cleaned): 4473
Sample: ['ABCB6', 'ABCB7', 'ABCC1', 'ABCE1', 'ABCF2', 'ABCF3', 'ABCG1', 'ABCG2', 'ABHD14A', 'ABI1', 'ABLIM3', 'ABO', 'ABRAXAS1', 'ABRAXAS2', 'ACAA1']

MSigDB additions (not in curated core): 4380
Sample: ['ABCB6', 'ABCB7', 'ABCC1', 'ABCE1', 'ABCF2', 'ABCF3', 'ABCG1', 'ABCG2', 'ABHD14A', 'ABI1', 'ABLIM3', 'ABO', 'ABRAXAS1', 'ABRAXAS2', 'ACAA1', 'ACADM', 'ACAP2', 'ACAT2', 'ACBD3', 'ACD']

Final DDR gene universe: 4485 genes
pathway
MSigDB_DDR    4380
FA              21
HR              20
ATM_ATR         19
BER             15
NER             14
MMR              8
NHEJ             8
Name: count, dtype: int64

── DDR–AD GWAS overlaps (expanded universe) ──
Total overlapping SNP-gene pairs: 73
       rsID ddr_gene ddr_pathway ddr_source       pvalue           study
  rs3851179      EED  MSigDB_DDR     MSigDB 3.000000e-48 Bellenguez_2022
  rs3851179   PICALM  MSigDB_DDR     MSigDB 3.000000e-48 Bellenguez_2022
 rs11787077    EPHX2  MSigDB_DDR     MSigDB 2.00000

Step 1 fix: Correct DDR universe (curated core + targeted MSigDB sets only), fix overlap file

In [59]:

import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Source: /mnt/datalake/msigdb/ — MSigDB human gene sets
c5 = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_c5_ontology_geneset.parquet')
h  = pd.read_parquet('/mnt/datalake/msigdb/msigdb_human_h_hallmark_geneset.parquet')

# ── 1. Curated core DDR gene universe (107 genes, manually curated) ────────
ddr_core = {
    "BRCA1":"HR","BRCA2":"HR","PALB2":"HR","RAD51":"HR","RAD51C":"HR","RAD51D":"HR",
    "BRIP1":"HR","BARD1":"HR","RAD52":"HR","RAD54L":"HR","RAD54B":"HR","RBBP8":"HR",
    "NBN":"HR","MRE11":"HR","RAD50":"HR","RPA1":"HR","RPA2":"HR","RPA3":"HR",
    "SLX4":"HR","GEN1":"HR",
    "LIG4":"NHEJ","XRCC4":"NHEJ","XRCC5":"NHEJ","XRCC6":"NHEJ","PRKDC":"NHEJ",
    "NHEJ1":"NHEJ","DNTT":"NHEJ","ARTEMIS":"NHEJ",
    "MLH1":"MMR","MSH2":"MMR","MSH6":"MMR","PMS2":"MMR","MSH3":"MMR","MLH3":"MMR",
    "PMS1":"MMR","PCNA":"MMR",
    "OGG1":"BER","MUTYH":"BER","APEX1":"BER","XRCC1":"BER","LIG3":"BER",
    "PARP1":"BER","PARP2":"BER","POLB":"BER","NEIL1":"BER","NEIL2":"BER",
    "NEIL3":"BER","UNG":"BER","SMUG1":"BER","TDG":"BER","MBD4":"BER",
    "XPA":"NER","XPC":"NER","ERCC1":"NER","ERCC2":"NER","ERCC3":"NER",
    "ERCC4":"NER","ERCC5":"NER","ERCC6":"NER","ERCC8":"NER","DDB1":"NER",
    "DDB2":"NER","RAD23A":"NER","RAD23B":"NER","CETN2":"NER",
    "ATM":"ATM_ATR","ATR":"ATM_ATR","CHEK1":"ATM_ATR","CHEK2":"ATM_ATR",
    "TP53":"ATM_ATR","MDM2":"ATM_ATR","H2AX":"ATM_ATR","TP53BP1":"ATM_ATR",
    "RNF8":"ATM_ATR","RNF168":"ATM_ATR","UIMC1":"ATM_ATR","PAXIP1":"ATM_ATR",
    "MDC1":"ATM_ATR","TOPBP1":"ATM_ATR","ATRIP":"ATM_ATR","RAD17":"ATM_ATR",
    "RAD9A":"ATM_ATR","HUS1":"ATM_ATR","RAD1":"ATM_ATR",
    "FANCA":"FA","FANCC":"FA","FANCD2":"FA","FANCG":"FA","FANCI":"FA",
    "FANCB":"FA","FANCE":"FA","FANCF":"FA","FANCJ":"FA","FANCL":"FA",
    "FANCM":"FA","FANCN":"FA","FANCO":"FA","FANCP":"FA","FANCQ":"FA",
    "FANCR":"FA","FANCS":"FA","FANCT":"FA","FANCU":"FA","FANCV":"FA","FANCW":"FA",
}

curated_df = pd.DataFrame([
    {"gene_symbol": g, "pathway": p, "source": "curated"}
    for g, p in ddr_core.items()
])
print(f"Curated core DDR genes: {len(curated_df)}")
print(curated_df['pathway'].value_counts())

# Save curated version
curated_df.to_csv('/mnt/results/ddr_gene_universe_curated.csv', index=False)
print("Saved: ddr_gene_universe_curated.csv")

# ── 2. MSigDB supplement: ONLY specific, well-defined DDR gene sets ────────
# Use only the most specific GO:BP and Hallmark DDR sets
SPECIFIC_DDR_SETS = [
    'HALLMARK_DNA_REPAIR',
    'GOBP_DNA_REPAIR',
    'GOBP_BASE_EXCISION_REPAIR',
    'GOBP_NUCLEOTIDE_EXCISION_REPAIR',
    'GOBP_MISMATCH_REPAIR',
    'GOBP_DOUBLE_STRAND_BREAK_REPAIR',
    'GOBP_DOUBLE_STRAND_BREAK_REPAIR_VIA_HOMOLOGOUS_RECOMBINATION',
    'GOBP_DOUBLE_STRAND_BREAK_REPAIR_VIA_NONHOMOLOGOUS_END_JOINING',
    'GOBP_DNA_DAMAGE_CHECKPOINT_SIGNALING',
    'GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_IN_RESPONSE_TO_DNA_DAMAGE',
    'GOBP_FANCONI_ANEMIA_PATHWAY',
    'GOBP_DNA_SYNTHESIS_INVOLVED_IN_DNA_REPAIR',
]

all_msig = pd.concat([c5, h], ignore_index=True)
msig_specific = all_msig[all_msig['chromosome_id'].isin(SPECIFIC_DDR_SETS)]
print(f"\nSpecific MSigDB DDR gene sets found: {msig_specific['chromosome_id'].nunique()}")
print(msig_specific['chromosome_id'].unique())

def extract_genes_clean(df):
    genes = set()
    for gs in df['geneSymbols']:
        if isinstance(gs, list):
            for g in gs:
                clean = str(g).strip().strip("'\"").upper()
                if re.match(r'^[A-Z][A-Z0-9\-]+$', clean):
                    genes.add(clean)
        elif isinstance(gs, str):
            for g in gs.split(','):
                clean = g.strip().strip("'\"").upper()
                if re.match(r'^[A-Z][A-Z0-9\-]+$', clean):
                    genes.add(clean)
    return genes

msigdb_specific_genes = extract_genes_clean(msig_specific)
print(f"Genes in specific MSigDB DDR sets: {len(msigdb_specific_genes)}")

# Additions not already in curated core
core_set = set(ddr_core.keys())
msigdb_additions = sorted(msigdb_specific_genes - core_set)
print(f"New genes from specific MSigDB sets: {len(msigdb_additions)}")
print("Sample:", msigdb_additions[:20])

# Build expanded universe
msigdb_rows = pd.DataFrame([
    {"gene_symbol": g, "pathway": "MSigDB_DDR_specific", "source": "MSigDB"}
    for g in msigdb_additions
])
ddr_expanded = pd.concat([curated_df, msigdb_rows], ignore_index=True)
ddr_expanded.to_csv('/mnt/results/ddr_gene_universe.csv', index=False)
print(f"\nExpanded DDR gene universe (curated + specific MSigDB): {len(ddr_expanded)} genes")
print(ddr_expanded['pathway'].value_counts())


Curated core DDR genes: 105
pathway
FA         21
HR         20
ATM_ATR    19
BER        15
NER        14
MMR         8
NHEJ        8
Name: count, dtype: int64
Saved: ddr_gene_universe_curated.csv

Specific MSigDB DDR gene sets found: 9
['GOBP_DNA_SYNTHESIS_INVOLVED_IN_DNA_REPAIR' 'GOBP_DNA_REPAIR'
 'GOBP_BASE_EXCISION_REPAIR' 'GOBP_NUCLEOTIDE_EXCISION_REPAIR'
 'GOBP_MISMATCH_REPAIR' 'GOBP_DOUBLE_STRAND_BREAK_REPAIR'
 'GOBP_DOUBLE_STRAND_BREAK_REPAIR_VIA_NONHOMOLOGOUS_END_JOINING'
 'GOBP_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_IN_RESPONSE_TO_DNA_DAMAGE'
 'HALLMARK_DNA_REPAIR']
Genes in specific MSigDB DDR sets: 792
New genes from specific MSigDB sets: 700
Sample: ['ABRAXAS1', 'ACKR3', 'ACTB', 'ACTL6A', 'ACTL6B', 'ACTR2', 'ACTR5', 'ACTR8', 'ADA', 'ADCY6', 'ADPRS', 'ADRM1', 'AEN', 'AGO4', 'AK1', 'AK3', 'ALKBH1', 'ALKBH2', 'ALKBH3', 'ALYREF']

Expanded DDR gene universe (curated + specific MSigDB): 805 genes
pathway
MSigDB_DDR_specific    700
FA                      21
HR                   

Step 1e: Rebuild DDR-AD overlap with corrected gene universe; summarize Step 1

In [61]:

import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Load corrected DDR universe and AD GWAS loci
ddr_df  = pd.read_csv('/mnt/results/ddr_gene_universe.csv')
gwas_df = pd.read_csv('/mnt/results/ad_gwas_loci.csv')

ddr_all_genes = set(ddr_df['gene_symbol'].str.upper())
ddr_curated   = set(ddr_df.loc[ddr_df['source']=='curated', 'gene_symbol'].str.upper())

def parse_genes(gene_str):
    if pd.isna(gene_str): return []
    return [g.strip().upper() for g in re.split(r'[/,;\s]+', str(gene_str)) if g.strip()]

gwas_df['gene_list'] = gwas_df['mapped_gene'].apply(parse_genes)

overlap_rows = []
for _, row in gwas_df.iterrows():
    for gene in row['gene_list']:
        if gene in ddr_all_genes:
            rec = ddr_df[ddr_df['gene_symbol'].str.upper() == gene].iloc[0]
            overlap_rows.append({
                'rsID':          row['rsID'],
                'chromosome':    row['chromosome'],
                'position':      row['position'],
                'ad_mapped_gene':row['mapped_gene'],
                'ddr_gene':      gene,
                'ddr_pathway':   rec['pathway'],
                'ddr_source':    rec['source'],
                'pvalue':        row['pvalue'],
                'odds_ratio':    row['odds_ratio'],
                'beta':          row['beta'],
                'pmid':          row['pmid'],
                'study':         row['study'],
            })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv('/mnt/results/ddr_ad_overlap.csv', index=False)

# ── Summary ────────────────────────────────────────────────────────────────
print("═══ STEP 1 SUMMARY ═══════════════════════════════════════════════")
print(f"\n[DDR Gene Universe]")
print(f"  Curated core genes:          {len(ddr_curated)}")
print(f"  MSigDB-specific DDR genes:   {len(ddr_all_genes) - len(ddr_curated)}")
print(f"  Total DDR universe:          {len(ddr_all_genes)}")

print(f"\n[AD GWAS Loci — GWAS Catalog, p ≤ 5×10⁻⁸, European ancestry]")
print(f"  Total associations:          {len(gwas_df)}")
print(f"  Unique rsIDs:                {gwas_df['rsID'].nunique()}")
print(f"  Unique mapped genes:         {gwas_df['mapped_gene'].nunique()}")
print(f"  Studies: Bellenguez 2022 (PMID 35379992), Lambert 2013 (PMID 24162737),")
print(f"           Jansen 2019 (PMID 30617256), Kunkle 2019 (PMID 30820047)")

print(f"\n[DDR–AD GWAS Overlaps]")
print(f"  Total overlapping SNP-gene pairs: {len(overlap_df)}")
curated_overlaps = overlap_df[overlap_df['ddr_source']=='curated']
msigdb_overlaps  = overlap_df[overlap_df['ddr_source']=='MSigDB']
print(f"  Curated DDR genes at AD loci:     {len(curated_overlaps)}")
print(f"  MSigDB-specific DDR genes at AD loci: {len(msigdb_overlaps)}")

print(f"\n  ── Curated DDR genes directly at AD GWAS loci ──")
print(curated_overlaps[['rsID','ddr_gene','ddr_pathway','pvalue','study']].to_string(index=False))

print(f"\n  ── Top MSigDB DDR genes at AD GWAS loci (p < 1e-10) ──")
top_msig = msigdb_overlaps[msigdb_overlaps['pvalue'] < 1e-10].sort_values('pvalue')
print(top_msig[['rsID','ddr_gene','ddr_pathway','pvalue','study']].to_string(index=False))

print(f"\n[Files saved]")
print("  ddr_gene_universe_curated.csv  — 105 curated core DDR genes")
print("  ddr_gene_universe.csv          — 805 genes (curated + MSigDB-specific)")
print("  ad_gwas_loci.csv               — 134 AD GWAS associations")
print("  ddr_ad_overlap.csv             — DDR–AD GWAS overlaps (annotated by source)")


═══ STEP 1 SUMMARY ═══════════════════════════════════════════════

[DDR Gene Universe]
  Curated core genes:          105
  MSigDB-specific DDR genes:   700
  Total DDR universe:          805

[AD GWAS Loci — GWAS Catalog, p ≤ 5×10⁻⁸, European ancestry]
  Total associations:          134
  Unique rsIDs:                122
  Unique mapped genes:         120
  Studies: Bellenguez 2022 (PMID 35379992), Lambert 2013 (PMID 24162737),
           Jansen 2019 (PMID 30617256), Kunkle 2019 (PMID 30820047)

[DDR–AD GWAS Overlaps]
  Total overlapping SNP-gene pairs: 18
  Curated DDR genes at AD loci:     2
  MSigDB-specific DDR genes at AD loci: 16

  ── Curated DDR genes directly at AD GWAS loci ──
     rsID ddr_gene ddr_pathway       pvalue           study
rs6584063     DNTT        NHEJ 7.000000e-11 Bellenguez_2022
rs6943429     RPA3          HR 1.000000e-10 Bellenguez_2022

  ── Top MSigDB DDR genes at AD GWAS loci (p < 1e-10) ──
      rsID ddr_gene         ddr_pathway       pvalue           s

Step 2a: Fetch AD brain transcriptomics from GEO - GSE5281 (microarray, multiple brain regions)

In [63]:

import GEOparse
import pandas as pd
import numpy as np
import os

np.random.seed(42)
os.makedirs('/mnt/results/tmp', exist_ok=True)

# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE5281
# GSE5281: Entorhinal cortex, hippocampus, frontal cortex, AD vs control
# Liang et al. 2008, Physiological Genomics
print("Downloading GSE5281 from GEO...")
gse = GEOparse.get_GEO(geo="GSE5281", destdir="/mnt/results/tmp", silent=True)
print(f"GSE5281 loaded: {len(gse.gsms)} samples, {len(gse.gpls)} platforms")

# Inspect sample metadata
meta_rows = []
for gsm_name, gsm in gse.gsms.items():
    meta_rows.append({
        'sample_id': gsm_name,
        'title': gsm.metadata.get('title', [''])[0],
        'source': gsm.metadata.get('source_name_ch1', [''])[0],
        'characteristics': gsm.metadata.get('characteristics_ch1', []),
        'description': gsm.metadata.get('description', [''])[0][:100],
    })

meta_df = pd.DataFrame(meta_rows)
print(f"\nSample metadata shape: {meta_df.shape}")
print("\nSample titles (first 10):")
print(meta_df['title'].head(10).tolist())
print("\nUnique sources:")
print(meta_df['source'].value_counts())


GSE5281 loaded: 161 samples, 1 platforms

Sample metadata shape: (161, 5)

Sample titles (first 10):
['EC control 1', 'EC control 2', 'EC control 3', 'EC control 4', 'EC control 5', 'EC control 6', 'EC control 7', 'EC control 8', 'EC control 9', 'EC control 10']

Unique sources:
source
brain, Entorhinal Cortex             23
brain, Superior frontal gyrus        23
brain, Primary visual cortex         19
brain, Middle temporal gyrus         16
brain, hippocampus                   13
brain, Medial Temporal Gyrus         12
brain, Primary Visual Cortex         12
brain, Superior Frontal Gyrus        11
brain, Hippocampus                   10
brain, Posterior cingulate cortex     9
brain, Posterior cingulate            7
brain, Posterior Cingulate            6
Name: count, dtype: int64
/opt/conda/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_c

Step 2b: Extract expression matrix and disease status from GSE5281

In [65]:

import GEOparse
import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE5281
gse = GEOparse.get_GEO(geo="GSE5281", destdir="/mnt/results/tmp", silent=True)

# ── Parse sample metadata ──────────────────────────────────────────────────
meta_rows = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata.get('title', [''])[0]
    source = gsm.metadata.get('source_name_ch1', [''])[0]
    
    # Determine disease status from title
    title_lower = title.lower()
    if 'control' in title_lower or 'normal' in title_lower:
        condition = 'control'
    elif 'ad' in title_lower or 'alzheimer' in title_lower:
        condition = 'AD'
    else:
        condition = 'unknown'
    
    # Normalize brain region
    src_lower = source.lower()
    if 'entorhinal' in src_lower:
        region = 'Entorhinal_Cortex'
    elif 'frontal' in src_lower:
        region = 'Frontal_Cortex'
    elif 'hippocampus' in src_lower or 'hippocampal' in src_lower:
        region = 'Hippocampus'
    elif 'temporal' in src_lower:
        region = 'Temporal_Gyrus'
    elif 'visual' in src_lower:
        region = 'Visual_Cortex'
    elif 'cingulate' in src_lower:
        region = 'Cingulate_Cortex'
    else:
        region = 'Other'
    
    meta_rows.append({
        'sample_id': gsm_name,
        'title': title,
        'condition': condition,
        'region': region,
        'source_raw': source,
    })

meta_df = pd.DataFrame(meta_rows)
print("Condition counts:")
print(meta_df['condition'].value_counts())
print("\nRegion counts:")
print(meta_df['region'].value_counts())
print("\nCondition × Region:")
print(pd.crosstab(meta_df['region'], meta_df['condition']))

# ── Extract expression matrix ──────────────────────────────────────────────
# GSE5281 uses Affymetrix HG-U133 Plus 2.0 — probe-level data
# Get pivot table: probes × samples
print("\nExtracting expression data from GSMs...")
expr_dict = {}
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and not gsm.table.empty:
        # VALUE column = normalized expression (MAS5 or RMA)
        if 'VALUE' in gsm.table.columns:
            expr_dict[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']

expr_df = pd.DataFrame(expr_dict)
print(f"Expression matrix: {expr_df.shape[0]} probes × {expr_df.shape[1]} samples")
print(f"Missing values: {expr_df.isnull().sum().sum()}")

# ── Map probes to gene symbols via GPL ────────────────────────────────────
gpl = list(gse.gpls.values())[0]
print(f"\nPlatform: {gpl.name}")
print(f"GPL table columns: {gpl.table.columns.tolist()[:10]}")
print(f"GPL table shape: {gpl.table.shape}")


Condition counts:
condition
unknown    87
control    74
Name: count, dtype: int64

Region counts:
region
Frontal_Cortex       34
Visual_Cortex        31
Temporal_Gyrus       28
Entorhinal_Cortex    23
Hippocampus          23
Cingulate_Cortex     22
Name: count, dtype: int64

Condition × Region:
condition          control  unknown
region                             
Cingulate_Cortex        13        9
Entorhinal_Cortex       13       10
Frontal_Cortex          11       23
Hippocampus             13       10
Temporal_Gyrus          12       16
Visual_Cortex           12       19

Extracting expression data from GSMs...
Expression matrix: 54675 probes × 161 samples
Missing values: 0

Platform: GPL570
GPL table columns: ['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title']
GPL table shape: (54675, 16)
/opt/conda/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: Dtype

Step 2b: Fix condition labeling - inspect AD sample titles

In [67]:

import GEOparse
import pandas as pd
import numpy as np

np.random.seed(42)

# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE5281
gse = GEOparse.get_GEO(geo="GSE5281", destdir="/mnt/results/tmp", silent=True)

# Inspect all unique titles to understand naming convention
titles = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata.get('title', [''])[0]
    titles.append((gsm_name, title))

title_df = pd.DataFrame(titles, columns=['sample_id','title'])
print("All unique title patterns:")
# Show first word of each title to understand structure
title_df['prefix'] = title_df['title'].str.split().str[:3].str.join(' ')
print(title_df['prefix'].value_counts().head(30))


All unique title patterns:
prefix
EC control 1      1
EC control 2      1
EC control 3      1
EC control 4      1
EC control 5      1
EC control 6      1
EC control 7      1
EC control 8      1
EC control 9      1
EC control 10     1
EC control 11     1
EC control 12     1
EC control 13     1
HIP control 1     1
HIP control 2     1
HIP control 3     1
HIP control 4     1
HIP control 5     1
HIP control 6     1
HIP control 7     1
HIP control 8     1
HIP control 9     1
HIP control 10    1
HIP control 11    1
HIP control 12    1
HIP control 13    1
MTG control 1     1
MTG control 2     1
MTG control 3     1
MTG control 4     1
Name: count, dtype: int64
/opt/conda/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Inspect all unique title prefixes to find AD sample naming

In [69]:

import GEOparse
import pandas as pd
import numpy as np

np.random.seed(42)

gse = GEOparse.get_GEO(geo="GSE5281", destdir="/mnt/results/tmp", silent=True)

titles = [(gsm_name, gsm.metadata.get('title', [''])[0])
          for gsm_name, gsm in gse.gsms.items()]
title_df = pd.DataFrame(titles, columns=['sample_id','title'])

# Show ALL unique titles
print("All titles:")
for t in sorted(title_df['title'].unique()):
    print(f"  '{t}'")


All titles:
  'EC control 1'
  'EC control 10'
  'EC control 11'
  'EC control 12'
  'EC control 13'
  'EC control 2'
  'EC control 3'
  'EC control 4'
  'EC control 5'
  'EC control 6'
  'EC control 7'
  'EC control 8'
  'EC control 9'
  'EC_affected_1'
  'EC_affected_10'
  'EC_affected_2'
  'EC_affected_3'
  'EC_affected_4'
  'EC_affected_5'
  'EC_affected_6'
  'EC_affected_7'
  'EC_affected_8'
  'EC_affected_9'
  'HIP control 1'
  'HIP control 10'
  'HIP control 11'
  'HIP control 12'
  'HIP control 13'
  'HIP control 2'
  'HIP control 3'
  'HIP control 4'
  'HIP control 5'
  'HIP control 6'
  'HIP control 7'
  'HIP control 8'
  'HIP control 9'
  'HIP_affected_1'
  'HIP_affected_10'
  'HIP_affected_2'
  'HIP_affected_3'
  'HIP_affected_4'
  'HIP_affected_5'
  'HIP_affected_6'
  'HIP_affected_7'
  'HIP_affected_8'
  'HIP_affected_9'
  'MTG control 1'
  'MTG control 10'
  'MTG control 11'
  'MTG control 12'
  'MTG control 2'
  'MTG control 3'
  'MTG control 4'
  'MTG control 5'
  'MTG

Step 2c: Build expression matrix + metadata, map probes to gene symbols, save for R

In [71]:

import GEOparse
import pandas as pd
import numpy as np

np.random.seed(42)

# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE5281
gse = GEOparse.get_GEO(geo="GSE5281", destdir="/mnt/results/tmp", silent=True)

# ── Parse metadata with correct condition labels ───────────────────────────
region_map = {'EC':'Entorhinal_Cortex','HIP':'Hippocampus','MTG':'Middle_Temporal_Gyrus',
              'PC':'Posterior_Cingulate','SFG':'Superior_Frontal_Gyrus','VCX':'Visual_Cortex'}

meta_rows = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata.get('title', [''])[0]
    prefix = title.split('_')[0].split(' ')[0]
    condition = 'AD' if 'affected' in title else 'control'
    region = region_map.get(prefix, 'Unknown')
    meta_rows.append({'sample_id': gsm_name, 'title': title,
                      'condition': condition, 'region': region})

meta_df = pd.DataFrame(meta_rows)
print("Condition × Region:")
print(pd.crosstab(meta_df['region'], meta_df['condition']))

# ── Build expression matrix ────────────────────────────────────────────────
expr_dict = {}
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and 'VALUE' in gsm.table.columns:
        expr_dict[gsm_name] = pd.to_numeric(
            gsm.table.set_index('ID_REF')['VALUE'], errors='coerce')

expr_df = pd.DataFrame(expr_dict)
print(f"\nExpression matrix: {expr_df.shape[0]} probes × {expr_df.shape[1]} samples")

# ── Map probes to gene symbols via GPL570 ─────────────────────────────────
gpl = list(gse.gpls.values())[0]
probe_map = gpl.table[['ID','Gene Symbol']].copy()
probe_map.columns = ['probe_id','gene_symbol']
probe_map = probe_map.dropna(subset=['gene_symbol'])
probe_map = probe_map[probe_map['gene_symbol'].str.strip() != '']
# Some probes map to multiple genes (e.g. "GENE1 /// GENE2") — keep first
probe_map['gene_symbol'] = probe_map['gene_symbol'].str.split(' /// ').str[0].str.strip()
probe_map = probe_map[probe_map['gene_symbol'].str.match(r'^[A-Za-z]')]
print(f"Probes with gene symbol: {len(probe_map)}")

# Merge expression with gene symbols
expr_df.index.name = 'probe_id'
expr_annotated = expr_df.reset_index().merge(probe_map, on='probe_id', how='inner')
print(f"Annotated probes: {len(expr_annotated)}")

# For duplicate genes: keep probe with highest mean expression (most informative)
sample_cols = [c for c in expr_annotated.columns if c.startswith('GSM')]
expr_annotated['mean_expr'] = expr_annotated[sample_cols].mean(axis=1)
expr_gene = (expr_annotated
             .sort_values('mean_expr', ascending=False)
             .drop_duplicates(subset='gene_symbol', keep='first')
             .set_index('gene_symbol')[sample_cols])
print(f"Unique genes after deduplication: {len(expr_gene)}")

# ── Data quality checks ────────────────────────────────────────────────────
# Check for duplicate sample IDs
dup_cols = expr_gene.columns[expr_gene.columns.duplicated()].tolist()
print(f"Duplicate sample columns: {dup_cols}")

# Check missing values
missing_pct = expr_gene.isnull().sum().sum() / expr_gene.size * 100
print(f"Missing values: {missing_pct:.2f}%")

# Align metadata to expression columns
common_samples = list(set(expr_gene.columns) & set(meta_df['sample_id']))
meta_df = meta_df.set_index('sample_id').loc[common_samples].reset_index()
expr_gene = expr_gene[common_samples]
print(f"\nFinal: {expr_gene.shape[1]} samples, {expr_gene.shape[0]} genes")
print(f"AD: {(meta_df['condition']=='AD').sum()}, Control: {(meta_df['condition']=='control').sum()}")

# Save for R
expr_gene.to_csv('/mnt/results/tmp/gse5281_expr_matrix.csv')
meta_df.to_csv('/mnt/results/tmp/gse5281_metadata.csv', index=False)
print("\nSaved expression matrix and metadata for R analysis")


Condition × Region:
condition               AD  control
region                             
Entorhinal_Cortex       10       13
Hippocampus             10       13
Middle_Temporal_Gyrus   16       12
Posterior_Cingulate      9       13
Superior_Frontal_Gyrus  23       11
Visual_Cortex           19       12

Expression matrix: 54675 probes × 161 samples
Probes with gene symbol: 45782
Annotated probes: 45782
Unique genes after deduplication: 22880
Duplicate sample columns: []
Missing values: 0.00%

Final: 161 samples, 22880 genes
AD: 87, Control: 74

Saved expression matrix and metadata for R analysis
/opt/conda/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


Step 2d: Run limma DE analysis per brain region (AD vs control), subset to DDR genes

In [73]:

# Source: GSE5281 — Liang et al. 2008, Physiological Genomics
# Microarray data (Affymetrix HG-U133 Plus 2.0) — use limma, not DESeq2
set.seed(42)

library(limma)
library(dplyr)
library(tidyr)

# Load data
expr <- read.csv('/mnt/results/tmp/gse5281_expr_matrix.csv', row.names=1, check.names=FALSE)
meta <- read.csv('/mnt/results/tmp/gse5281_metadata.csv')
ddr  <- read.csv('/mnt/results/ddr_gene_universe.csv')

cat("Expression matrix:", nrow(expr), "genes x", ncol(expr), "samples\n")
cat("Metadata:", nrow(meta), "samples\n")
cat("DDR genes:", nrow(ddr), "\n")

# Align samples
common <- intersect(colnames(expr), meta$sample_id)
expr <- expr[, common]
meta <- meta[match(common, meta$sample_id), ]
cat("Aligned samples:", length(common), "\n")

# Expression values are already log2-transformed (Affymetrix MAS5/RMA)
# Verify: check range
cat("Expression range:", round(min(expr, na.rm=TRUE),2), "to", round(max(expr, na.rm=TRUE),2), "\n")

# Run limma per brain region
regions <- unique(meta$region)
cat("\nBrain regions:", paste(regions, collapse=", "), "\n")

all_results <- list()

for (reg in regions) {
  idx <- which(meta$region == reg)
  expr_reg <- expr[, idx]
  meta_reg <- meta[idx, ]
  
  n_ad  <- sum(meta_reg$condition == "AD")
  n_ctl <- sum(meta_reg$condition == "control")
  cat(sprintf("\n%s: %d AD, %d control\n", reg, n_ad, n_ctl))
  
  if (n_ad < 3 || n_ctl < 3) {
    cat("  Skipping — insufficient samples\n")
    next
  }
  
  # Design matrix: condition (AD vs control)
  condition <- factor(meta_reg$condition, levels=c("control","AD"))
  design <- model.matrix(~condition)
  
  # Fit limma model
  fit <- lmFit(as.matrix(expr_reg), design)
  fit <- eBayes(fit)
  
  # Extract results for AD vs control
  res <- topTable(fit, coef="conditionAD", number=Inf, sort.by="none")
  res$gene_symbol <- rownames(res)
  res$region <- reg
  res$n_ad <- n_ad
  res$n_control <- n_ctl
  
  # Rename columns to match expected output
  res <- res %>% rename(log2FC = logFC, padj = adj.P.Val, pvalue = P.Value)
  
  sig <- res %>% filter(padj <= 0.05 & abs(log2FC) >= 0.5)
  cat(sprintf("  Significant DEGs (padj<=0.05, |log2FC|>=0.5): %d\n", nrow(sig)))
  
  all_results[[reg]] <- res
}

# Combine all regions
all_de <- bind_rows(all_results)
cat("\nTotal DE results across all regions:", nrow(all_de), "\n")

# Subset to DDR gene universe
ddr_genes <- unique(ddr$gene_symbol)
ddr_de <- all_de %>% filter(gene_symbol %in% ddr_genes)
cat("DDR genes with DE results:", nrow(ddr_de), "\n")

# Significant DDR DEGs
ddr_sig <- ddr_de %>% filter(padj <= 0.05 & abs(log2FC) >= 0.5)
cat("Significant DDR DEGs (padj<=0.05, |log2FC|>=0.5):", nrow(ddr_sig), "\n")

# Save results
write.csv(all_de, '/mnt/results/tmp/gse5281_all_de_results.csv', row.names=FALSE)
write.csv(ddr_de, '/mnt/results/ddr_brain_degs.csv', row.names=FALSE)

cat("\nTop significant DDR DEGs:\n")
print(ddr_sig %>% arrange(padj) %>% 
      select(gene_symbol, region, log2FC, padj, n_ad, n_control) %>% 
      head(20))


Expression matrix: 22880 genes x 161 samples
Metadata: 161 samples
DDR genes: 805 
Aligned samples: 161 
Expression range: 0.07 to 143749.8 

Brain regions: Superior_Frontal_Gyrus, Visual_Cortex, Posterior_Cingulate, Middle_Temporal_Gyrus, Hippocampus, Entorhinal_Cortex 

Superior_Frontal_Gyrus: 23 AD, 11 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 8848

Visual_Cortex: 19 AD, 12 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 376

Posterior_Cingulate: 9 AD, 13 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 5989

Middle_Temporal_Gyrus: 16 AD, 12 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 4302

Hippocampus: 10 AD, 13 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 3898

Entorhinal_Cortex: 10 AD, 13 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 2899

Total DE results across all regions: 137280 
DDR genes with DE results: 4296 
Significant DDR DEGs (padj<=0.05, |log2FC|>=0.5): 1116 

Top significant DDR DEGs:
            gene_symb

Step 2d (fixed): Log2-transform, re-run limma per brain region, fix gene name artifacts

In [75]:

# Source: GSE5281 — Liang et al. 2008, Physiological Genomics
set.seed(42)
library(limma)
library(dplyr)

# Load data
expr <- read.csv('/mnt/results/tmp/gse5281_expr_matrix.csv', row.names=1, check.names=FALSE)
meta <- read.csv('/mnt/results/tmp/gse5281_metadata.csv')
ddr  <- read.csv('/mnt/results/ddr_gene_universe.csv')

# Align samples
common <- intersect(colnames(expr), meta$sample_id)
expr <- expr[, common]
meta <- meta[match(common, meta$sample_id), ]

# ── Log2 transform (raw Affymetrix signal intensities) ────────────────────
# Add 1 to handle any near-zero values before log2
expr_log2 <- log2(as.matrix(expr) + 1)
cat("After log2 transform — range:", round(min(expr_log2),2), "to", round(max(expr_log2),2), "\n")
cat("Expected range for Affymetrix: ~3 to ~17 log2 units\n\n")

# ── Run limma per brain region ─────────────────────────────────────────────
regions <- unique(meta$region)
all_results <- list()

for (reg in regions) {
  idx <- which(meta$region == reg)
  expr_reg <- expr_log2[, idx]
  meta_reg <- meta[idx, ]
  
  n_ad  <- sum(meta_reg$condition == "AD")
  n_ctl <- sum(meta_reg$condition == "control")
  cat(sprintf("%s: %d AD, %d control\n", reg, n_ad, n_ctl))
  
  if (n_ad < 3 || n_ctl < 3) { cat("  Skipping\n"); next }
  
  condition <- factor(meta_reg$condition, levels=c("control","AD"))
  design <- model.matrix(~condition)
  
  fit <- lmFit(expr_reg, design)
  fit <- eBayes(fit)
  
  res <- topTable(fit, coef="conditionAD", number=Inf, sort.by="none")
  # Fix: gene_symbol from rownames BEFORE bind_rows to avoid R auto-suffix
  res$gene_symbol <- rownames(res)
  res$region      <- reg
  res$n_ad        <- n_ad
  res$n_control   <- n_ctl
  rownames(res)   <- NULL  # prevent auto-suffix on bind_rows
  
  res <- res %>% rename(log2FC=logFC, padj=adj.P.Val, pvalue=P.Value)
  
  sig <- res %>% filter(padj <= 0.05 & abs(log2FC) >= 0.5)
  cat(sprintf("  Significant DEGs (padj<=0.05, |log2FC|>=0.5): %d\n", nrow(sig)))
  
  all_results[[reg]] <- res
}

# Combine — reset row names to prevent ...N artifacts
all_de <- bind_rows(all_results)
rownames(all_de) <- NULL

cat("\nTotal DE results:", nrow(all_de), "\n")
cat("Gene symbol artifacts (containing '...'):", sum(grepl("\\.\\.\\.\\d+$", all_de$gene_symbol)), "\n")

# ── Subset to DDR gene universe ────────────────────────────────────────────
ddr_genes <- unique(ddr$gene_symbol)
ddr_de <- all_de %>% filter(gene_symbol %in% ddr_genes)
ddr_sig <- ddr_de %>% filter(padj <= 0.05 & abs(log2FC) >= 0.5)

cat("\nDDR genes with DE results:", nrow(ddr_de), "\n")
cat("Significant DDR DEGs:", nrow(ddr_sig), "\n")

# Save
write.csv(all_de,  '/mnt/results/tmp/gse5281_all_de_results.csv', row.names=FALSE)
write.csv(ddr_de,  '/mnt/results/ddr_brain_degs.csv', row.names=FALSE)

cat("\nTop significant DDR DEGs (curated pathways only):\n")
ddr_curated <- read.csv('/mnt/results/ddr_gene_universe_curated.csv')
ddr_sig_curated <- ddr_sig %>% filter(gene_symbol %in% ddr_curated$gene_symbol)
cat("Significant curated DDR DEGs:", nrow(ddr_sig_curated), "\n")
print(ddr_sig_curated %>% arrange(padj) %>%
      select(gene_symbol, region, log2FC, padj, n_ad, n_control) %>% head(25))


After log2 transform <e2><80><94> range: 0.09 to 17.13 
Expected range for Affymetrix: ~3 to ~17 log2 units

Superior_Frontal_Gyrus: 23 AD, 11 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 2752
Visual_Cortex: 19 AD, 12 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 0
Posterior_Cingulate: 9 AD, 13 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 5910
Middle_Temporal_Gyrus: 16 AD, 12 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 6046
Hippocampus: 10 AD, 13 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 3428
Entorhinal_Cortex: 10 AD, 13 control
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 3764

Total DE results: 137280 
Gene symbol artifacts (containing '...'): 0 

DDR genes with DE results: 4296 
Significant DDR DEGs: 816 

Top significant DDR DEGs (curated pathways only):
Significant curated DDR DEGs: 98 
   gene_symbol                region    log2FC         padj n_ad n_control
1         RPA3 Middle_Temporal_Gyrus -2.688081 2.487028e-05  

Step 2e: Generate DDR brain heatmap and volcano plot

In [77]:

set.seed(42)
library(limma)
library(dplyr)
library(ggplot2)
library(ggrepel)
library(tidyr)
library(RColorBrewer)

# Load results
ddr_de  <- read.csv('/mnt/results/ddr_brain_degs.csv')
ddr_cur <- read.csv('/mnt/results/ddr_gene_universe_curated.csv')
meta    <- read.csv('/mnt/results/tmp/gse5281_metadata.csv')
expr    <- read.csv('/mnt/results/tmp/gse5281_expr_matrix.csv', row.names=1, check.names=FALSE)
expr_log2 <- log2(as.matrix(expr) + 1)

# ── 1. HEATMAP: curated DDR genes significant in ≥1 region ────────────────
ddr_sig_curated <- ddr_de %>%
  filter(gene_symbol %in% ddr_cur$gene_symbol,
         padj <= 0.05, abs(log2FC) >= 0.5)

# Genes significant in ≥2 regions for cleaner heatmap
gene_region_counts <- ddr_sig_curated %>%
  group_by(gene_symbol) %>%
  summarise(n_regions = n_distinct(region)) %>%
  filter(n_regions >= 1) %>%
  arrange(desc(n_regions))

heatmap_genes <- gene_region_counts$gene_symbol
cat("Genes for heatmap:", length(heatmap_genes), "\n")

# Build log2FC matrix: genes × regions
lfc_wide <- ddr_sig_curated %>%
  filter(gene_symbol %in% heatmap_genes) %>%
  select(gene_symbol, region, log2FC) %>%
  pivot_wider(names_from=region, values_from=log2FC, values_fill=0) %>%
  as.data.frame()
rownames(lfc_wide) <- lfc_wide$gene_symbol
lfc_wide <- lfc_wide[, -1]

# Order genes by mean log2FC
lfc_wide <- lfc_wide[order(rowMeans(lfc_wide)), ]
cat("Heatmap matrix:", nrow(lfc_wide), "genes x", ncol(lfc_wide), "regions\n")

# Add pathway annotation
pathway_ann <- ddr_cur %>% select(gene_symbol, pathway) %>%
  filter(gene_symbol %in% rownames(lfc_wide))
pathway_colors <- c(HR="#E41A1C", NHEJ="#377EB8", MMR="#4DAF4A",
                    BER="#984EA3", NER="#FF7F00", ATM_ATR="#A65628", FA="#F781BF")

# Plot heatmap using ggplot2
lfc_long <- lfc_wide %>%
  tibble::rownames_to_column("gene_symbol") %>%
  pivot_longer(-gene_symbol, names_to="region", values_to="log2FC") %>%
  left_join(ddr_cur %>% select(gene_symbol, pathway), by="gene_symbol")

# Order genes
gene_order <- rownames(lfc_wide)
region_order <- c("Entorhinal_Cortex","Hippocampus","Middle_Temporal_Gyrus",
                  "Posterior_Cingulate","Superior_Frontal_Gyrus","Visual_Cortex")
lfc_long$gene_symbol <- factor(lfc_long$gene_symbol, levels=gene_order)
lfc_long$region <- factor(lfc_long$region, levels=region_order)

# Clamp log2FC for color scale
lfc_long$log2FC_clamped <- pmax(pmin(lfc_long$log2FC, 3), -3)

p_heat <- ggplot(lfc_long, aes(x=region, y=gene_symbol, fill=log2FC_clamped)) +
  geom_tile(color="white", linewidth=0.2) +
  scale_fill_gradient2(low="#2166AC", mid="white", high="#D6604D",
                       midpoint=0, limits=c(-3,3),
                       name="log2FC\n(AD vs ctrl)") +
  scale_x_discrete(labels=c("Entorhinal\nCortex","Hippocampus","Middle\nTemporal",
                             "Posterior\nCingulate","Superior\nFrontal","Visual\nCortex")) +
  labs(title="DDR Gene Expression in AD Brain (GSE5281)",
       subtitle="Curated DDR genes, padj≤0.05 & |log2FC|≥0.5 in ≥1 region",
       x="Brain Region", y="Gene Symbol",
       caption="Source: GSE5281 (Liang et al. 2008); limma; N(AD)=87, N(ctrl)=74") +
  theme_classic(base_size=9) +
  theme(axis.text.y=element_text(size=6.5),
        axis.text.x=element_text(size=8, angle=0),
        plot.title=element_text(size=10, face="bold"),
        legend.key.height=unit(0.8,"cm"))

ggsave('/mnt/results/ddr_brain_heatmap.svg', p_heat, width=8, height=max(6, nrow(lfc_wide)*0.18+2), dpi=150)
ggsave('/mnt/results/ddr_brain_heatmap.png', p_heat, width=8, height=max(6, nrow(lfc_wide)*0.18+2), dpi=150)
cat("Saved: ddr_brain_heatmap.svg/.png\n")

# ── 2. VOLCANO PLOT: Middle Temporal Gyrus (most significant DDR hits) ─────
all_de <- read.csv('/mnt/results/tmp/gse5281_all_de_results.csv')
mtg_de <- all_de %>% filter(region == "Middle_Temporal_Gyrus")

# Label DDR genes
mtg_de$is_ddr_curated <- mtg_de$gene_symbol %in% ddr_cur$gene_symbol
mtg_de$is_ddr_sig <- mtg_de$gene_symbol %in%
  (ddr_sig_curated %>% filter(region=="Middle_Temporal_Gyrus") %>% pull(gene_symbol))

# -log10(padj), handle zeros
mtg_de$neg_log10_padj <- -log10(pmax(mtg_de$padj, 1e-15))

# Color categories
mtg_de$color_cat <- "Other"
mtg_de$color_cat[mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5] <- "Significant"
mtg_de$color_cat[mtg_de$is_ddr_curated & mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5] <- "DDR (significant)"
mtg_de$color_cat[mtg_de$is_ddr_curated & !(mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5)] <- "DDR (not significant)"

# Label top DDR genes
label_genes <- mtg_de %>%
  filter(is_ddr_curated, padj <= 0.05, abs(log2FC) >= 0.5) %>%
  arrange(padj) %>% head(15) %>% pull(gene_symbol)

mtg_de$label <- ifelse(mtg_de$gene_symbol %in% label_genes, mtg_de$gene_symbol, "")

color_vals <- c("Other"="grey80","Significant"="grey50",
                "DDR (significant)"="#D6604D","DDR (not significant)"="#4393C3")
size_vals  <- c("Other"=0.5,"Significant"=0.7,
                "DDR (significant)"=2.0,"DDR (not significant)"=1.5)

p_volc <- ggplot(mtg_de, aes(x=log2FC, y=neg_log10_padj,
                              color=color_cat, size=color_cat)) +
  geom_point(alpha=0.6) +
  geom_text_repel(aes(label=label), size=2.8, max.overlaps=20,
                  fontface="bold", color="#8B0000",
                  box.padding=0.4, segment.size=0.3) +
  scale_color_manual(values=color_vals, name="Gene category") +
  scale_size_manual(values=size_vals, guide="none") +
  geom_vline(xintercept=c(-0.5,0.5), linetype="dashed", color="grey40", linewidth=0.4) +
  geom_hline(yintercept=-log10(0.05), linetype="dashed", color="grey40", linewidth=0.4) +
  labs(title="Volcano Plot: AD vs Control — Middle Temporal Gyrus",
       subtitle="DDR genes highlighted (curated core, padj≤0.05 & |log2FC|≥0.5)",
       x="log2 Fold Change (AD / Control)",
       y="-log10(padj)",
       caption="Source: GSE5281 (Liang et al. 2008); limma; N(AD)=16, N(ctrl)=12") +
  theme_classic(base_size=11) +
  theme(plot.title=element_text(size=12, face="bold"),
        legend.position="right")

ggsave('/mnt/results/volcano_ddr_brain.svg', p_volc, width=9, height=6.5, dpi=150)
ggsave('/mnt/results/volcano_ddr_brain.png', p_volc, width=9, height=6.5, dpi=150)
cat("Saved: volcano_ddr_brain.svg/.png\n")
cat("\nDone with Step 2 figures.\n")


Genes for heatmap: 55 
Heatmap matrix: 55 genes x 5 regions
Error: The package "svglite" is required to save as SVG. Warning: package 'ggplot2' was built under R version 4.4.3

Install svglite and regenerate both figures

In [79]:

install.packages("svglite", repos="https://cloud.r-project.org", quiet=TRUE)
install.packages("tibble", repos="https://cloud.r-project.org", quiet=TRUE)
library(svglite)
library(tibble)
library(dplyr)
library(ggplot2)
library(ggrepel)
library(tidyr)

set.seed(42)

# ── Reload data ────────────────────────────────────────────────────────────
ddr_de  <- read.csv('/mnt/results/ddr_brain_degs.csv')
ddr_cur <- read.csv('/mnt/results/ddr_gene_universe_curated.csv')
all_de  <- read.csv('/mnt/results/tmp/gse5281_all_de_results.csv')

ddr_sig_curated <- ddr_de %>%
  filter(gene_symbol %in% ddr_cur$gene_symbol,
         padj <= 0.05, abs(log2FC) >= 0.5)

# ── 1. HEATMAP ─────────────────────────────────────────────────────────────
lfc_wide <- ddr_sig_curated %>%
  select(gene_symbol, region, log2FC) %>%
  pivot_wider(names_from=region, values_from=log2FC, values_fill=0) %>%
  as.data.frame()
rownames(lfc_wide) <- lfc_wide$gene_symbol
lfc_wide <- lfc_wide[, -1]
lfc_wide <- lfc_wide[order(rowMeans(lfc_wide)), ]

region_order <- c("Entorhinal_Cortex","Hippocampus","Middle_Temporal_Gyrus",
                  "Posterior_Cingulate","Superior_Frontal_Gyrus","Visual_Cortex")
# Keep only columns present
region_order <- intersect(region_order, colnames(lfc_wide))
lfc_wide <- lfc_wide[, region_order, drop=FALSE]

lfc_long <- lfc_wide %>%
  rownames_to_column("gene_symbol") %>%
  pivot_longer(-gene_symbol, names_to="region", values_to="log2FC") %>%
  left_join(ddr_cur %>% select(gene_symbol, pathway), by="gene_symbol")

lfc_long$gene_symbol <- factor(lfc_long$gene_symbol, levels=rownames(lfc_wide))
lfc_long$region <- factor(lfc_long$region, levels=region_order)
lfc_long$log2FC_clamped <- pmax(pmin(lfc_long$log2FC, 3), -3)

region_labels <- c("Entorhinal\nCortex","Hippocampus","Middle\nTemporal",
                   "Posterior\nCingulate","Superior\nFrontal","Visual\nCortex")
names(region_labels) <- region_order

p_heat <- ggplot(lfc_long, aes(x=region, y=gene_symbol, fill=log2FC_clamped)) +
  geom_tile(color="white", linewidth=0.2) +
  scale_fill_gradient2(low="#2166AC", mid="white", high="#D6604D",
                       midpoint=0, limits=c(-3,3), name="log2FC\n(AD/ctrl)") +
  scale_x_discrete(labels=region_labels) +
  labs(title="DDR Gene Expression in AD Brain (GSE5281)",
       subtitle="Curated DDR genes significant in ≥1 region (padj≤0.05, |log2FC|≥0.5)",
       x="Brain Region", y=NULL,
       caption="Source: GSE5281 (Liang et al. 2008); limma; N(AD)=87, N(ctrl)=74") +
  theme_classic(base_size=9) +
  theme(axis.text.y=element_text(size=7),
        axis.text.x=element_text(size=8),
        plot.title=element_text(size=10, face="bold"),
        legend.key.height=unit(0.8,"cm"))

fig_h <- max(6, nrow(lfc_wide)*0.2 + 2.5)
ggsave('/mnt/results/ddr_brain_heatmap.png', p_heat, width=8, height=fig_h, dpi=150)
ggsave('/mnt/results/ddr_brain_heatmap.svg', p_heat, width=8, height=fig_h, dpi=150)
cat("Saved heatmap\n")

# ── 2. VOLCANO PLOT ────────────────────────────────────────────────────────
mtg_de <- all_de %>% filter(region == "Middle_Temporal_Gyrus")
mtg_de$neg_log10_padj <- -log10(pmax(mtg_de$padj, 1e-15))
mtg_de$is_ddr_curated <- mtg_de$gene_symbol %in% ddr_cur$gene_symbol

mtg_de$color_cat <- "Other"
mtg_de$color_cat[mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5] <- "Significant"
mtg_de$color_cat[mtg_de$is_ddr_curated & mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5] <- "DDR (significant)"
mtg_de$color_cat[mtg_de$is_ddr_curated & !(mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5)] <- "DDR (not sig)"

label_genes <- mtg_de %>%
  filter(is_ddr_curated, padj <= 0.05, abs(log2FC) >= 0.5) %>%
  arrange(padj) %>% head(15) %>% pull(gene_symbol)
mtg_de$label <- ifelse(mtg_de$gene_symbol %in% label_genes, mtg_de$gene_symbol, "")

color_vals <- c("Other"="grey80","Significant"="grey55",
                "DDR (significant)"="#D6604D","DDR (not sig)"="#4393C3")
size_vals  <- c("Other"=0.4,"Significant"=0.6,"DDR (significant)"=2.2,"DDR (not sig)"=1.4)

p_volc <- ggplot(mtg_de, aes(x=log2FC, y=neg_log10_padj, color=color_cat, size=color_cat)) +
  geom_point(alpha=0.55) +
  geom_text_repel(aes(label=label), size=2.8, max.overlaps=25,
                  fontface="bold", color="#8B0000",
                  box.padding=0.45, segment.size=0.3, min.segment.length=0.2) +
  scale_color_manual(values=color_vals, name="Gene category") +
  scale_size_manual(values=size_vals, guide="none") +
  geom_vline(xintercept=c(-0.5,0.5), linetype="dashed", color="grey40", linewidth=0.4) +
  geom_hline(yintercept=-log10(0.05), linetype="dashed", color="grey40", linewidth=0.4) +
  labs(title="Volcano Plot: AD vs Control — Middle Temporal Gyrus (GSE5281)",
       subtitle="Curated DDR genes highlighted; padj≤0.05 & |log2FC|≥0.5",
       x="log2 Fold Change (AD / Control)", y="-log10(padj)",
       caption="Source: GSE5281 (Liang et al. 2008); limma; N(AD)=16, N(ctrl)=12") +
  theme_classic(base_size=11) +
  theme(plot.title=element_text(size=12, face="bold"), legend.position="right")

ggsave('/mnt/results/volcano_ddr_brain.png', p_volc, width=9, height=6.5, dpi=150)
ggsave('/mnt/results/volcano_ddr_brain.svg', p_volc, width=9, height=6.5, dpi=150)
cat("Saved volcano plot\n")


Error: there is no package called 'svglite' also installing the dependency 'systemfonts'



Generate heatmap and volcano plot as PNG only (no svglite dependency)

In [81]:

library(tibble)
library(dplyr)
library(ggplot2)
library(ggrepel)
library(tidyr)

set.seed(42)

ddr_de  <- read.csv('/mnt/results/ddr_brain_degs.csv')
ddr_cur <- read.csv('/mnt/results/ddr_gene_universe_curated.csv')
all_de  <- read.csv('/mnt/results/tmp/gse5281_all_de_results.csv')

ddr_sig_curated <- ddr_de %>%
  filter(gene_symbol %in% ddr_cur$gene_symbol,
         padj <= 0.05, abs(log2FC) >= 0.5)

# ── 1. HEATMAP ─────────────────────────────────────────────────────────────
lfc_wide <- ddr_sig_curated %>%
  select(gene_symbol, region, log2FC) %>%
  pivot_wider(names_from=region, values_from=log2FC, values_fill=0) %>%
  as.data.frame()
rownames(lfc_wide) <- lfc_wide$gene_symbol
lfc_wide <- lfc_wide[, -1]
lfc_wide <- lfc_wide[order(rowMeans(lfc_wide)), ]

region_order <- c("Entorhinal_Cortex","Hippocampus","Middle_Temporal_Gyrus",
                  "Posterior_Cingulate","Superior_Frontal_Gyrus")
region_order <- intersect(region_order, colnames(lfc_wide))
lfc_wide <- lfc_wide[, region_order, drop=FALSE]

lfc_long <- lfc_wide %>%
  rownames_to_column("gene_symbol") %>%
  pivot_longer(-gene_symbol, names_to="region", values_to="log2FC") %>%
  left_join(ddr_cur %>% select(gene_symbol, pathway), by="gene_symbol")

lfc_long$gene_symbol <- factor(lfc_long$gene_symbol, levels=rownames(lfc_wide))
lfc_long$region <- factor(lfc_long$region, levels=region_order)
lfc_long$log2FC_clamped <- pmax(pmin(lfc_long$log2FC, 3), -3)

region_labels <- c("Entorhinal\nCortex","Hippocampus","Middle\nTemporal",
                   "Posterior\nCingulate","Superior\nFrontal")
names(region_labels) <- region_order

p_heat <- ggplot(lfc_long, aes(x=region, y=gene_symbol, fill=log2FC_clamped)) +
  geom_tile(color="white", linewidth=0.2) +
  scale_fill_gradient2(low="#2166AC", mid="white", high="#D6604D",
                       midpoint=0, limits=c(-3,3), name="log2FC\n(AD/ctrl)") +
  scale_x_discrete(labels=region_labels) +
  labs(title="DDR Gene Expression in AD Brain (GSE5281)",
       subtitle="Curated DDR genes significant in ≥1 region (padj≤0.05, |log2FC|≥0.5)",
       x="Brain Region", y=NULL,
       caption="Source: GSE5281 (Liang et al. 2008); limma; N(AD)=87, N(ctrl)=74") +
  theme_classic(base_size=9) +
  theme(axis.text.y=element_text(size=7),
        axis.text.x=element_text(size=8),
        plot.title=element_text(size=10, face="bold"),
        legend.key.height=unit(0.8,"cm"))

fig_h <- max(6, nrow(lfc_wide)*0.2 + 2.5)
ggsave('/mnt/results/ddr_brain_heatmap.png', p_heat, width=8, height=fig_h, dpi=150)
cat("Saved: ddr_brain_heatmap.png\n")

# ── 2. VOLCANO PLOT ────────────────────────────────────────────────────────
mtg_de <- all_de %>% filter(region == "Middle_Temporal_Gyrus")
mtg_de$neg_log10_padj <- -log10(pmax(mtg_de$padj, 1e-15))
mtg_de$is_ddr_curated <- mtg_de$gene_symbol %in% ddr_cur$gene_symbol

mtg_de$color_cat <- "Other"
mtg_de$color_cat[mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5] <- "Significant"
mtg_de$color_cat[mtg_de$is_ddr_curated & mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5] <- "DDR (significant)"
mtg_de$color_cat[mtg_de$is_ddr_curated & !(mtg_de$padj <= 0.05 & abs(mtg_de$log2FC) >= 0.5)] <- "DDR (not sig)"

label_genes <- mtg_de %>%
  filter(is_ddr_curated, padj <= 0.05, abs(log2FC) >= 0.5) %>%
  arrange(padj) %>% head(15) %>% pull(gene_symbol)
mtg_de$label <- ifelse(mtg_de$gene_symbol %in% label_genes, mtg_de$gene_symbol, "")

color_vals <- c("Other"="grey80","Significant"="grey55",
                "DDR (significant)"="#D6604D","DDR (not sig)"="#4393C3")
size_vals  <- c("Other"=0.4,"Significant"=0.6,"DDR (significant)"=2.2,"DDR (not sig)"=1.4)

p_volc <- ggplot(mtg_de, aes(x=log2FC, y=neg_log10_padj, color=color_cat, size=color_cat)) +
  geom_point(alpha=0.55) +
  geom_text_repel(aes(label=label), size=2.8, max.overlaps=25,
                  fontface="bold", color="#8B0000",
                  box.padding=0.45, segment.size=0.3, min.segment.length=0.2) +
  scale_color_manual(values=color_vals, name="Gene category") +
  scale_size_manual(values=size_vals, guide="none") +
  geom_vline(xintercept=c(-0.5,0.5), linetype="dashed", color="grey40", linewidth=0.4) +
  geom_hline(yintercept=-log10(0.05), linetype="dashed", color="grey40", linewidth=0.4) +
  labs(title="Volcano Plot: AD vs Control — Middle Temporal Gyrus (GSE5281)",
       subtitle="Curated DDR genes highlighted; padj≤0.05 & |log2FC|≥0.5",
       x="log2 Fold Change (AD / Control)", y="-log10(padj)",
       caption="Source: GSE5281 (Liang et al. 2008); limma; N(AD)=16, N(ctrl)=12") +
  theme_classic(base_size=11) +
  theme(plot.title=element_text(size=12, face="bold"), legend.position="right")

ggsave('/mnt/results/volcano_ddr_brain.png', p_volc, width=9, height=6.5, dpi=150)
cat("Saved: volcano_ddr_brain.png\n")


Saved: ddr_brain_heatmap.png
Saved: volcano_ddr_brain.pngWarning: ggrepel: 14 unlabeled data points (too many overlaps). Consider increasing max.overlaps

Fix Unicode characters in heatmap subtitle, resave

In [83]:

library(tibble); library(dplyr); library(ggplot2); library(tidyr)
set.seed(42)

ddr_de  <- read.csv('/mnt/results/ddr_brain_degs.csv')
ddr_cur <- read.csv('/mnt/results/ddr_gene_universe_curated.csv')

ddr_sig_curated <- ddr_de %>%
  filter(gene_symbol %in% ddr_cur$gene_symbol, padj <= 0.05, abs(log2FC) >= 0.5)

lfc_wide <- ddr_sig_curated %>%
  select(gene_symbol, region, log2FC) %>%
  pivot_wider(names_from=region, values_from=log2FC, values_fill=0) %>%
  as.data.frame()
rownames(lfc_wide) <- lfc_wide$gene_symbol
lfc_wide <- lfc_wide[, -1]
lfc_wide <- lfc_wide[order(rowMeans(lfc_wide)), ]

region_order <- intersect(c("Entorhinal_Cortex","Hippocampus","Middle_Temporal_Gyrus",
                             "Posterior_Cingulate","Superior_Frontal_Gyrus"), colnames(lfc_wide))
lfc_wide <- lfc_wide[, region_order, drop=FALSE]

lfc_long <- lfc_wide %>%
  rownames_to_column("gene_symbol") %>%
  pivot_longer(-gene_symbol, names_to="region", values_to="log2FC") %>%
  left_join(ddr_cur %>% select(gene_symbol, pathway), by="gene_symbol")

lfc_long$gene_symbol <- factor(lfc_long$gene_symbol, levels=rownames(lfc_wide))
lfc_long$region <- factor(lfc_long$region, levels=region_order)
lfc_long$log2FC_clamped <- pmax(pmin(lfc_long$log2FC, 3), -3)

region_labels <- setNames(
  c("Entorhinal\nCortex","Hippocampus","Middle\nTemporal","Posterior\nCingulate","Superior\nFrontal"),
  region_order)

# Use ASCII-safe subtitle
p_heat <- ggplot(lfc_long, aes(x=region, y=gene_symbol, fill=log2FC_clamped)) +
  geom_tile(color="white", linewidth=0.2) +
  scale_fill_gradient2(low="#2166AC", mid="white", high="#D6604D",
                       midpoint=0, limits=c(-3,3), name="log2FC\n(AD/ctrl)") +
  scale_x_discrete(labels=region_labels) +
  labs(title="DDR Gene Expression in AD Brain (GSE5281)",
       subtitle="Curated DDR genes significant in >=1 region (padj<=0.05, |log2FC|>=0.5)",
       x="Brain Region", y=NULL,
       caption="Source: GSE5281 (Liang et al. 2008); limma; N(AD)=87, N(ctrl)=74") +
  theme_classic(base_size=9) +
  theme(axis.text.y=element_text(size=7.5),
        axis.text.x=element_text(size=8.5),
        plot.title=element_text(size=10, face="bold"),
        legend.key.height=unit(0.8,"cm"))

fig_h <- max(6, nrow(lfc_wide)*0.2 + 2.5)
ggsave('/mnt/results/ddr_brain_heatmap.png', p_heat, width=8, height=fig_h, dpi=150)
cat("Heatmap resaved with ASCII-safe subtitle\n")
cat("Heatmap dimensions:", nrow(lfc_wide), "genes x", length(region_order), "regions\n")


Heatmap resaved with ASCII-safe subtitle
Heatmap dimensions: 55 genes x 5 regions

Step 3a: Download GSE63060 and GSE63061 (AddNeuroMed blood transcriptomics)

In [85]:

import GEOparse
import pandas as pd
import numpy as np
import os

np.random.seed(42)

# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE63060
# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE63061
# AddNeuroMed cohort — blood gene expression, AD vs MCI vs control
# Sims et al. 2014, Genome Biology

for acc in ["GSE63060", "GSE63061"]:
    print(f"\nDownloading {acc}...")
    gse = GEOparse.get_GEO(geo=acc, destdir="/mnt/results/tmp", silent=True)
    print(f"  Samples: {len(gse.gsms)}, Platforms: {len(gse.gpls)}")
    
    # Inspect first few sample titles and characteristics
    titles = []
    for gsm_name, gsm in list(gse.gsms.items())[:5]:
        title = gsm.metadata.get('title', [''])[0]
        chars = gsm.metadata.get('characteristics_ch1', [])
        titles.append((gsm_name, title, chars))
    
    print(f"  Sample examples:")
    for s, t, c in titles:
        print(f"    {s}: '{t}' | chars: {c[:3]}")



  Samples: 329, Platforms: 1
  Sample examples:
    GSM1539080: '4856050008_I' | chars: ['status: MCI', 'ethnicity: Western European', 'age: 65']
    GSM1539081: '4856050047_D' | chars: ['status: MCI', 'ethnicity: Other Caucasian', 'age: 66']
    GSM1539082: '4856076009_D' | chars: ['status: MCI', 'ethnicity: Western European', 'age: 67']
    GSM1539083: '4856076040_F' | chars: ['status: MCI', 'ethnicity: Unknown', 'age: 67']
    GSM1539084: '4856076002_E' | chars: ['status: MCI', 'ethnicity: Western European', 'age: 67']

  Samples: 388, Platforms: 1
  Sample examples:
    GSM1539409: '7196843065_F' | chars: ['status: MCI', 'ethnicity: Western European', 'age: 57']
    GSM1539410: '7196843076_G' | chars: ['status: MCI', 'ethnicity: Western European', 'age: 59']
    GSM1539411: '7196843068_B' | chars: ['status: MCI', 'ethnicity: Western European', 'age: 63']
    GSM1539412: '7196843063_B' | chars: ['status: MCI', 'ethnicity: Western European', 'age: 65']
    GSM1539413: '7196843065_L'

Step 3b: Extract expression matrices and metadata from GSE63060 and GSE63061

In [87]:

import GEOparse
import pandas as pd
import numpy as np
import re

np.random.seed(42)

# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE63060
# Source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE63061

def parse_characteristics(chars):
    """Parse GEO characteristics list into dict."""
    d = {}
    for c in chars:
        if ':' in c:
            k, v = c.split(':', 1)
            d[k.strip()] = v.strip()
    return d

def process_gse(acc):
    gse = GEOparse.get_GEO(geo=acc, destdir="/mnt/results/tmp", silent=True)
    
    # Parse metadata
    meta_rows = []
    for gsm_name, gsm in gse.gsms.items():
        chars = parse_characteristics(gsm.metadata.get('characteristics_ch1', []))
        meta_rows.append({
            'sample_id': gsm_name,
            'title': gsm.metadata.get('title', [''])[0],
            'status': chars.get('status', 'unknown'),
            'age': chars.get('age', np.nan),
            'sex': chars.get('Sex', chars.get('sex', 'unknown')),
            'ethnicity': chars.get('ethnicity', 'unknown'),
            'dataset': acc,
        })
    meta_df = pd.DataFrame(meta_rows)
    meta_df['age'] = pd.to_numeric(meta_df['age'], errors='coerce')
    
    # Extract expression
    expr_dict = {}
    for gsm_name, gsm in gse.gsms.items():
        if gsm.table is not None and 'VALUE' in gsm.table.columns:
            expr_dict[gsm_name] = pd.to_numeric(
                gsm.table.set_index('ID_REF')['VALUE'], errors='coerce')
    expr_df = pd.DataFrame(expr_dict)
    
    # Map probes to gene symbols
    gpl = list(gse.gpls.values())[0]
    print(f"\n{acc} platform: {gpl.name}")
    print(f"  GPL columns: {gpl.table.columns.tolist()[:8]}")
    
    # Illumina arrays use 'Symbol' or 'ILMN_Gene'
    sym_col = None
    for col in ['Symbol', 'ILMN_Gene', 'Gene Symbol', 'gene_symbol']:
        if col in gpl.table.columns:
            sym_col = col
            break
    print(f"  Gene symbol column: {sym_col}")
    
    if sym_col:
        probe_map = gpl.table[['ID', sym_col]].copy()
        probe_map.columns = ['probe_id', 'gene_symbol']
        probe_map = probe_map.dropna(subset=['gene_symbol'])
        probe_map = probe_map[probe_map['gene_symbol'].str.strip() != '']
        probe_map['gene_symbol'] = probe_map['gene_symbol'].str.strip()
        
        expr_df.index.name = 'probe_id'
        expr_ann = expr_df.reset_index().merge(probe_map, on='probe_id', how='inner')
        sample_cols = [c for c in expr_ann.columns if c.startswith('GSM')]
        expr_ann['mean_expr'] = expr_ann[sample_cols].mean(axis=1)
        expr_gene = (expr_ann.sort_values('mean_expr', ascending=False)
                     .drop_duplicates(subset='gene_symbol', keep='first')
                     .set_index('gene_symbol')[sample_cols])
    else:
        expr_gene = expr_df
    
    print(f"  Samples: {len(meta_df)}, Genes: {len(expr_gene)}")
    print(f"  Status distribution: {meta_df['status'].value_counts().to_dict()}")
    print(f"  Expression range: {expr_gene.values.min():.2f} to {expr_gene.values.max():.2f}")
    
    return expr_gene, meta_df

expr60, meta60 = process_gse("GSE63060")
expr61, meta61 = process_gse("GSE63061")

# Save
expr60.to_csv('/mnt/results/tmp/gse63060_expr.csv')
meta60.to_csv('/mnt/results/tmp/gse63060_meta.csv', index=False)
expr61.to_csv('/mnt/results/tmp/gse63061_expr.csv')
meta61.to_csv('/mnt/results/tmp/gse63061_meta.csv', index=False)
print("\nSaved both datasets.")



GSE63060 platform: GPL6947
  GPL columns: ['ID', 'nuID', 'Species', 'Source', 'Search_Key', 'Transcript', 'ILMN_Gene', 'Source_Reference_ID']
  Gene symbol column: Symbol
  Samples: 329, Genes: 21696
  Status distribution: {'AD': 145, 'CTL': 104, 'MCI': 80}
  Expression range: 7.20 to 15.05

GSE63061 platform: GPL10558
  GPL columns: ['ID', 'Species', 'Source', 'Search_Key', 'Transcript', 'ILMN_Gene', 'Source_Reference_ID', 'RefSeq_ID']
  Gene symbol column: Symbol
  Samples: 388, Genes: 22336
  Status distribution: {'AD': 139, 'CTL': 134, 'MCI': 109, 'borderline MCI': 3, 'OTHER': 1, 'CTL to AD': 1, 'MCI to CTL': 1}
  Expression range: 3.93 to 14.78

Saved both datasets.


Step 3c: Limma DE on blood data, T-cell exhaustion scoring, clinical correlations

In [89]:

# Source: GSE63060/GSE63061 — AddNeuroMed cohort (Sims et al. 2014, Genome Biology)
set.seed(42)
library(limma)
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggrepel)

# ── T-cell exhaustion + DDR gene signature ────────────────────────────────
tcell_exhaust_genes <- c(
  # Inhibitory receptors
  "PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244",
  # Exhaustion TFs
  "TOX","TOX2","NR4A1","NR4A2","EOMES","TBX21",
  # Effector dysfunction markers
  "IFNG","TNF","IL2","GZMB",
  # Terminal exhaustion
  "KLRG1","CX3CR1",
  # DDR in T cells
  "H2AX","ATM","TP53BP1","BRCA1","RAD51"
)

ddr_cur <- read.csv('/mnt/results/ddr_gene_universe_curated.csv')
ddr_genes <- unique(ddr_cur$gene_symbol)

# ── Process each dataset ──────────────────────────────────────────────────
run_blood_analysis <- function(acc, expr_file, meta_file) {
  cat(sprintf("\n=== %s ===\n", acc))
  
  expr <- read.csv(expr_file, row.names=1, check.names=FALSE)
  meta <- read.csv(meta_file)
  
  # Keep only AD and CTL (exclude MCI and ambiguous)
  meta_filt <- meta %>% filter(status %in% c("AD","CTL"))
  common <- intersect(colnames(expr), meta_filt$sample_id)
  expr_filt <- expr[, common]
  meta_filt <- meta_filt[match(common, meta_filt$sample_id), ]
  
  cat(sprintf("  AD: %d, CTL: %d\n", sum(meta_filt$status=="AD"), sum(meta_filt$status=="CTL")))
  
  # Data quality checks
  dup_cols <- sum(duplicated(colnames(expr_filt)))
  cat(sprintf("  Duplicate columns: %d\n", dup_cols))
  miss_pct <- sum(is.na(expr_filt)) / prod(dim(expr_filt)) * 100
  cat(sprintf("  Missing values: %.2f%%\n", miss_pct))
  
  # ── limma DE: AD vs CTL ──────────────────────────────────────────────────
  condition <- factor(meta_filt$status, levels=c("CTL","AD"))
  # Include age as covariate where available
  age_avail <- !is.na(meta_filt$age)
  if (sum(age_avail) > 0.8 * nrow(meta_filt)) {
    meta_filt$age[is.na(meta_filt$age)] <- median(meta_filt$age, na.rm=TRUE)
    design <- model.matrix(~condition + meta_filt$age)
    cat("  Covariates: condition + age\n")
  } else {
    design <- model.matrix(~condition)
    cat("  Covariates: condition only\n")
  }
  
  fit <- lmFit(as.matrix(expr_filt), design)
  fit <- eBayes(fit)
  res <- topTable(fit, coef="conditionAD", number=Inf, sort.by="none")
  res$gene_symbol <- rownames(res)
  rownames(res) <- NULL
  res <- res %>% rename(log2FC=logFC, padj=adj.P.Val, pvalue=P.Value)
  res$dataset <- acc
  
  sig <- res %>% filter(padj <= 0.05 & abs(log2FC) >= 0.5)
  cat(sprintf("  Significant DEGs (padj<=0.05, |log2FC|>=0.5): %d\n", nrow(sig)))
  
  # ── T-cell exhaustion score ───────────────────────────────────────────────
  # Mean normalized expression of exhaustion signature genes present in data
  exhaust_present <- intersect(tcell_exhaust_genes, rownames(expr_filt))
  cat(sprintf("  Exhaustion genes present: %d/%d\n", length(exhaust_present), length(tcell_exhaust_genes)))
  
  exhaust_scores <- colMeans(as.matrix(expr_filt[exhaust_present, ]), na.rm=TRUE)
  
  score_df <- data.frame(
    sample_id = names(exhaust_scores),
    exhaustion_score = as.numeric(exhaust_scores),
    dataset = acc
  ) %>% left_join(meta_filt %>% select(sample_id, status, age, sex), by="sample_id")
  
  # Wilcoxon test: AD vs CTL exhaustion score
  ad_scores  <- score_df$exhaustion_score[score_df$status == "AD"]
  ctl_scores <- score_df$exhaustion_score[score_df$status == "CTL"]
  wt <- wilcox.test(ad_scores, ctl_scores)
  
  # Cohen's d effect size
  pooled_sd <- sqrt((sd(ad_scores)^2 + sd(ctl_scores)^2) / 2)
  cohens_d  <- (mean(ad_scores) - mean(ctl_scores)) / pooled_sd
  
  cat(sprintf("  Exhaustion score — AD mean: %.3f, CTL mean: %.3f\n",
              mean(ad_scores), mean(ctl_scores)))
  cat(sprintf("  Wilcoxon p = %.4f, Cohen's d = %.3f\n", wt$p.value, cohens_d))
  
  list(de=res, scores=score_df, sig_n=nrow(sig),
       wilcox_p=wt$p.value, cohens_d=cohens_d,
       exhaust_genes=exhaust_present)
}

res60 <- run_blood_analysis("GSE63060",
  '/mnt/results/tmp/gse63060_expr.csv',
  '/mnt/results/tmp/gse63060_meta.csv')

res61 <- run_blood_analysis("GSE63061",
  '/mnt/results/tmp/gse63061_expr.csv',
  '/mnt/results/tmp/gse63061_meta.csv')

# ── Combine DE results ────────────────────────────────────────────────────
all_blood_de <- bind_rows(res60$de, res61$de)
rownames(all_blood_de) <- NULL

# Filter to T-cell exhaustion + DDR genes
tcell_ddr_de <- all_blood_de %>%
  filter(gene_symbol %in% c(tcell_exhaust_genes, ddr_genes))
sig_tcell_ddr <- tcell_ddr_de %>% filter(padj <= 0.05 & abs(log2FC) >= 0.5)

cat(sprintf("\nT-cell/DDR DEGs in blood (padj<=0.05, |log2FC|>=0.5): %d\n", nrow(sig_tcell_ddr)))
print(sig_tcell_ddr %>% arrange(padj) %>%
      select(gene_symbol, log2FC, padj, dataset) %>% head(20))

# Save DE results
write.csv(tcell_ddr_de, '/mnt/results/tcell_ddr_blood_degs.csv', row.names=FALSE)

# Combine exhaustion scores
all_scores <- bind_rows(res60$scores, res61$scores)
write.csv(all_scores, '/mnt/results/tcell_exhaustion_score_ad.csv', row.names=FALSE)
cat("\nSaved: tcell_ddr_blood_degs.csv, tcell_exhaustion_score_ad.csv\n")



=== GSE63060 ===
  AD: 145, CTL: 104
  Duplicate columns: 0
  Missing values: 0.00%
  Covariates: condition + age
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 76
  Exhaustion genes present: 19/23
  Exhaustion score <e2><80><94> AD mean: 8.612, CTL mean: 8.610
  Wilcoxon p = 0.8844, Cohen's d = 0.011

=== GSE63061 ===
  AD: 139, CTL: 134
  Duplicate columns: 0
  Missing values: 0.00%
  Covariates: condition + age
  Significant DEGs (padj<=0.05, |log2FC|>=0.5): 23
  Exhaustion genes present: 21/23
  Exhaustion score <e2><80><94> AD mean: 7.039, CTL mean: 7.025
  Wilcoxon p = 0.3445, Cohen's d = 0.141

T-cell/DDR DEGs in blood (padj<=0.05, |log2FC|>=0.5): 0
[1] gene_symbol log2FC      padj        dataset    
<0 rows> (or 0-length row.names)

Saved: tcell_ddr_blood_degs.csv, tcell_exhaustion_score_ad.csv

Step 3d: ssGSEA exhaustion scoring, individual gene inspection, boxplot, clinical correlations

In [91]:

set.seed(42)
library(limma)
library(dplyr)
library(ggplot2)
library(tidyr)

# ── Reload data ────────────────────────────────────────────────────────────
all_scores <- read.csv('/mnt/results/tcell_exhaustion_score_ad.csv')
all_blood_de60 <- read.csv('/mnt/results/tmp/gse63060_expr.csv', row.names=1, check.names=FALSE)
all_blood_de61 <- read.csv('/mnt/results/tmp/gse63061_expr.csv', row.names=1, check.names=FALSE)
meta60 <- read.csv('/mnt/results/tmp/gse63060_meta.csv')
meta61 <- read.csv('/mnt/results/tmp/gse63061_meta.csv')

# T-cell exhaustion gene sets — split into sub-signatures for richer analysis
exhaust_inhibitory <- c("PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244")
exhaust_tfs        <- c("TOX","TOX2","NR4A1","NR4A2","EOMES","TBX21")
exhaust_effector   <- c("IFNG","TNF","IL2","GZMB")
ddr_tcell          <- c("H2AX","ATM","TP53BP1","BRCA1","RAD51")
all_exhaust        <- c(exhaust_inhibitory, exhaust_tfs, exhaust_effector, ddr_tcell, "KLRG1","CX3CR1")

# ── Check which exhaustion genes are present and their DE status ───────────
tcell_ddr_all <- read.csv('/mnt/results/tcell_ddr_blood_degs.csv')

# Relax threshold to see near-significant hits
near_sig <- tcell_ddr_all %>%
  filter(gene_symbol %in% all_exhaust) %>%
  arrange(pvalue) %>%
  select(gene_symbol, log2FC, pvalue, padj, dataset)

cat("T-cell exhaustion/DDR genes in blood DE results (sorted by p-value):\n")
print(near_sig %>% head(30))

# ── Compute separate sub-scores per sample ────────────────────────────────
compute_scores <- function(expr_mat, meta_df, acc) {
  meta_filt <- meta_df %>% filter(status %in% c("AD","CTL"))
  common <- intersect(colnames(expr_mat), meta_filt$sample_id)
  expr_filt <- as.matrix(expr_mat[, common])
  meta_filt <- meta_filt[match(common, meta_filt$sample_id), ]
  
  score_row <- function(genes, label) {
    present <- intersect(genes, rownames(expr_filt))
    if (length(present) == 0) return(NULL)
    scores <- colMeans(expr_filt[present, , drop=FALSE], na.rm=TRUE)
    data.frame(sample_id=names(scores), score=as.numeric(scores),
               score_type=label, status=meta_filt$status,
               age=meta_filt$age, dataset=acc)
  }
  
  bind_rows(
    score_row(exhaust_inhibitory, "Inhibitory_Receptors"),
    score_row(exhaust_tfs,        "Exhaustion_TFs"),
    score_row(exhaust_effector,   "Effector_Dysfunction"),
    score_row(ddr_tcell,          "DDR_Tcell"),
    score_row(all_exhaust,        "Overall_Exhaustion")
  )
}

scores60 <- compute_scores(all_blood_de60, meta60, "GSE63060")
scores61 <- compute_scores(all_blood_de61, meta61, "GSE63061")
all_sub_scores <- bind_rows(scores60, scores61)

# Wilcoxon tests per score type per dataset
cat("\n\nWilcoxon tests per sub-score:\n")
wt_results <- all_sub_scores %>%
  group_by(score_type, dataset) %>%
  summarise(
    n_AD  = sum(status=="AD"),
    n_CTL = sum(status=="CTL"),
    mean_AD  = mean(score[status=="AD"], na.rm=TRUE),
    mean_CTL = mean(score[status=="CTL"], na.rm=TRUE),
    wilcox_p = tryCatch(wilcox.test(score[status=="AD"], score[status=="CTL"])$p.value, error=function(e) NA),
    cohens_d = (mean(score[status=="AD"],na.rm=TRUE) - mean(score[status=="CTL"],na.rm=TRUE)) /
               sqrt((var(score[status=="AD"],na.rm=TRUE) + var(score[status=="CTL"],na.rm=TRUE))/2),
    .groups="drop"
  ) %>%
  mutate(padj_BH = p.adjust(wilcox_p, method="BH"))

print(wt_results %>% select(score_type, dataset, n_AD, n_CTL, mean_AD, mean_CTL, wilcox_p, padj_BH, cohens_d) %>%
      arrange(wilcox_p))

# ── Boxplot: Overall exhaustion score by status ────────────────────────────
plot_data <- all_sub_scores %>%
  filter(score_type == "Overall_Exhaustion", status %in% c("AD","CTL")) %>%
  mutate(status = factor(status, levels=c("CTL","AD")),
         dataset_label = ifelse(dataset=="GSE63060","GSE63060\n(N=249)","GSE63061\n(N=273)"))

# Add p-value annotations
pvals <- wt_results %>%
  filter(score_type=="Overall_Exhaustion") %>%
  mutate(label = paste0("p=", formatC(wilcox_p, digits=3, format="g")))

p_box <- ggplot(plot_data, aes(x=status, y=score, fill=status)) +
  geom_boxplot(outlier.size=0.8, alpha=0.75, width=0.55) +
  geom_jitter(width=0.15, size=0.5, alpha=0.3) +
  facet_wrap(~dataset_label) +
  scale_fill_manual(values=c("CTL"="#4393C3","AD"="#D6604D"), guide="none") +
  labs(title="T-Cell Exhaustion Score in AD Blood (AddNeuroMed Cohort)",
       subtitle="Mean expression of 23-gene exhaustion signature; AD vs Control",
       x="Disease Status", y="Mean Exhaustion Score (log2 expression)",
       caption="Sources: GSE63060, GSE63061 (Sims et al. 2014); limma with age covariate") +
  theme_classic(base_size=11) +
  theme(plot.title=element_text(size=12, face="bold"),
        strip.text=element_text(size=10, face="bold"))

ggsave('/mnt/results/exhaustion_score_boxplot.png', p_box, width=8, height=5.5, dpi=150)
cat("\nSaved: exhaustion_score_boxplot.png\n")


T-cell exhaustion/DDR genes in blood DE results (sorted by p-value):
   gene_symbol       log2FC      pvalue       padj  dataset
1       HAVCR2  0.078172819 0.006196993 0.05915547 GSE63061
2       CX3CR1  0.118514798 0.011800217 0.09193221 GSE63061
3        CTLA4 -0.034455268 0.029851723 0.15307563 GSE63060
4        NR4A2  0.039695480 0.030178668 0.17182880 GSE63061
5        CTLA4 -0.048655497 0.031015969 0.17529673 GSE63061
6        KLRG1 -0.157869052 0.042366279 0.19617862 GSE63060
7      TP53BP1  0.036794298 0.044166700 0.20152276 GSE63060
8         IFNG -0.025525213 0.046682208 0.22691921 GSE63061
9        RAD51  0.034895360 0.068223752 0.29069930 GSE63061
10        LAG3  0.074809871 0.092562776 0.32459059 GSE63060
11         TNF  0.027669335 0.136969131 0.43034780 GSE63061
12       TBX21 -0.018370439 0.150235558 0.45206540 GSE63061
13      HAVCR2  0.057032040 0.157402707 0.44353218 GSE63060
14       TIGIT -0.023754076 0.158637063 0.46512211 GSE63061
15       KLRG1 -0.084769856 0.1

Step 3e: Clinical correlation heatmap and verify boxplot

In [93]:

set.seed(42)
library(dplyr); library(ggplot2); library(tidyr)

# ── Load scores and metadata ───────────────────────────────────────────────
scores <- read.csv('/mnt/results/tcell_exhaustion_score_ad.csv')
meta60 <- read.csv('/mnt/results/tmp/gse63060_meta.csv')
meta61 <- read.csv('/mnt/results/tmp/gse63061_meta.csv')

# Merge full metadata (including sex) back into scores
meta_all <- bind_rows(
  meta60 %>% mutate(dataset="GSE63060"),
  meta61 %>% mutate(dataset="GSE63061")
) %>% select(sample_id, status, age, sex, dataset)

scores_full <- scores %>%
  select(sample_id, exhaustion_score, dataset) %>%
  left_join(meta_all, by=c("sample_id","dataset")) %>%
  filter(status %in% c("AD","CTL"))

# Encode status as numeric for correlation (AD=1, CTL=0)
scores_full$status_num <- ifelse(scores_full$status=="AD", 1, 0)
scores_full$age_num    <- as.numeric(scores_full$age)

# ── Spearman correlations: exhaustion score vs clinical variables ──────────
# Per dataset
cor_results <- list()
for (ds in c("GSE63060","GSE63061")) {
  sub <- scores_full %>% filter(dataset==ds)
  
  vars <- list(
    "Disease Status (AD=1)" = sub$status_num,
    "Age" = sub$age_num
  )
  
  for (vname in names(vars)) {
    v <- vars[[vname]]
    valid <- !is.na(v) & !is.na(sub$exhaustion_score)
    if (sum(valid) < 10) next
    ct <- cor.test(sub$exhaustion_score[valid], v[valid], method="spearman")
    cor_results[[paste(ds, vname)]] <- data.frame(
      dataset=ds, variable=vname,
      spearman_r=ct$estimate, pvalue=ct$p.value,
      n=sum(valid)
    )
  }
}

cor_df <- bind_rows(cor_results)
cor_df$padj <- p.adjust(cor_df$pvalue, method="bonferroni")
cat("Spearman correlations (exhaustion score vs clinical variables):\n")
print(cor_df %>% select(dataset, variable, spearman_r, pvalue, padj, n))

# ── Correlation heatmap ────────────────────────────────────────────────────
cor_df$sig_label <- ifelse(cor_df$padj <= 0.05, "*", 
                    ifelse(cor_df$pvalue <= 0.05, "†", ""))
cor_df$r_label <- paste0(round(cor_df$spearman_r, 3), cor_df$sig_label)

p_cor <- ggplot(cor_df, aes(x=dataset, y=variable, fill=spearman_r)) +
  geom_tile(color="white", linewidth=0.5) +
  geom_text(aes(label=r_label), size=4, fontface="bold") +
  scale_fill_gradient2(low="#2166AC", mid="white", high="#D6604D",
                       midpoint=0, limits=c(-0.3,0.3),
                       name="Spearman r") +
  labs(title="T-Cell Exhaustion Score: Spearman Correlations with Clinical Variables",
       subtitle="* padj<=0.05 (Bonferroni); dagger p<=0.05 (uncorrected)",
       x="Dataset", y="Clinical Variable",
       caption="Sources: GSE63060, GSE63061 (AddNeuroMed cohort)") +
  theme_classic(base_size=11) +
  theme(plot.title=element_text(size=11, face="bold"),
        axis.text=element_text(size=10))

ggsave('/mnt/results/exhaustion_clinical_correlation.png', p_cor, width=7, height=4, dpi=150)
cat("\nSaved: exhaustion_clinical_correlation.png\n")

# ── Summary of Step 3 ─────────────────────────────────────────────────────
cat("\n=== STEP 3 SUMMARY ===\n")
cat("Blood transcriptomics: GSE63060 (145 AD, 104 CTL) + GSE63061 (139 AD, 134 CTL)\n")
cat("Platform: Illumina HumanHT-12 (already log2-transformed)\n")
cat("Covariates: condition + age\n\n")
cat("Key findings:\n")
cat("1. T-cell exhaustion score: NO significant difference AD vs CTL in either dataset\n")
cat("   (Wilcoxon p=0.88 [GSE63060], p=0.34 [GSE63061]; Cohen's d near zero)\n")
cat("2. Near-significant individual genes:\n")
cat("   - HAVCR2 (TIM-3): log2FC=+0.078, p=0.006, padj=0.059 (GSE63061) — trending UP in AD\n")
cat("   - CX3CR1: log2FC=+0.119, p=0.012, padj=0.092 (GSE63061) — trending UP in AD\n")
cat("   - CTLA4: log2FC=-0.034/-0.049, p~0.03 in both datasets — trending DOWN in AD\n")
cat("   - KLRG1: log2FC=-0.158, p=0.042, padj=0.196 (GSE63060) — trending DOWN in AD\n")
cat("3. No curated DDR genes significantly dysregulated in AD blood at padj<=0.05\n")
cat("4. Inhibitory receptor sub-score shows strongest (non-significant) trend: p=0.077, d=0.246\n")
cat("\nInterpretation: Bulk blood transcriptomics lacks resolution to detect T-cell-specific\n")
cat("exhaustion signals (T cells are ~5-15% of PBMCs). Single-cell data (Step 4) needed.\n")


Spearman correlations (exhaustion score vs clinical variables):
         dataset              variable   spearman_r    pvalue padj   n
rho...1 GSE63060 Disease Status (AD=1) -0.009289834 0.8840346    1 249
rho...2 GSE63060                   Age -0.024043417 0.7057692    1 249
rho...3 GSE63061 Disease Status (AD=1)  0.057366223 0.3450295    1 273
rho...4 GSE63061                   Age  0.059593076 0.3265960    1 273

Saved: exhaustion_clinical_correlation.png

=== STEP 3 SUMMARY ===
Blood transcriptomics: GSE63060 (145 AD, 104 CTL) + GSE63061 (139 AD, 134 CTL)
Platform: Illumina HumanHT-12 (already log2-transformed)
Covariates: condition + age

Key findings:
1. T-cell exhaustion score: NO significant difference AD vs CTL in either dataset
   (Wilcoxon p=0.88 [GSE63060], p=0.34 [GSE63061]; Cohen's d near zero)
2. Near-significant individual genes:
   - HAVCR2 (TIM-3): log2FC=+0.078, p=0.006, padj=0.059 (GSE63061) <e2><80><94> trending UP in AD
   - CX3CR1: log2FC=+0.119, p=0.012, padj=0.

Fix clipped title on correlation heatmap - wider figure and wrapped title

In [95]:

library(dplyr); library(ggplot2)
set.seed(42)

scores <- read.csv('/mnt/results/tcell_exhaustion_score_ad.csv')
meta60 <- read.csv('/mnt/results/tmp/gse63060_meta.csv')
meta61 <- read.csv('/mnt/results/tmp/gse63061_meta.csv')

meta_all <- bind_rows(
  meta60 %>% mutate(dataset="GSE63060"),
  meta61 %>% mutate(dataset="GSE63061")
) %>% select(sample_id, status, age, sex, dataset)

scores_full <- scores %>%
  select(sample_id, exhaustion_score, dataset) %>%
  left_join(meta_all, by=c("sample_id","dataset")) %>%
  filter(status %in% c("AD","CTL")) %>%
  mutate(status_num = ifelse(status=="AD", 1, 0),
         age_num    = as.numeric(age))

cor_results <- list()
for (ds in c("GSE63060","GSE63061")) {
  sub <- scores_full %>% filter(dataset==ds)
  for (vname in c("Disease Status (AD=1)","Age")) {
    v <- if (vname=="Disease Status (AD=1)") sub$status_num else sub$age_num
    valid <- !is.na(v) & !is.na(sub$exhaustion_score)
    if (sum(valid) < 10) next
    ct <- cor.test(sub$exhaustion_score[valid], v[valid], method="spearman")
    cor_results[[paste(ds,vname)]] <- data.frame(
      dataset=ds, variable=vname,
      spearman_r=ct$estimate, pvalue=ct$p.value, n=sum(valid))
  }
}
cor_df <- bind_rows(cor_results)
cor_df$padj <- p.adjust(cor_df$pvalue, method="bonferroni")
cor_df$sig_label <- ifelse(cor_df$padj<=0.05,"*", ifelse(cor_df$pvalue<=0.05,"†",""))
cor_df$r_label <- paste0(round(cor_df$spearman_r,3), cor_df$sig_label)

p_cor <- ggplot(cor_df, aes(x=dataset, y=variable, fill=spearman_r)) +
  geom_tile(color="white", linewidth=0.5) +
  geom_text(aes(label=r_label), size=4.5, fontface="bold") +
  scale_fill_gradient2(low="#2166AC", mid="white", high="#D6604D",
                       midpoint=0, limits=c(-0.3,0.3), name="Spearman r") +
  labs(
    title="T-Cell Exhaustion Score:\nSpearman Correlations with Clinical Variables",
    subtitle="* padj<=0.05 (Bonferroni); dagger p<=0.05 (uncorrected)",
    x="Dataset", y="Clinical Variable",
    caption="Sources: GSE63060, GSE63061 (AddNeuroMed cohort)"
  ) +
  theme_classic(base_size=11) +
  theme(plot.title=element_text(size=11, face="bold"),
        axis.text=element_text(size=10))

ggsave('/mnt/results/exhaustion_clinical_correlation.png', p_cor,
       width=8, height=4.5, dpi=150)
cat("Resaved: exhaustion_clinical_correlation.png\n")


Resaved: exhaustion_clinical_correlation.pngWarning: Cannot compute exact p-value with ties

Step 4a: Query CellxGene Census for AD PBMC/blood scRNA-seq datasets with T cell annotations

In [97]:

import cellxgene_census
import pandas as pd
import numpy as np

np.random.seed(42)

# Source: https://cellxgene.cziscience.com/
print("Opening CellxGene Census...")
census = cellxgene_census.open_soma()

# Query cell metadata for: Homo sapiens, blood/PBMC tissue, Alzheimer's disease
# Focus on peripheral blood where T cells are abundant
print("\nQuerying cell metadata for AD blood/PBMC samples...")

obs = census["census_data"]["homo_sapiens"].obs

# First check what disease and tissue values are available for AD
value_filter = (
    "disease == 'Alzheimer disease' and "
    "tissue_general in ['blood', 'brain']"
)

cell_meta = obs.read(
    value_filter=value_filter,
    column_names=["soma_joinid", "cell_type", "tissue", "tissue_general",
                  "disease", "sex", "development_stage", "dataset_id",
                  "assay", "suspension_type", "donor_id"]
).concat().to_pandas()

print(f"Total cells matching AD + blood/brain: {len(cell_meta)}")
print(f"\nTissue breakdown:")
print(cell_meta['tissue_general'].value_counts())
print(f"\nCell type breakdown (top 20):")
print(cell_meta['cell_type'].value_counts().head(20))
print(f"\nDataset IDs:")
print(cell_meta['dataset_id'].value_counts())


Opening CellxGene Census...

Querying cell metadata for AD blood/PBMC samples...
Total cells matching AD + blood/brain: 952022

Tissue breakdown:
tissue_general
brain              952022
abdomen                 0
adrenal gland           0
axilla                  0
bladder organ           0
                    ...  
ureter                  0
urinary bladder         0
uterus                  0
vasculature             0
yolk sac                0
Name: count, Length: 70, dtype: int64

Cell type breakdown (top 20):
cell_type
glutamatergic neuron                 491665
GABAergic neuron                     128997
oligodendrocyte                      126719
neuron                                50059
astrocyte                             33857
microglial cell                       29063
inhibitory interneuron                22475
mature astrocyte                      20106
oligodendrocyte precursor cell        17479
mature microglial cell                16829
fibroblast                        

Step 4b: Extract AD and control brain T cells + microglia from CellxGene Census

In [99]:

import cellxgene_census
import pandas as pd
import numpy as np
import scanpy as sc

np.random.seed(42)

# Source: https://cellxgene.cziscience.com/ (census version 2025-11-08)
census = cellxgene_census.open_soma(census_version="2025-11-08")
obs = census["census_data"]["homo_sapiens"].obs

# ── Query AD brain T cells ─────────────────────────────────────────────────
print("Querying AD brain T cells...")
ad_tcell_meta = obs.read(
    value_filter=(
        "disease == 'Alzheimer disease' and "
        "tissue_general == 'brain' and "
        "cell_type in ['T cell', 'CD4-positive, alpha-beta T cell', "
        "'CD8-positive, alpha-beta T cell', 'regulatory T cell', "
        "'effector memory CD8-positive, alpha-beta T cell', "
        "'naive thymus-derived CD4-positive, alpha-beta T cell']"
    ),
    column_names=["soma_joinid","cell_type","tissue","disease","sex",
                  "development_stage","dataset_id","donor_id","assay"]
).concat().to_pandas()
print(f"AD brain T cells: {len(ad_tcell_meta)}")
print(ad_tcell_meta['cell_type'].value_counts())

# ── Query normal brain T cells (control) ──────────────────────────────────
print("\nQuerying normal brain T cells...")
ctl_tcell_meta = obs.read(
    value_filter=(
        "disease == 'normal' and "
        "tissue_general == 'brain' and "
        "cell_type in ['T cell', 'CD4-positive, alpha-beta T cell', "
        "'CD8-positive, alpha-beta T cell', 'regulatory T cell', "
        "'effector memory CD8-positive, alpha-beta T cell', "
        "'naive thymus-derived CD4-positive, alpha-beta T cell']"
    ),
    column_names=["soma_joinid","cell_type","tissue","disease","sex",
                  "development_stage","dataset_id","donor_id","assay"]
).concat().to_pandas()
print(f"Normal brain T cells: {len(ctl_tcell_meta)}")
print(ctl_tcell_meta['cell_type'].value_counts())

# ── Also query microglia for crosstalk analysis ────────────────────────────
print("\nQuerying AD brain microglia...")
ad_micro_meta = obs.read(
    value_filter=(
        "disease == 'Alzheimer disease' and "
        "tissue_general == 'brain' and "
        "cell_type in ['microglial cell', 'mature microglial cell', "
        "'disease associated microglia']"
    ),
    column_names=["soma_joinid","cell_type","tissue","disease","sex",
                  "development_stage","dataset_id","donor_id","assay"]
).concat().to_pandas()
print(f"AD brain microglia: {len(ad_micro_meta)}")
print(ad_micro_meta['cell_type'].value_counts())


Querying AD brain T cells...
AD brain T cells: 429
cell_type
T cell                                             429
microfold cell of epithelium of small intestine      0
microglial cell                                      0
midbrain dopaminergic neuron                         0
midget ganglion cell of retina                       0
                                                  ... 
endothelial stalk cell                               0
endothelial tip cell                                 0
enteric neuron                                       0
enteric smooth muscle cell                           0
endothelial cell of coronary artery                  0
Name: count, Length: 898, dtype: int64

Querying normal brain T cells...
Normal brain T cells: 5286
cell_type
T cell                                             5286
microfold cell of epithelium of small intestine       0
microglial cell                                       0
midbrain dopaminergic neuron                          0


Step 4c: Fetch expression data for T cells (AD + control) for exhaustion/DDR genes

In [101]:

import cellxgene_census
import pandas as pd
import numpy as np
import scanpy as sc
from scipy import sparse

np.random.seed(42)

# Source: https://cellxgene.cziscience.com/ (census version 2025-11-08)
census = cellxgene_census.open_soma(census_version="2025-11-08")

# Signature genes to fetch
exhaust_genes = ["PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244",
                 "TOX","TOX2","NR4A1","NR4A2","EOMES","TBX21",
                 "IFNG","TNF","IL2","GZMB","KLRG1","CX3CR1",
                 "H2AX","ATM","TP53BP1","BRCA1","RAD51",
                 # Additional DDR genes from curated list
                 "CHEK1","CHEK2","PARP1","RPA1","RPA2","RPA3",
                 "XRCC6","MSH2","RAD51C","ERCC3","PRKDC"]

# T cell soma_joinids
obs = census["census_data"]["homo_sapiens"].obs

ad_tcell_ids = obs.read(
    value_filter="disease == 'Alzheimer disease' and tissue_general == 'brain' and cell_type == 'T cell'",
    column_names=["soma_joinid","cell_type","disease","donor_id","dataset_id","sex","development_stage"]
).concat().to_pandas()

ctl_tcell_ids = obs.read(
    value_filter="disease == 'normal' and tissue_general == 'brain' and cell_type == 'T cell'",
    column_names=["soma_joinid","cell_type","disease","donor_id","dataset_id","sex","development_stage"]
).concat().to_pandas()

all_tcell_ids = pd.concat([ad_tcell_ids, ctl_tcell_ids], ignore_index=True)
print(f"Total T cells: {len(all_tcell_ids)} (AD={len(ad_tcell_ids)}, CTL={len(ctl_tcell_ids)})")
print(f"AD datasets: {ad_tcell_ids['dataset_id'].unique()}")
print(f"CTL datasets (sample): {ctl_tcell_ids['dataset_id'].unique()[:5]}")

# Fetch expression for these cells + signature genes
print("\nFetching expression data...")
adata = cellxgene_census.get_anndata(
    census=census,
    organism="Homo sapiens",
    obs_value_filter=(
        "(disease == 'Alzheimer disease' or disease == 'normal') and "
        "tissue_general == 'brain' and cell_type == 'T cell'"
    ),
    var_value_filter=f"feature_name in {exhaust_genes}",
    column_names={"obs": ["soma_joinid","cell_type","disease","donor_id",
                          "dataset_id","sex","development_stage"],
                  "var": ["feature_id","feature_name"]}
)

print(f"\nAnnData: {adata.shape[0]} cells x {adata.shape[1]} genes")
print(f"Genes retrieved: {adata.var['feature_name'].tolist()}")
print(f"Disease breakdown: {adata.obs['disease'].value_counts().to_dict()}")
print(f"Donors: AD={adata.obs[adata.obs['disease']=='Alzheimer disease']['donor_id'].nunique()}, "
      f"CTL={adata.obs[adata.obs['disease']=='normal']['donor_id'].nunique()}")


Total T cells: 5715 (AD=429, CTL=5286)
AD datasets: ['a1b9c51e-a408-4f7f-bccb-abefe20ae2a5', '203025fe-fa99-4d57-81da-458ed8f0c334', 'ac0c6561-7a48-4185-af6f-af799f699172']
Categories (1086, object): ['0041b9c3-6a49-4bf7-8514-9bc7190067a7', '00476f9f-ebc1-4b72-b541-32f912ce36ea',
                            '00593d17-0693-4646-acad-89dbefba11bb', '0087cde2-967d-4f7c-8e6e-40e4c9ad1891', ...,
                            'ff45e623-7f5f-46e3-b47d-56be0341f66b', 'ff4cfa86-9c0c-4b7c-abd6-90547657d04f',
                            'ff7d15fa-f4b6-4a0e-992e-fd0c9d088ded', 'ff995299-bf56-4d11-914d-b77949f02fb1']
CTL datasets (sample): ['95d3b0fb-b8a7-4bec-a2e8-cded3222ff07', '07dd8c34-4625-480a-8ee4-273a06c3082b', 'f3c49918-4707-4d92-bb6d-2b5b4eb9d1b4', 'be884a28-0eac-4eca-9c23-1294a93cf56e', '203025fe-fa99-4d57-81da-458ed8f0c334']
Categories (1086, object): ['0041b9c3-6a49-4bf7-8514-9bc7190067a7', '00476f9f-ebc1-4b72-b541-32f912ce36ea',
                            '00593d17-0693-4646-acad-89dbe

Step 4d: Score T cells for exhaustion/DDR, pseudobulk DE, UMAP visualization

In [103]:

import cellxgene_census
import pandas as pd
import numpy as np
import scanpy as sc
from scipy import stats
from scipy.sparse import issparse
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Source: https://cellxgene.cziscience.com/ (census version 2025-11-08)
# adata already fetched above — re-fetch to ensure clean state
census = cellxgene_census.open_soma(census_version="2025-11-08")

exhaust_genes = ["PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244",
                 "TOX","TOX2","NR4A1","NR4A2","EOMES","TBX21",
                 "IFNG","TNF","IL2","GZMB","KLRG1","CX3CR1",
                 "H2AX","ATM","TP53BP1","BRCA1","RAD51",
                 "CHEK1","CHEK2","PARP1","RPA1","RPA2","RPA3",
                 "XRCC6","MSH2","RAD51C","ERCC3","PRKDC"]

adata = cellxgene_census.get_anndata(
    census=census,
    organism="Homo sapiens",
    obs_value_filter=(
        "(disease == 'Alzheimer disease' or disease == 'normal') and "
        "tissue_general == 'brain' and cell_type == 'T cell'"
    ),
    var_value_filter=f"feature_name in {exhaust_genes}",
    obs_column_names=["soma_joinid","cell_type","disease","donor_id",
                      "dataset_id","sex","development_stage"],
    var_column_names=["feature_id","feature_name"]
)

print(f"AnnData: {adata.shape}")

# ── Normalize: log1p of raw counts ────────────────────────────────────────
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Set gene names as var index
adata.var_names = adata.var['feature_name'].values
adata.obs['condition'] = adata.obs['disease'].map(
    {'Alzheimer disease': 'AD', 'normal': 'CTL'})

# ── Define sub-signatures ─────────────────────────────────────────────────
sig_inhibitory = [g for g in ["PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244"] if g in adata.var_names]
sig_ddr        = [g for g in ["H2AX","ATM","TP53BP1","BRCA1","RAD51","CHEK1","CHEK2",
                               "PARP1","RPA1","RPA2","RPA3","XRCC6","MSH2","RAD51C","ERCC3","PRKDC"] if g in adata.var_names]
sig_effector   = [g for g in ["IFNG","TNF","IL2","GZMB"] if g in adata.var_names]
sig_all        = [g for g in exhaust_genes if g in adata.var_names]

def mean_score(adata, genes):
    present = [g for g in genes if g in adata.var_names]
    if not present: return np.zeros(adata.n_obs)
    X = adata[:, present].X
    if issparse(X): X = X.toarray()
    return X.mean(axis=1)

adata.obs['score_exhaustion']  = mean_score(adata, sig_inhibitory)
adata.obs['score_ddr']         = mean_score(adata, sig_ddr)
adata.obs['score_effector']    = mean_score(adata, sig_effector)
adata.obs['score_overall']     = mean_score(adata, sig_all)

print(f"\nSignature genes present:")
print(f"  Inhibitory receptors: {sig_inhibitory}")
print(f"  DDR genes: {sig_ddr}")
print(f"  Effector: {sig_effector}")

# ── Wilcoxon tests: AD vs CTL per score ───────────────────────────────────
print("\n── Score comparisons (AD vs CTL brain T cells) ──")
results = []
for score_name in ['score_exhaustion','score_ddr','score_effector','score_overall']:
    ad_vals  = adata.obs.loc[adata.obs['condition']=='AD',  score_name].values
    ctl_vals = adata.obs.loc[adata.obs['condition']=='CTL', score_name].values
    wt = stats.mannwhitneyu(ad_vals, ctl_vals, alternative='two-sided')
    pooled_sd = np.sqrt((np.var(ad_vals) + np.var(ctl_vals)) / 2)
    d = (np.mean(ad_vals) - np.mean(ctl_vals)) / pooled_sd if pooled_sd > 0 else 0
    results.append({'score': score_name, 'mean_AD': np.mean(ad_vals),
                    'mean_CTL': np.mean(ctl_vals), 'p_value': wt.pvalue, 'cohens_d': d,
                    'n_AD': len(ad_vals), 'n_CTL': len(ctl_vals)})
    print(f"  {score_name}: AD={np.mean(ad_vals):.4f}, CTL={np.mean(ctl_vals):.4f}, "
          f"p={wt.pvalue:.4f}, d={d:.3f}")

score_df = pd.DataFrame(results)
score_df['padj'] = score_df['p_value'] * len(score_df)  # Bonferroni
score_df['padj'] = score_df['padj'].clip(upper=1.0)

# ── Pseudobulk DE: aggregate per donor ────────────────────────────────────
print("\n── Pseudobulk DE (donor-level aggregation) ──")
pb_rows = []
for donor in adata.obs['donor_id'].unique():
    sub = adata[adata.obs['donor_id'] == donor]
    cond = sub.obs['condition'].iloc[0]
    X = sub.X
    if issparse(X): X = X.toarray()
    mean_expr = X.mean(axis=0)
    row = {'donor_id': donor, 'condition': cond, 'n_cells': sub.n_obs}
    for i, g in enumerate(adata.var_names):
        row[g] = mean_expr[i]
    pb_rows.append(row)

pb_df = pd.DataFrame(pb_rows)
print(f"Pseudobulk donors: AD={sum(pb_df['condition']=='AD')}, CTL={sum(pb_df['condition']=='CTL')}")

# Wilcoxon on pseudobulk per gene
pb_results = []
for gene in adata.var_names:
    ad_vals  = pb_df.loc[pb_df['condition']=='AD',  gene].values
    ctl_vals = pb_df.loc[pb_df['condition']=='CTL', gene].values
    if len(ad_vals) < 3 or len(ctl_vals) < 3: continue
    wt = stats.mannwhitneyu(ad_vals, ctl_vals, alternative='two-sided')
    log2fc = np.log2(np.mean(ad_vals)+0.01) - np.log2(np.mean(ctl_vals)+0.01)
    pb_results.append({'gene': gene, 'log2FC': log2fc, 'pvalue': wt.pvalue,
                       'mean_AD': np.mean(ad_vals), 'mean_CTL': np.mean(ctl_vals),
                       'n_AD_donors': len(ad_vals), 'n_CTL_donors': len(ctl_vals)})

pb_res_df = pd.DataFrame(pb_results).sort_values('pvalue')
pb_res_df['padj'] = (pb_res_df['pvalue'] * len(pb_res_df)).clip(upper=1.0)
print("\nTop pseudobulk DE genes (AD vs CTL brain T cells):")
print(pb_res_df.head(15).to_string(index=False))

# Save outputs
pb_res_df.to_csv('/mnt/results/pseudobulk_de_tcells.csv', index=False)
score_df.to_csv('/mnt/results/scrnaseq_tcell_ddr_scores.csv', index=False)
adata.obs.to_csv('/mnt/results/tmp/tcell_obs_scores.csv')
print("\nSaved: pseudobulk_de_tcells.csv, scrnaseq_tcell_ddr_scores.csv")


AnnData: (5715, 34)

Signature genes present:
  Inhibitory receptors: ['PDCD1', 'HAVCR2', 'LAG3', 'TIGIT', 'CTLA4', 'CD244']
  DDR genes: ['H2AX', 'ATM', 'TP53BP1', 'BRCA1', 'RAD51', 'CHEK1', 'CHEK2', 'PARP1', 'RPA1', 'RPA2', 'RPA3', 'XRCC6', 'MSH2', 'RAD51C', 'ERCC3', 'PRKDC']
  Effector: ['IFNG', 'TNF', 'IL2', 'GZMB']

── Score comparisons (AD vs CTL brain T cells) ──
  score_exhaustion: AD=0.2895, CTL=0.4129, p=0.0003, d=-0.182
  score_ddr: AD=0.7004, CTL=0.9765, p=0.0000, d=-0.439
  score_effector: AD=0.1829, CTL=0.2740, p=0.0007, d=-0.129
  score_overall: AD=0.5898, CTL=0.7993, p=0.0000, d=-0.493

── Pseudobulk DE (donor-level aggregation) ──
Pseudobulk donors: AD=25, CTL=348

Top pseudobulk DE genes (AD vs CTL brain T cells):
   gene    log2FC   pvalue  mean_AD  mean_CTL  n_AD_donors  n_CTL_donors     padj
  PDCD1  3.621637 0.000002 0.221281  0.008790           25           348 0.000079
 CX3CR1  2.293632 0.000054 0.655552  0.125747           25           348 0.001847
    ATM -0.5

Step 4e: Generate UMAP of brain T cells colored by exhaustion/DDR scores and AD/CTL status

In [105]:

import cellxgene_census
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.sparse import issparse
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Source: https://cellxgene.cziscience.com/ (census version 2025-11-08)
census = cellxgene_census.open_soma(census_version="2025-11-08")

exhaust_genes = ["PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244",
                 "TOX","TOX2","NR4A1","NR4A2","EOMES","TBX21",
                 "IFNG","TNF","IL2","GZMB","KLRG1","CX3CR1",
                 "H2AX","ATM","TP53BP1","BRCA1","RAD51",
                 "CHEK1","CHEK2","PARP1","RPA1","RPA2","RPA3",
                 "XRCC6","MSH2","RAD51C","ERCC3","PRKDC"]

adata = cellxgene_census.get_anndata(
    census=census,
    organism="Homo sapiens",
    obs_value_filter=(
        "(disease == 'Alzheimer disease' or disease == 'normal') and "
        "tissue_general == 'brain' and cell_type == 'T cell'"
    ),
    var_value_filter=f"feature_name in {exhaust_genes}",
    obs_column_names=["soma_joinid","cell_type","disease","donor_id","dataset_id","sex"],
    var_column_names=["feature_id","feature_name"]
)

adata.var_names = adata.var['feature_name'].values
adata.obs['condition'] = adata.obs['disease'].map({'Alzheimer disease':'AD','normal':'CTL'})

# Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Compute scores
def mean_score(adata, genes):
    present = [g for g in genes if g in adata.var_names]
    if not present: return np.zeros(adata.n_obs)
    X = adata[:, present].X
    if issparse(X): X = X.toarray()
    return X.mean(axis=1)

sig_inhibitory = ["PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244"]
sig_ddr        = ["H2AX","ATM","TP53BP1","BRCA1","RAD51","CHEK1","CHEK2",
                  "PARP1","RPA1","RPA2","RPA3","XRCC6","MSH2","RAD51C","ERCC3","PRKDC"]

adata.obs['score_exhaustion'] = mean_score(adata, sig_inhibitory)
adata.obs['score_ddr']        = mean_score(adata, sig_ddr)

# Compute PCA + UMAP on all 34 genes
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=min(20, adata.n_vars-1), random_state=42)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=15, random_state=42)
sc.tl.umap(adata, random_state=42)

print(f"UMAP computed for {adata.n_obs} cells")

# ── Plot: 3-panel UMAP ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Brain-Infiltrating T Cells: AD vs Control (CellxGene Census 2025-11-08)",
             fontsize=13, fontweight='bold', y=1.02)

umap_coords = adata.obsm['X_umap']
condition   = adata.obs['condition'].values
exh_score   = adata.obs['score_exhaustion'].values
ddr_score   = adata.obs['score_ddr'].values

# Panel 1: AD vs CTL
colors = ['#D6604D' if c=='AD' else '#4393C3' for c in condition]
axes[0].scatter(umap_coords[:,0], umap_coords[:,1], c=colors, s=3, alpha=0.5)
from matplotlib.patches import Patch
legend_els = [Patch(facecolor='#D6604D', label=f'AD (n=429)'),
              Patch(facecolor='#4393C3', label=f'CTL (n=5,286)')]
axes[0].legend(handles=legend_els, fontsize=9, loc='upper right')
axes[0].set_title('Disease Status', fontweight='bold')
axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')
axes[0].set_aspect('equal')

# Panel 2: Exhaustion score
sc2 = axes[1].scatter(umap_coords[:,0], umap_coords[:,1], c=exh_score,
                      cmap='RdYlBu_r', s=3, alpha=0.6,
                      vmin=np.percentile(exh_score,5), vmax=np.percentile(exh_score,95))
plt.colorbar(sc2, ax=axes[1], shrink=0.8, label='Score')
axes[1].set_title('Inhibitory Receptor Score\n(PDCD1, HAVCR2, LAG3, TIGIT, CTLA4, CD244)',
                  fontweight='bold', fontsize=9)
axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')
axes[1].set_aspect('equal')

# Panel 3: DDR score
sc3 = axes[2].scatter(umap_coords[:,0], umap_coords[:,1], c=ddr_score,
                      cmap='RdYlBu', s=3, alpha=0.6,
                      vmin=np.percentile(ddr_score,5), vmax=np.percentile(ddr_score,95))
plt.colorbar(sc3, ax=axes[2], shrink=0.8, label='Score')
axes[2].set_title('DDR Gene Score\n(ATM, PRKDC, RPA1-3, BRCA1, RAD51, etc.)',
                  fontweight='bold', fontsize=9)
axes[2].set_xlabel('UMAP 1'); axes[2].set_ylabel('UMAP 2')
axes[2].set_aspect('equal')

# Caption
fig.text(0.5, -0.02,
         'Source: CellxGene Census 2025-11-08; N=5,715 brain T cells (AD=429 from 25 donors, CTL=5,286 from 348 donors)\n'
         'Pseudobulk DE: PDCD1 log2FC=+3.6 padj=8e-5; ATM log2FC=-0.55 padj=0.017; DDR score p<0.0001 Cohen\'s d=-0.44',
         ha='center', fontsize=8, style='italic')

plt.tight_layout()
plt.savefig('/mnt/results/umap_tcell_exhaustion.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: umap_tcell_exhaustion.png")


UMAP computed for 5715 cells
Saved: umap_tcell_exhaustion.png
  File "/opt/conda/lib/python3.11/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Step 5a: Query GTEx v8 eQTL API for AD GWAS SNPs in brain and blood tissues

In [107]:

import requests
import pandas as pd
import numpy as np
import time

np.random.seed(42)

# Source: https://gtexportal.org/api/v2/
# GTEx v8 REST API — query eQTLs for AD GWAS SNPs in brain and blood tissues
GTEX_API = "https://gtexportal.org/api/v2"

# Load AD GWAS loci
gwas_df = pd.read_csv('/mnt/results/ad_gwas_loci.csv')
ddr_cur = pd.read_csv('/mnt/results/ddr_gene_universe_curated.csv')
ddr_genes = set(ddr_cur['gene_symbol'].str.upper())

# T-cell exhaustion genes
tcell_genes = {"PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","CD244","TOX","TOX2",
               "NR4A1","NR4A2","EOMES","TBX21","IFNG","TNF","IL2","GZMB",
               "KLRG1","CX3CR1","H2AX","ATM","TP53BP1","BRCA1","RAD51"}

target_genes = ddr_genes | tcell_genes

# Tissues of interest
tissues = [
    "Brain_Frontal_Cortex_BA9",
    "Brain_Hippocampus",
    "Whole_Blood",
]

# Get unique rsIDs from AD GWAS
rsids = gwas_df['rsID'].dropna().unique().tolist()
print(f"AD GWAS SNPs to query: {len(rsids)}")
print(f"Target genes (DDR + T-cell): {len(target_genes)}")
print(f"Tissues: {tissues}")

# Query GTEx eQTL API per SNP
def query_gtex_eqtl(rsid, tissue):
    """Query GTEx v8 single-tissue eQTL for a given rsID and tissue."""
    url = f"{GTEX_API}/association/singleTissueEqtl"
    params = {
        "variantId": rsid,
        "tissueSiteDetailId": tissue,
        "datasetId": "gtex_v8",
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        if resp.status_code == 200:
            data = resp.json()
            return data.get("data", [])
    except Exception as e:
        pass
    return []

# Test with a known AD SNP first
print("\nTesting API with rs6733839 (BIN1 locus)...")
test = query_gtex_eqtl("rs6733839", "Brain_Frontal_Cortex_BA9")
print(f"Test result: {len(test)} eQTL associations")
if test:
    print(f"Sample: {test[0]}")


AD GWAS SNPs to query: 122
Target genes (DDR + T-cell): 123
Tissues: ['Brain_Frontal_Cortex_BA9', 'Brain_Hippocampus', 'Whole_Blood']

Testing API with rs6733839 (BIN1 locus)...
Test result: 0 eQTL associations


Check GTEx API endpoint structure and correct variant query format

In [109]:

import requests
import json

GTEX_API = "https://gtexportal.org/api/v2"

# Check available endpoints
resp = requests.get(f"{GTEX_API}/association/singleTissueEqtl", 
                    params={"gencodeId": "ENSG00000136997.14",  # MYC as test
                            "tissueSiteDetailId": "Whole_Blood",
                            "datasetId": "gtex_v8"},
                    timeout=20)
print("Gene-based query status:", resp.status_code)
if resp.status_code == 200:
    data = resp.json()
    print("Keys:", list(data.keys()))
    results = data.get("data", [])
    print(f"Results: {len(results)}")
    if results:
        print("Sample result keys:", list(results[0].keys()))
        print("Sample:", results[0])

# Try querying by gene symbol for a DDR gene
print("\n--- Querying ATM eQTLs in Whole Blood ---")
resp2 = requests.get(f"{GTEX_API}/association/singleTissueEqtl",
                     params={"geneSymbol": "ATM",
                             "tissueSiteDetailId": "Whole_Blood",
                             "datasetId": "gtex_v8"},
                     timeout=20)
print("ATM query status:", resp2.status_code)
if resp2.status_code == 200:
    data2 = resp2.json()
    results2 = data2.get("data", [])
    print(f"ATM eQTLs in Whole Blood: {len(results2)}")
    if results2:
        print("Sample:", results2[0])


Gene-based query status: 200
Keys: ['data', 'paging_info']
Results: 0

--- Querying ATM eQTLs in Whole Blood ---
ATM query status: 400


Try GTEx API with correct Ensembl gene ID format for DDR genes

In [111]:

import requests
import pandas as pd
import time

GTEX_API = "https://gtexportal.org/api/v2"

# First get Ensembl IDs for key DDR/T-cell genes via GTEx gene search
key_genes = ["ATM", "BRCA1", "BRCA2", "CHEK2", "RPA3", "DNTT", "PDCD1", 
             "HAVCR2", "LAG3", "PRKDC", "RAD51C", "MSH2", "XRCC6"]

# Look up Ensembl IDs via GTEx gene endpoint
gene_ids = {}
for gene in key_genes[:5]:  # test first
    resp = requests.get(f"{GTEX_API}/reference/gene",
                        params={"geneSymbol": gene, "gencodeVersion": "v26", 
                                "genomeBuild": "GRCh38/hg38"},
                        timeout=15)
    if resp.status_code == 200:
        data = resp.json().get("data", [])
        if data:
            gene_ids[gene] = data[0].get("gencodeId", "")
            print(f"{gene}: {gene_ids[gene]}")
        else:
            print(f"{gene}: not found")
    time.sleep(0.2)


(No output)

Explore GTEx API endpoints for eQTL data

In [113]:

import requests

# Source: https://gtexportal.org/api/v2
base_url = "https://gtexportal.org/api/v2"

# Check available endpoints
resp = requests.get(f"{base_url}/")
print(resp.status_code)
print(resp.text[:2000])


200
{"id":"org.gtexportal.rest.v2","name":"GTEx Portal V2 API","version":"2.0.0","organization":{"name":"GTEx Project","url":"https://gtexportal.org"},"description":"This service provides access to the data powering the GTEx portal.","contactUrl":"https://gtexportal.org/home/contact","documentationUrl":"https://gtexportal.org/api/v2/docs","environment":"prod"}


Check GTEx API docs and tissue names

In [115]:

import requests

base_url = "https://gtexportal.org/api/v2"

# Get tissue list
# Source: https://gtexportal.org/api/v2/dataset/tissueInfo
resp = requests.get(f"{base_url}/dataset/tissueInfo", params={"datasetId": "gtex_v8", "format": "json"})
print(resp.status_code)
data = resp.json()
# Find relevant tissues
tissues = data.get("tissueInfo", [])
for t in tissues:
    name = t.get("tissueSiteDetailId", "")
    if any(kw in name.lower() for kw in ["frontal", "hippocampus", "blood", "cortex"]):
        print(name, "|", t.get("tissueSiteDetail", ""))


404


Check GTEx API docs for correct endpoints

In [117]:

import requests

base_url = "https://gtexportal.org/api/v2"

# Source: https://gtexportal.org/api/v2/docs
resp = requests.get(f"{base_url}/docs")
print(resp.status_code)
print(resp.text[:3000])


200

    <!DOCTYPE html>
    <html>
    <head>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link type="text/css" rel="stylesheet" href="https://cdn.jsdelivr.net/npm/swagger-ui-dist@5/swagger-ui.css">
    <link rel="shortcut icon" href="https://fastapi.tiangolo.com/img/favicon.png">
    <title>GTEx Portal API - Swagger UI</title>
    </head>
    <body>
    <div id="swagger-ui">
    </div>
    <script src="https://cdn.jsdelivr.net/npm/swagger-ui-dist@5/swagger-ui-bundle.js"></script>
    <!-- `SwaggerUIBundle` is now available on the page -->
    <script>
    const ui = SwaggerUIBundle({
        url: '/api/v2/openapi.json',
    "dom_id": "#swagger-ui",
"layout": "BaseLayout",
"deepLinking": true,
"showExtensions": true,
"showCommonExtensions": true,
oauth2RedirectUrl: window.location.origin + '/docs/oauth2-redirect',
    presets: [
        SwaggerUIBundle.presets.apis,
        SwaggerUIBundle.SwaggerUIStandalonePreset
        ],
    })
    </script>
   

Get GTEx OpenAPI spec to find eQTL endpoints

In [119]:

import requests, json

base_url = "https://gtexportal.org/api/v2"

# Source: https://gtexportal.org/api/v2/openapi.json
resp = requests.get(f"{base_url}/openapi.json")
spec = resp.json()

# List all paths
for path in spec.get("paths", {}).keys():
    print(path)


/api/v2/
/api/v2/admin/maintenanceMessage
/api/v2/admin/newsItem
/api/v2/association/egene
/api/v2/association/fineMapping
/api/v2/association/independentEqtl
/api/v2/association/metasoft
/api/v2/association/sgene
/api/v2/association/singleTissueEqtl
/api/v2/association/singleTissueEqtlByLocation
/api/v2/association/singleTissueSqtl
/api/v2/association/singleTissueIEqtl
/api/v2/association/singleTissueISqtl
/api/v2/association/dyneqtl
/api/v2/association/dynieqtl
/api/v2/association/dynsqtl
/api/v2/association/dynisqtl
/api/v2/biobank/sample
/api/v2/biobank/download
/api/v2/dataset/annotation
/api/v2/dataset/collapsedGeneModelExon
/api/v2/dataset/fileList
/api/v2/dataset/openAccessFilesMetadata
/api/v2/dataset/fullCollapsedGeneModelExon
/api/v2/dataset/functionalAnnotation
/api/v2/dataset/sample
/api/v2/dataset/tissueSiteDetail
/api/v2/dataset/subject
/api/v2/dataset/variant
/api/v2/dataset/ld
/api/v2/dataset/ldByVariant
/api/v2/expression/expressionPca
/api/v2/expression/geneExpressio

Check eQTL endpoint parameters and tissue list

In [121]:

import requests, json

base_url = "https://gtexportal.org/api/v2"
spec = requests.get(f"{base_url}/openapi.json").json()

# Check singleTissueEqtl and egene endpoints
for path in ["/api/v2/association/singleTissueEqtl", "/api/v2/association/egene", "/api/v2/dataset/tissueSiteDetail"]:
    print(f"\n=== {path} ===")
    info = spec["paths"].get(path, {})
    for method, details in info.items():
        params = details.get("parameters", [])
        for p in params:
            print(f"  {p.get('name')}: {p.get('required', False)} | {p.get('schema', {})}")



=== /api/v2/association/singleTissueEqtl ===
  gencodeId: False | {'type': 'array', 'items': {'type': 'string'}, 'minItems': 1, 'maxItems': 60000, 'description': 'A list of Versioned GENCODE IDs, e.g. ENSG00000065613.9,ENSG00000203782.5', 'title': 'Gencodeid'}
  variantId: False | {'type': 'array', 'items': {'type': 'string'}, 'description': 'A list of GTEx variant IDs', 'title': 'Variantid'}
  tissueSiteDetailId: False | {'type': 'array', 'items': {'anyOf': [{'$ref': '#/components/schemas/TissueSiteDetailId'}, {'$ref': '#/components/schemas/TissueSiteOntologyId'}]}, 'description': 'A list of Tissue IDs of the tissue(s) of interest. Can be a GTEx specific ID or an Ontology ID', 'title': 'Tissuesitedetailid'}
  datasetId: False | {'$ref': '#/components/schemas/DatasetId', 'description': 'Unique identifier of a dataset. Usually includes a data source and data release.', 'default': 'gtex_v10'}
  page: False | {'type': 'integer', 'maximum': 1000000.0, 'minimum': 0, 'description': 'The 0-b

Get tissue IDs and gene GENCODE IDs for GTEx v8

In [123]:

import requests, json

base_url = "https://gtexportal.org/api/v2"

# Source: https://gtexportal.org/api/v2/dataset/tissueSiteDetail
resp = requests.get(f"{base_url}/dataset/tissueSiteDetail", params={"datasetId": "gtex_v8", "itemsPerPage": 500})
data = resp.json()
tissues = data.get("data", [])
print(f"Total tissues: {len(tissues)}")
for t in tissues:
    name = t.get("tissueSiteDetailId", "")
    if any(kw in name.lower() for kw in ["frontal", "hippocampus", "blood", "cortex"]):
        print(name, "|", t.get("tissueSiteDetail", ""))


Total tissues: 54
Brain_Anterior_cingulate_cortex_BA24 | Brain - Anterior cingulate cortex (BA24)
Brain_Cortex | Brain - Cortex
Brain_Frontal_Cortex_BA9 | Brain - Frontal Cortex (BA9)
Brain_Hippocampus | Brain - Hippocampus
Kidney_Cortex | Kidney - Cortex
Whole_Blood | Whole Blood


Get GENCODE IDs for all target genes

In [125]:

import requests

base_url = "https://gtexportal.org/api/v2"

genes = ["ATM", "BRCA1", "BRCA2", "CHEK2", "RPA3", "DNTT", "PDCD1", "HAVCR2",
         "LAG3", "PRKDC", "RAD51C", "MSH2", "XRCC6", "CHEK1", "TP53BP1"]

gene_ids = {}
for gene in genes:
    # Source: https://gtexportal.org/api/v2/reference/gene
    resp = requests.get(f"{base_url}/reference/gene", params={"geneSymbol": gene, "datasetId": "gtex_v8"})
    data = resp.json()
    gene_data = data.get("data", [])
    if gene_data:
        gid = gene_data[0].get("gencodeId", "")
        gene_ids[gene] = gid
        print(f"{gene}: {gid}")
    else:
        print(f"{gene}: NOT FOUND")
        gene_ids[gene] = None


ATM: NOT FOUND
BRCA1: NOT FOUND
BRCA2: NOT FOUND
CHEK2: NOT FOUND
RPA3: NOT FOUND
DNTT: NOT FOUND
PDCD1: NOT FOUND
HAVCR2: NOT FOUND
LAG3: NOT FOUND
PRKDC: NOT FOUND
RAD51C: NOT FOUND
MSH2: NOT FOUND
XRCC6: NOT FOUND
CHEK1: NOT FOUND
TP53BP1: NOT FOUND


Try geneSearch endpoint for gene lookup

In [127]:

import requests

base_url = "https://gtexportal.org/api/v2"

# Source: https://gtexportal.org/api/v2/reference/geneSearch
resp = requests.get(f"{base_url}/reference/geneSearch", params={"geneSymbol": "ATM", "datasetId": "gtex_v8"})
print(resp.status_code)
print(resp.text[:2000])


422
{"detail":[{"type":"missing","loc":["query","geneId"],"msg":"Field required","input":null}]}


Try gene endpoint with different parameters

In [129]:

import requests

base_url = "https://gtexportal.org/api/v2"

# Check the spec for gene endpoint params
spec = requests.get(f"{base_url}/openapi.json").json()
gene_path = spec["paths"].get("/api/v2/reference/gene", {})
for method, details in gene_path.items():
    params = details.get("parameters", [])
    for p in params:
        print(f"  {p.get('name')}: required={p.get('required', False)} | {p.get('schema', {})}")


  geneId: required=True | {'type': 'array', 'items': {'type': 'string', 'minLength': 1}, 'minItems': 1, 'description': 'A gene symbol, versioned gencodeId, or unversioned gencodeId.', 'title': 'Geneid'}
  gencodeVersion: required=False | {'$ref': '#/components/schemas/GencodeVersion', 'description': 'GENCODE annotation release.', 'default': 'v26'}
  genomeBuild: required=False | {'$ref': '#/components/schemas/GenomeBuild', 'default': 'GRCh38/hg38'}
  page: required=False | {'type': 'integer', 'maximum': 1000000.0, 'minimum': 0, 'description': 'The 0-based numeric ID of the page to retrieve', 'default': 0, 'title': 'Page'}
  itemsPerPage: required=False | {'type': 'integer', 'maximum': 100000, 'minimum': 1, 'default': 250, 'title': 'Itemsperpage'}


Get GENCODE IDs using geneId parameter

In [131]:

import requests

base_url = "https://gtexportal.org/api/v2"

genes = ["ATM", "BRCA1", "BRCA2", "CHEK2", "RPA3", "DNTT", "PDCD1", "HAVCR2",
         "LAG3", "PRKDC", "RAD51C", "MSH2", "XRCC6", "CHEK1", "TP53BP1"]

gene_ids = {}
for gene in genes:
    # Source: https://gtexportal.org/api/v2/reference/gene
    resp = requests.get(f"{base_url}/reference/gene", params={"geneId": gene, "gencodeVersion": "v26", "genomeBuild": "GRCh38/hg38"})
    data = resp.json()
    gene_data = data.get("data", [])
    if gene_data:
        gid = gene_data[0].get("gencodeId", "")
        gene_ids[gene] = gid
        print(f"{gene}: {gid}")
    else:
        print(f"{gene}: NOT FOUND | {resp.text[:200]}")
        gene_ids[gene] = None


ATM: ENSG00000149311.18
BRCA1: ENSG00000012048.20
BRCA2: ENSG00000139618.14
CHEK2: ENSG00000183765.20
RPA3: ENSG00000106399.11
DNTT: ENSG00000107447.7
PDCD1: ENSG00000188389.10
HAVCR2: ENSG00000135077.8
LAG3: ENSG00000089692.8
PRKDC: ENSG00000253729.7
RAD51C: ENSG00000108384.14
MSH2: ENSG00000095002.13
XRCC6: ENSG00000196419.12
CHEK1: ENSG00000149554.12
TP53BP1: ENSG00000067369.13


Query singleTissueEqtl for all gene-tissue combinations in GTEx v8

In [133]:

import requests
import pandas as pd
import time

base_url = "https://gtexportal.org/api/v2"

gene_ids = {
    "ATM": "ENSG00000149311.18",
    "BRCA1": "ENSG00000012048.20",
    "BRCA2": "ENSG00000139618.14",
    "CHEK2": "ENSG00000183765.20",
    "RPA3": "ENSG00000106399.11",
    "DNTT": "ENSG00000107447.7",
    "PDCD1": "ENSG00000188389.10",
    "HAVCR2": "ENSG00000135077.8",
    "LAG3": "ENSG00000089692.8",
    "PRKDC": "ENSG00000253729.7",
    "RAD51C": "ENSG00000108384.14",
    "MSH2": "ENSG00000095002.13",
    "XRCC6": "ENSG00000196419.12",
    "CHEK1": "ENSG00000149554.12",
    "TP53BP1": "ENSG00000067369.13"
}

tissues = {
    "Brain_Frontal_Cortex_BA9": "Brain - Frontal Cortex (BA9)",
    "Brain_Hippocampus": "Brain - Hippocampus",
    "Whole_Blood": "Whole Blood"
}

all_results = []

for gene_symbol, gencode_id in gene_ids.items():
    for tissue_id, tissue_name in tissues.items():
        # Source: https://gtexportal.org/api/v2/association/singleTissueEqtl
        params = {
            "gencodeId": gencode_id,
            "tissueSiteDetailId": tissue_id,
            "datasetId": "gtex_v8",
            "itemsPerPage": 10000,
            "page": 0
        }
        resp = requests.get(f"{base_url}/association/singleTissueEqtl", params=params)
        if resp.status_code == 200:
            data = resp.json()
            records = data.get("data", [])
            print(f"{gene_symbol} | {tissue_id}: {len(records)} eQTLs")
            for r in records:
                r["gene_symbol"] = gene_symbol
                r["tissue_label"] = tissue_name
                all_results.append(r)
        else:
            print(f"{gene_symbol} | {tissue_id}: ERROR {resp.status_code} - {resp.text[:100]}")
        time.sleep(0.1)

print(f"\nTotal records: {len(all_results)}")


ATM | Brain_Frontal_Cortex_BA9: 0 eQTLs
ATM | Brain_Hippocampus: 0 eQTLs
ATM | Whole_Blood: 481 eQTLs
BRCA1 | Brain_Frontal_Cortex_BA9: 0 eQTLs
BRCA1 | Brain_Hippocampus: 0 eQTLs
BRCA1 | Whole_Blood: 0 eQTLs
BRCA2 | Brain_Frontal_Cortex_BA9: 0 eQTLs
BRCA2 | Brain_Hippocampus: 0 eQTLs
BRCA2 | Whole_Blood: 0 eQTLs
CHEK2 | Brain_Frontal_Cortex_BA9: 0 eQTLs
CHEK2 | Brain_Hippocampus: 0 eQTLs
CHEK2 | Whole_Blood: 0 eQTLs
RPA3 | Brain_Frontal_Cortex_BA9: 42 eQTLs
RPA3 | Brain_Hippocampus: 81 eQTLs
RPA3 | Whole_Blood: 2 eQTLs
DNTT | Brain_Frontal_Cortex_BA9: 0 eQTLs
DNTT | Brain_Hippocampus: 0 eQTLs
DNTT | Whole_Blood: 0 eQTLs
PDCD1 | Brain_Frontal_Cortex_BA9: 0 eQTLs
PDCD1 | Brain_Hippocampus: 0 eQTLs
PDCD1 | Whole_Blood: 265 eQTLs
HAVCR2 | Brain_Frontal_Cortex_BA9: 0 eQTLs
HAVCR2 | Brain_Hippocampus: 0 eQTLs
HAVCR2 | Whole_Blood: 88 eQTLs
LAG3 | Brain_Frontal_Cortex_BA9: 0 eQTLs
LAG3 | Brain_Hippocampus: 0 eQTLs
LAG3 | Whole_Blood: 0 eQTLs
PRKDC | Brain_Frontal_Cortex_BA9: 0 eQTLs
PRKDC | B

Inspect data structure and check for FDR/q-value fields

In [135]:

import json

# Check structure of first record
if all_results:
    print(json.dumps(all_results[0], indent=2))


{
  "snpId": "rs148680806",
  "pos": 107798154,
  "snpIdUpper": "RS148680806",
  "variantId": "chr11_107798154_C_G_b38",
  "geneSymbol": "ATM",
  "pValue": 7.21087e-05,
  "geneSymbolUpper": "ATM",
  "datasetId": "gtex_v8",
  "tissueSiteDetailId": "Whole_Blood",
  "ontologyId": "UBERON:0013756",
  "chromosome": "chr11",
  "gencodeId": "ENSG00000149311.18",
  "nes": 0.17194,
  "gene_symbol": "ATM",
  "tissue_label": "Whole Blood"
}


Check egene endpoint for FDR/q-value data for our genes

In [137]:

import requests
import time

base_url = "https://gtexportal.org/api/v2"

# Check egene endpoint spec
spec = requests.get(f"{base_url}/openapi.json").json()
egene_path = spec["paths"].get("/api/v2/association/egene", {})
for method, details in egene_path.items():
    params = details.get("parameters", [])
    for p in params:
        print(f"  {p.get('name')}: required={p.get('required', False)} | {p.get('schema', {})}")

# Also check response schema
print("\nResponse schema:")
resp_schema = egene_path.get("get", {}).get("responses", {})
print(json.dumps(resp_schema, indent=2)[:2000])


  tissueSiteDetailId: required=False | {'type': 'array', 'items': {'anyOf': [{'$ref': '#/components/schemas/TissueSiteDetailId'}, {'$ref': '#/components/schemas/TissueSiteOntologyId'}]}, 'description': 'A list of Tissue IDs of the tissue(s) of interest. Can be a GTEx specific ID or an Ontology ID', 'title': 'Tissuesitedetailid'}
  datasetId: required=False | {'$ref': '#/components/schemas/DatasetId', 'description': 'Unique identifier of a dataset. Usually includes a data source and data release.', 'default': 'gtex_v10'}
  page: required=False | {'type': 'integer', 'maximum': 1000000.0, 'minimum': 0, 'description': 'The 0-based numeric ID of the page to retrieve', 'default': 0, 'title': 'Page'}
  itemsPerPage: required=False | {'type': 'integer', 'maximum': 100000, 'minimum': 1, 'default': 250, 'title': 'Itemsperpage'}

Response schema:
{
  "200": {
    "description": "Successful Response",
    "content": {
      "application/json": {
        "schema": {
          "$ref": "#/components/

Query egene endpoint for FDR/q-value data per tissue

In [139]:

import requests
import json
import time

base_url = "https://gtexportal.org/api/v2"

tissues = ["Brain_Frontal_Cortex_BA9", "Brain_Hippocampus", "Whole_Blood"]

# Get egene data for our tissues - this has q-values
egene_data = {}
for tissue in tissues:
    # Source: https://gtexportal.org/api/v2/association/egene
    resp = requests.get(f"{base_url}/association/egene", params={
        "tissueSiteDetailId": tissue,
        "datasetId": "gtex_v8",
        "itemsPerPage": 100000,
        "page": 0
    })
    data = resp.json()
    records = data.get("data", [])
    print(f"{tissue}: {len(records)} egenes")
    egene_data[tissue] = {r.get("gencodeId", ""): r for r in records}
    time.sleep(0.2)

# Check structure of first egene record
for tissue, d in egene_data.items():
    if d:
        first_key = list(d.keys())[0]
        print(f"\nSample egene record from {tissue}:")
        print(json.dumps(d[first_key], indent=2))
        break


Brain_Frontal_Cortex_BA9: 7335 egenes
Brain_Hippocampus: 5517 egenes
Whole_Blood: 12360 egenes

Sample egene record from Brain_Frontal_Cortex_BA9:
{
  "tissueSiteDetailId": "Brain_Frontal_Cortex_BA9",
  "ontologyId": "UBERON:0009834",
  "datasetId": "gtex_v8",
  "empiricalPValue": 0.00162504,
  "gencodeId": "ENSG00000241860.6",
  "geneSymbol": "RP11-34P13.13",
  "log2AllelicFoldChange": 1.885102,
  "pValue": 1.93296e-06,
  "pValueThreshold": 9.67761e-05,
  "qValue": 0.00394038
}


Build comprehensive eQTL dataset with FDR from egene and top SNPs from singleTissueEqtl

In [141]:

import requests
import pandas as pd
import json
import time

base_url = "https://gtexportal.org/api/v2"

gene_ids = {
    "ATM": "ENSG00000149311.18",
    "BRCA1": "ENSG00000012048.20",
    "BRCA2": "ENSG00000139618.14",
    "CHEK2": "ENSG00000183765.20",
    "RPA3": "ENSG00000106399.11",
    "DNTT": "ENSG00000107447.7",
    "PDCD1": "ENSG00000188389.10",
    "HAVCR2": "ENSG00000135077.8",
    "LAG3": "ENSG00000089692.8",
    "PRKDC": "ENSG00000253729.7",
    "RAD51C": "ENSG00000108384.14",
    "MSH2": "ENSG00000095002.13",
    "XRCC6": "ENSG00000196419.12",
    "CHEK1": "ENSG00000149554.12",
    "TP53BP1": "ENSG00000067369.13"
}

tissues = {
    "Brain_Frontal_Cortex_BA9": "Brain - Frontal Cortex (BA9)",
    "Brain_Hippocampus": "Brain - Hippocampus",
    "Whole_Blood": "Whole Blood"
}

# egene_data already loaded above - has q-values
# Now build the top eQTL table

rows = []

for gene_symbol, gencode_id in gene_ids.items():
    for tissue_id, tissue_name in tissues.items():
        # Get egene info (q-value, p-value threshold)
        egene_info = egene_data.get(tissue_id, {}).get(gencode_id, None)
        
        # Get all singleTissueEqtl records
        # Source: https://gtexportal.org/api/v2/association/singleTissueEqtl
        params = {
            "gencodeId": gencode_id,
            "tissueSiteDetailId": tissue_id,
            "datasetId": "gtex_v8",
            "itemsPerPage": 10000,
            "page": 0
        }
        resp = requests.get(f"{base_url}/association/singleTissueEqtl", params=params)
        eqtl_records = []
        if resp.status_code == 200:
            eqtl_records = resp.json().get("data", [])
        
        if not eqtl_records and egene_info is None:
            # No eQTL data at all
            rows.append({
                "gene_symbol": gene_symbol,
                "gencode_id": gencode_id,
                "tissue": tissue_name,
                "tissue_id": tissue_id,
                "variant_id": None,
                "rsID": None,
                "chromosome": None,
                "position": None,
                "effect_size_NES": None,
                "pvalue": None,
                "qvalue": None,
                "pvalue_threshold": None,
                "is_egene": False,
                "is_significant_fdr05": False,
                "note": "No eQTL data"
            })
            continue
        
        # Sort by p-value ascending to get top SNP
        if eqtl_records:
            eqtl_records_sorted = sorted(eqtl_records, key=lambda x: x.get("pValue", 1))
            top_snp = eqtl_records_sorted[0]
        else:
            top_snp = None
        
        # Determine significance
        q_val = egene_info.get("qValue") if egene_info else None
        p_thresh = egene_info.get("pValueThreshold") if egene_info else None
        is_egene = egene_info is not None
        is_sig = (q_val is not None and q_val < 0.05) if q_val is not None else False
        
        if top_snp:
            rows.append({
                "gene_symbol": gene_symbol,
                "gencode_id": gencode_id,
                "tissue": tissue_name,
                "tissue_id": tissue_id,
                "variant_id": top_snp.get("variantId"),
                "rsID": top_snp.get("snpId"),
                "chromosome": top_snp.get("chromosome"),
                "position": top_snp.get("pos"),
                "effect_size_NES": top_snp.get("nes"),
                "pvalue": top_snp.get("pValue"),
                "qvalue": q_val,
                "pvalue_threshold": p_thresh,
                "is_egene": is_egene,
                "is_significant_fdr05": is_sig,
                "note": f"{len(eqtl_records)} total eQTLs tested"
            })
        elif egene_info:
            # egene exists but no singleTissueEqtl returned (shouldn't happen often)
            rows.append({
                "gene_symbol": gene_symbol,
                "gencode_id": gencode_id,
                "tissue": tissue_name,
                "tissue_id": tissue_id,
                "variant_id": None,
                "rsID": None,
                "chromosome": None,
                "position": None,
                "effect_size_NES": None,
                "pvalue": egene_info.get("pValue"),
                "qvalue": q_val,
                "pvalue_threshold": p_thresh,
                "is_egene": True,
                "is_significant_fdr05": is_sig,
                "note": "egene only, no singleTissueEqtl returned"
            })
        
        time.sleep(0.05)

df = pd.DataFrame(rows)
print(df.to_string())
print(f"\nTotal rows: {len(df)}")
print(f"Significant (FDR<0.05): {df['is_significant_fdr05'].sum()}")


   gene_symbol          gencode_id                        tissue                 tissue_id                      variant_id         rsID chromosome     position  effect_size_NES        pvalue        qvalue  pvalue_threshold  is_egene  is_significant_fdr05                     note
0          ATM  ENSG00000149311.18  Brain - Frontal Cortex (BA9)  Brain_Frontal_Cortex_BA9                            None         None       None          NaN              NaN           NaN           NaN               NaN     False                 False             No eQTL data
1          ATM  ENSG00000149311.18           Brain - Hippocampus         Brain_Hippocampus                            None         None       None          NaN              NaN           NaN           NaN               NaN     False                 False             No eQTL data
2          ATM  ENSG00000149311.18                   Whole Blood               Whole_Blood  chr11_108374194_TCCCTGAC_T_b38  rs138029336      chr11  108374194.0 

Also fetch top 5 eQTLs per significant gene-tissue combination and save all results

In [143]:

import requests
import pandas as pd
import json
import time

base_url = "https://gtexportal.org/api/v2"

gene_ids = {
    "ATM": "ENSG00000149311.18",
    "BRCA1": "ENSG00000012048.20",
    "BRCA2": "ENSG00000139618.14",
    "CHEK2": "ENSG00000183765.20",
    "RPA3": "ENSG00000106399.11",
    "DNTT": "ENSG00000107447.7",
    "PDCD1": "ENSG00000188389.10",
    "HAVCR2": "ENSG00000135077.8",
    "LAG3": "ENSG00000089692.8",
    "PRKDC": "ENSG00000253729.7",
    "RAD51C": "ENSG00000108384.14",
    "MSH2": "ENSG00000095002.13",
    "XRCC6": "ENSG00000196419.12",
    "CHEK1": "ENSG00000149554.12",
    "TP53BP1": "ENSG00000067369.13"
}

tissues = {
    "Brain_Frontal_Cortex_BA9": "Brain - Frontal Cortex (BA9)",
    "Brain_Hippocampus": "Brain - Hippocampus",
    "Whole_Blood": "Whole Blood"
}

# Build top-5 eQTL table for significant gene-tissue pairs
top5_rows = []

for gene_symbol, gencode_id in gene_ids.items():
    for tissue_id, tissue_name in tissues.items():
        egene_info = egene_data.get(tissue_id, {}).get(gencode_id, None)
        q_val = egene_info.get("qValue") if egene_info else None
        p_thresh = egene_info.get("pValueThreshold") if egene_info else None
        is_sig = (q_val is not None and q_val < 0.05)
        
        if not is_sig:
            continue
        
        # Source: https://gtexportal.org/api/v2/association/singleTissueEqtl
        params = {
            "gencodeId": gencode_id,
            "tissueSiteDetailId": tissue_id,
            "datasetId": "gtex_v8",
            "itemsPerPage": 10000,
            "page": 0
        }
        resp = requests.get(f"{base_url}/association/singleTissueEqtl", params=params)
        eqtl_records = []
        if resp.status_code == 200:
            eqtl_records = resp.json().get("data", [])
        
        # Sort by p-value and take top 5
        eqtl_sorted = sorted(eqtl_records, key=lambda x: x.get("pValue", 1))
        for rank, snp in enumerate(eqtl_sorted[:5], 1):
            top5_rows.append({
                "rank": rank,
                "gene_symbol": gene_symbol,
                "gencode_id": gencode_id,
                "tissue": tissue_name,
                "tissue_id": tissue_id,
                "variant_id": snp.get("variantId"),
                "rsID": snp.get("snpId"),
                "chromosome": snp.get("chromosome"),
                "position": snp.get("pos"),
                "effect_size_NES": snp.get("nes"),
                "pvalue": snp.get("pValue"),
                "qvalue_egene": q_val,
                "pvalue_threshold": p_thresh,
                "is_significant_fdr05": True
            })
        time.sleep(0.05)

df_top5 = pd.DataFrame(top5_rows)
print(f"Top-5 eQTLs for significant gene-tissue pairs: {len(df_top5)} rows")
print(df_top5.to_string())


Top-5 eQTLs for significant gene-tissue pairs: 67 rows
    rank gene_symbol          gencode_id                        tissue                 tissue_id                          variant_id         rsID chromosome   position  effect_size_NES        pvalue  qvalue_egene  pvalue_threshold  is_significant_fdr05
0      1         ATM  ENSG00000149311.18                   Whole Blood               Whole_Blood      chr11_108374194_TCCCTGAC_T_b38  rs138029336      chr11  108374194        -0.179899  6.299670e-27  5.729980e-21          0.000218                  True
1      2         ATM  ENSG00000149311.18                   Whole Blood               Whole_Blood            chr11_108158382_T_TA_b38   rs61380955      chr11  108158382        -0.172364  9.444260e-27  5.729980e-21          0.000218                  True
2      3         ATM  ENSG00000149311.18                   Whole Blood               Whole_Blood             chr11_108184195_G_A_b38   rs11212543      chr11  108184195        -0.171830  

Save both summary and top-5 tables to CSV files

In [145]:

import os
import pandas as pd

os.makedirs("/mnt/results", exist_ok=True)

# ── 1. Summary table: top eQTL per gene-tissue (all 45 combinations) ──────────
df_summary = pd.DataFrame(rows)

# Reorder / rename columns for clarity
df_summary = df_summary[[
    "gene_symbol", "gencode_id", "tissue", "tissue_id",
    "variant_id", "rsID", "chromosome", "position",
    "effect_size_NES", "pvalue", "qvalue", "pvalue_threshold",
    "is_egene", "is_significant_fdr05", "note"
]]
df_summary.columns = [
    "gene_symbol", "gencode_id", "tissue", "tissue_id",
    "variant_id", "rsID", "chromosome", "position",
    "effect_size_NES", "pvalue", "qvalue_FDR", "pvalue_threshold",
    "is_egene", "is_significant_FDR05", "note"
]

df_summary.to_csv("/mnt/results/gtex_v8_DDR_Tcell_eQTL_summary.csv", index=False)
print(f"Summary table saved: {len(df_summary)} rows")

# ── 2. Top-5 eQTLs for significant gene-tissue pairs ─────────────────────────
df_top5_out = df_top5[[
    "rank", "gene_symbol", "gencode_id", "tissue", "tissue_id",
    "variant_id", "rsID", "chromosome", "position",
    "effect_size_NES", "pvalue", "qvalue_egene", "pvalue_threshold",
    "is_significant_fdr05"
]]
df_top5_out.columns = [
    "rank", "gene_symbol", "gencode_id", "tissue", "tissue_id",
    "variant_id", "rsID", "chromosome", "position",
    "effect_size_NES", "pvalue", "qvalue_FDR", "pvalue_threshold",
    "is_significant_FDR05"
]

df_top5_out.to_csv("/mnt/results/gtex_v8_DDR_Tcell_eQTL_top5_significant.csv", index=False)
print(f"Top-5 significant table saved: {len(df_top5_out)} rows")

# ── 3. Print a clean summary ──────────────────────────────────────────────────
print("\n=== SIGNIFICANT eQTLs (FDR < 0.05) — TOP SNP PER GENE-TISSUE ===")
sig = df_summary[df_summary["is_significant_FDR05"] == True].copy()
sig_display = sig[[
    "gene_symbol", "tissue", "rsID", "variant_id",
    "effect_size_NES", "pvalue", "qvalue_FDR"
]].reset_index(drop=True)
sig_display["pvalue"] = sig_display["pvalue"].apply(lambda x: f"{x:.2e}" if pd.notna(x) else "N/A")
sig_display["qvalue_FDR"] = sig_display["qvalue_FDR"].apply(lambda x: f"{x:.2e}" if pd.notna(x) else "N/A")
sig_display["effect_size_NES"] = sig_display["effect_size_NES"].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")
print(sig_display.to_string(index=False))

print("\n=== NOT SIGNIFICANT / NO eQTL DATA ===")
not_sig = df_summary[df_summary["is_significant_FDR05"] == False][["gene_symbol","tissue","note"]].reset_index(drop=True)
print(not_sig.to_string(index=False))


Summary table saved: 45 rows
Top-5 significant table saved: 67 rows

=== SIGNIFICANT eQTLs (FDR < 0.05) — TOP SNP PER GENE-TISSUE ===
gene_symbol                       tissue        rsID                     variant_id effect_size_NES   pvalue qvalue_FDR
        ATM                  Whole Blood rs138029336 chr11_108374194_TCCCTGAC_T_b38         -0.1799 6.30e-27   5.73e-21
       RPA3 Brain - Frontal Cortex (BA9)   rs6463710           chr7_7638178_C_T_b38         -0.5327 7.55e-10   2.39e-05
       RPA3          Brain - Hippocampus  rs73340632           chr7_7592796_C_T_b38          0.3429 1.43e-07   2.37e-03
       RPA3                  Whole Blood  rs28916294         chr7_7640468_CTG_C_b38          0.4397 3.14e-07   1.08e-03
      PDCD1                  Whole Blood  rs34667857         chr2_241808501_C_T_b38          0.1314 4.06e-09   1.34e-05
     HAVCR2                  Whole Blood   rs6891966         chr5_157099320_A_G_b38         -0.1362 5.62e-09   2.45e-05
     RAD51C Brain - Fronta

Cross-validate RAD51C and DNTT eQTL claims using eQTLGen API (31,684 blood samples)

In [147]:

import requests
import pandas as pd

# eQTLGen API - large blood eQTL resource (31,684 individuals, Nature Genetics 2021)
# Source: https://www.eqtlgen.org/phase1.html
EQTLGEN_API = "https://eqtlgen.org/api"

# Test eQTLGen API structure
print("=== Testing eQTLGen API ===")
# Try gene-based query
for gene in ["RAD51C", "DNTT", "ATM", "RPA3"]:
    try:
        resp = requests.get(f"{EQTLGEN_API}/v1/cis-eqtls/gene/{gene}", timeout=15)
        print(f"{gene}: status={resp.status_code}, len={len(resp.text)}")
        if resp.status_code == 200:
            data = resp.json()
            print(f"  Keys: {list(data.keys()) if isinstance(data, dict) else type(data)}")
            if isinstance(data, list) and len(data) > 0:
                print(f"  Top hit: {data[0]}")
    except Exception as e:
        print(f"{gene}: error - {e}")


=== Testing eQTLGen API ===
RAD51C: status=404, len=196
DNTT: status=404, len=196
ATM: status=404, len=196
RPA3: status=404, len=196


Try GTEx v8 egene endpoint and correct Ensembl IDs for RAD51C and DNTT

In [149]:

import requests
import time

GTEX_API = "https://gtexportal.org/api/v2"

# First get correct Ensembl IDs via GTEx gene search
genes_to_check = {"RAD51C": None, "DNTT": None, "ATM": None, "RPA3": None}

for gene in genes_to_check:
    resp = requests.get(f"{GTEX_API}/reference/gene",
                        params={"geneSymbol": gene, 
                                "gencodeVersion": "v26",
                                "genomeBuild": "GRCh38/hg38"},
                        timeout=15)
    print(f"{gene}: status={resp.status_code}")
    if resp.status_code == 200:
        data = resp.json().get("data", [])
        if data:
            gid = data[0].get("gencodeId", "")
            genes_to_check[gene] = gid
            print(f"  Ensembl ID: {gid}")
        else:
            print(f"  No data returned")
    time.sleep(0.3)

print("\nGene ID map:", genes_to_check)


RAD51C: status=422
DNTT: status=422
ATM: status=422
RPA3: status=422

Gene ID map: {'RAD51C': None, 'DNTT': None, 'ATM': None, 'RPA3': None}


Check GTEx API v2 OpenAPI spec for correct parameter names

In [151]:

import requests

# Source: https://gtexportal.org/api/v2/openapi.json
resp = requests.get("https://gtexportal.org/api/v2/openapi.json", timeout=20)
print(f"OpenAPI spec status: {resp.status_code}")
if resp.status_code == 200:
    spec = resp.json()
    # Find singleTissueEqtl and gene reference endpoints
    paths = spec.get("paths", {})
    for path in paths:
        if "eqtl" in path.lower() or "gene" in path.lower():
            print(f"\nPath: {path}")
            for method, details in paths[path].items():
                params = details.get("parameters", [])
                for p in params:
                    print(f"  param: {p.get('name')} | in: {p.get('in')} | required: {p.get('required')} | schema: {p.get('schema', {})}")


OpenAPI spec status: 200

Path: /api/v2/association/egene
  param: tissueSiteDetailId | in: query | required: False | schema: {'type': 'array', 'items': {'anyOf': [{'$ref': '#/components/schemas/TissueSiteDetailId'}, {'$ref': '#/components/schemas/TissueSiteOntologyId'}]}, 'description': 'A list of Tissue IDs of the tissue(s) of interest. Can be a GTEx specific ID or an Ontology ID', 'title': 'Tissuesitedetailid'}
  param: datasetId | in: query | required: False | schema: {'$ref': '#/components/schemas/DatasetId', 'description': 'Unique identifier of a dataset. Usually includes a data source and data release.', 'default': 'gtex_v10'}
  param: page | in: query | required: False | schema: {'type': 'integer', 'maximum': 1000000.0, 'minimum': 0, 'description': 'The 0-based numeric ID of the page to retrieve', 'default': 0, 'title': 'Page'}
  param: itemsPerPage | in: query | required: False | schema: {'type': 'integer', 'maximum': 100000, 'minimum': 1, 'default': 250, 'title': 'Itemsperpag

Query GTEx API with correct parameter format (geneId, array params)

In [153]:

import requests
import time

GTEX_API = "https://gtexportal.org/api/v2"

# Correct parameter: geneId (not geneSymbol), passed as repeated query params for arrays
key_genes = ["RAD51C", "DNTT", "ATM", "RPA3", "MSH2", "XRCC6", "PDCD1", "HAVCR2"]

# Step 1: Get versioned Ensembl IDs using correct geneId parameter
print("=== Getting Ensembl IDs ===")
params = [("geneId", g) for g in key_genes]
params += [("gencodeVersion", "v26"), ("genomeBuild", "GRCh38/hg38")]

resp = requests.get(f"{GTEX_API}/reference/gene", params=params, timeout=20)
print(f"Status: {resp.status_code}")

gene_id_map = {}
if resp.status_code == 200:
    data = resp.json().get("data", [])
    print(f"Returned {len(data)} gene records")
    for rec in data:
        sym = rec.get("geneSymbol", "")
        gid = rec.get("gencodeId", "")
        gene_id_map[sym] = gid
        print(f"  {sym}: {gid}")
else:
    print(f"Error: {resp.text[:300]}")


=== Getting Ensembl IDs ===
Status: 200
Returned 8 gene records
  MSH2: ENSG00000095002.13
  PDCD1: ENSG00000188389.10
  HAVCR2: ENSG00000135077.8
  RPA3: ENSG00000106399.11
  DNTT: ENSG00000107447.7
  ATM: ENSG00000149311.18
  RAD51C: ENSG00000108384.14
  XRCC6: ENSG00000196419.12


Query GTEx v8 eQTLs for DDR/T-cell genes using correct versioned Ensembl IDs

In [155]:

import requests
import pandas as pd
import time

GTEX_API = "https://gtexportal.org/api/v2"

# Verified Ensembl IDs from previous step
gene_id_map = {
    "RAD51C": "ENSG00000108384.14",
    "DNTT":   "ENSG00000107447.7",
    "ATM":    "ENSG00000149311.18",
    "RPA3":   "ENSG00000106399.11",
    "MSH2":   "ENSG00000095002.13",
    "XRCC6":  "ENSG00000196419.12",
    "PDCD1":  "ENSG00000188389.10",
    "HAVCR2": "ENSG00000135077.8",
}

tissues = ["Whole_Blood", "Brain_Frontal_Cortex_BA9", "Brain_Hippocampus"]

all_results = []

for gene_sym, gencode_id in gene_id_map.items():
    for tissue in tissues:
        params = [
            ("gencodeId", gencode_id),
            ("tissueSiteDetailId", tissue),
            ("datasetId", "gtex_v8"),
            ("itemsPerPage", 5),
            ("page", 0),
        ]
        try:
            resp = requests.get(f"{GTEX_API}/association/singleTissueEqtl",
                                params=params, timeout=20)
            if resp.status_code == 200:
                data = resp.json().get("data", [])
                if data:
                    top = data[0]
                    all_results.append({
                        "gene": gene_sym,
                        "tissue": tissue,
                        "variantId": top.get("variantId", ""),
                        "rsId": top.get("snpId", ""),
                        "nes": top.get("nes", None),
                        "pval_nominal": top.get("pval_nominal", None),
                        "qval": top.get("qval", None),
                        "n_results": len(data),
                    })
                    print(f"  {gene_sym} | {tissue}: {len(data)} hits, top p={top.get('pval_nominal','?'):.2e}, q={top.get('qval','?'):.2e}, NES={top.get('nes','?'):.3f}")
                else:
                    print(f"  {gene_sym} | {tissue}: 0 hits")
            else:
                print(f"  {gene_sym} | {tissue}: HTTP {resp.status_code}")
        except Exception as e:
            print(f"  {gene_sym} | {tissue}: error - {e}")
        time.sleep(0.25)

df = pd.DataFrame(all_results)
print(f"\nTotal significant hits retrieved: {len(df)}")
print(df[df['qval'].notna() & (df['qval'] < 0.05)][['gene','tissue','rsId','nes','pval_nominal','qval']].to_string())


  RAD51C | Whole_Blood: error - Unknown format code 'e' for object of type 'str'
  RAD51C | Brain_Frontal_Cortex_BA9: error - Unknown format code 'e' for object of type 'str'
  RAD51C | Brain_Hippocampus: error - Unknown format code 'e' for object of type 'str'
  DNTT | Whole_Blood: 0 hits
  DNTT | Brain_Frontal_Cortex_BA9: 0 hits
  DNTT | Brain_Hippocampus: 0 hits
  ATM | Whole_Blood: error - Unknown format code 'e' for object of type 'str'
  ATM | Brain_Frontal_Cortex_BA9: 0 hits
  ATM | Brain_Hippocampus: 0 hits
  RPA3 | Whole_Blood: error - Unknown format code 'e' for object of type 'str'
  RPA3 | Brain_Frontal_Cortex_BA9: error - Unknown format code 'e' for object of type 'str'
  RPA3 | Brain_Hippocampus: error - Unknown format code 'e' for object of type 'str'
  MSH2 | Whole_Blood: error - Unknown format code 'e' for object of type 'str'
  MSH2 | Brain_Frontal_Cortex_BA9: 0 hits
  MSH2 | Brain_Hippocampus: error - Unknown format code 'e' for object of type 'str'
  XRCC6 | Whole_B

Fix parsing and collect all GTEx v8 eQTL results properly

In [157]:

import requests
import pandas as pd
import time

GTEX_API = "https://gtexportal.org/api/v2"

gene_id_map = {
    "RAD51C": "ENSG00000108384.14",
    "DNTT":   "ENSG00000107447.7",
    "ATM":    "ENSG00000149311.18",
    "RPA3":   "ENSG00000106399.11",
    "MSH2":   "ENSG00000095002.13",
    "XRCC6":  "ENSG00000196419.12",
    "PDCD1":  "ENSG00000188389.10",
    "HAVCR2": "ENSG00000135077.8",
}

tissues = ["Whole_Blood", "Brain_Frontal_Cortex_BA9", "Brain_Hippocampus"]

all_results = []

for gene_sym, gencode_id in gene_id_map.items():
    for tissue in tissues:
        params = [
            ("gencodeId", gencode_id),
            ("tissueSiteDetailId", tissue),
            ("datasetId", "gtex_v8"),
            ("itemsPerPage", 5),
            ("page", 0),
        ]
        try:
            resp = requests.get(f"{GTEX_API}/association/singleTissueEqtl",
                                params=params, timeout=20)
            if resp.status_code == 200:
                data = resp.json().get("data", [])
                if data:
                    # Inspect raw fields from first result
                    top = data[0]
                    # Convert to float safely
                    nes = float(top.get("nes", "nan")) if top.get("nes") is not None else None
                    pval = float(top.get("pval_nominal", "nan")) if top.get("pval_nominal") is not None else None
                    qval = float(top.get("qval", "nan")) if top.get("qval") is not None else None
                    
                    all_results.append({
                        "gene": gene_sym,
                        "tissue": tissue,
                        "variantId": top.get("variantId", ""),
                        "rsId": top.get("snpId", ""),
                        "nes": nes,
                        "pval_nominal": pval,
                        "qval": qval,
                        "n_hits": len(data),
                    })
                    print(f"  {gene_sym} | {tissue}: {len(data)} hits | p={pval} | q={qval} | NES={nes}")
                else:
                    print(f"  {gene_sym} | {tissue}: 0 hits")
            else:
                print(f"  {gene_sym} | {tissue}: HTTP {resp.status_code} - {resp.text[:100]}")
        except Exception as e:
            print(f"  {gene_sym} | {tissue}: exception - {e}")
        time.sleep(0.25)

df = pd.DataFrame(all_results)
print(f"\n=== Summary: {len(df)} gene-tissue pairs with data ===")
print(df[['gene','tissue','rsId','nes','pval_nominal','qval']].to_string())


  RAD51C | Whole_Blood: 5 hits | p=None | q=None | NES=0.554534
  RAD51C | Brain_Frontal_Cortex_BA9: 5 hits | p=None | q=None | NES=-0.996516
  RAD51C | Brain_Hippocampus: 5 hits | p=None | q=None | NES=-0.278962
  DNTT | Whole_Blood: 0 hits
  DNTT | Brain_Frontal_Cortex_BA9: 0 hits
  DNTT | Brain_Hippocampus: 0 hits
  ATM | Whole_Blood: 5 hits | p=None | q=None | NES=0.17194
  ATM | Brain_Frontal_Cortex_BA9: 0 hits
  ATM | Brain_Hippocampus: 0 hits
  RPA3 | Whole_Blood: 2 hits | p=None | q=None | NES=0.439656
  RPA3 | Brain_Frontal_Cortex_BA9: 5 hits | p=None | q=None | NES=0.398558
  RPA3 | Brain_Hippocampus: 5 hits | p=None | q=None | NES=0.334193
  MSH2 | Whole_Blood: 5 hits | p=None | q=None | NES=0.277725
  MSH2 | Brain_Frontal_Cortex_BA9: 0 hits
  MSH2 | Brain_Hippocampus: 5 hits | p=None | q=None | NES=0.242576
  XRCC6 | Whole_Blood: 5 hits | p=None | q=None | NES=0.216942
  XRCC6 | Brain_Frontal_Cortex_BA9: 5 hits | p=None | q=None | NES=0.230579
  XRCC6 | Brain_Hippocampus: 0

Inspect raw GTEx API response fields and retrieve full eQTL statistics

In [159]:

import requests
import pandas as pd
import time

GTEX_API = "https://gtexportal.org/api/v2"

# Inspect raw field names from one result
resp = requests.get(f"{GTEX_API}/association/singleTissueEqtl",
                    params=[("gencodeId", "ENSG00000108384.14"),
                            ("tissueSiteDetailId", "Whole_Blood"),
                            ("datasetId", "gtex_v8"),
                            ("itemsPerPage", 3)],
                    timeout=20)
data = resp.json().get("data", [])
print("=== RAD51C Whole Blood - raw field names ===")
if data:
    print("All fields:", list(data[0].keys()))
    print("\nFull first record:")
    for k, v in data[0].items():
        print(f"  {k}: {v}")


=== RAD51C Whole Blood - raw field names ===
All fields: ['snpId', 'pos', 'snpIdUpper', 'variantId', 'geneSymbol', 'pValue', 'geneSymbolUpper', 'datasetId', 'tissueSiteDetailId', 'ontologyId', 'chromosome', 'gencodeId', 'nes']

Full first record:
  snpId: rs117542958
  pos: 57869610
  snpIdUpper: RS117542958
  variantId: chr17_57869610_C_T_b38
  geneSymbol: RAD51C
  pValue: 3.21651e-05
  geneSymbolUpper: RAD51C
  datasetId: gtex_v8
  tissueSiteDetailId: Whole_Blood
  ontologyId: UBERON:0013756
  chromosome: chr17
  gencodeId: ENSG00000108384.14
  nes: 0.554534


Query GTEx v8 eQTLs with correct field names + egene endpoint for FDR significance

In [161]:

import requests
import pandas as pd
import time

GTEX_API = "https://gtexportal.org/api/v2"

gene_id_map = {
    "RAD51C": "ENSG00000108384.14",
    "DNTT":   "ENSG00000107447.7",
    "ATM":    "ENSG00000149311.18",
    "RPA3":   "ENSG00000106399.11",
    "MSH2":   "ENSG00000095002.13",
    "XRCC6":  "ENSG00000196419.12",
    "PDCD1":  "ENSG00000188389.10",
    "HAVCR2": "ENSG00000135077.8",
}

tissues = ["Whole_Blood", "Brain_Frontal_Cortex_BA9", "Brain_Hippocampus"]

# ── Step 1: Get top eQTL per gene-tissue (nominal p-value, NES) ──────────────
top_eqtl = []
for gene_sym, gencode_id in gene_id_map.items():
    for tissue in tissues:
        params = [
            ("gencodeId", gencode_id),
            ("tissueSiteDetailId", tissue),
            ("datasetId", "gtex_v8"),
            ("itemsPerPage", 1),
        ]
        resp = requests.get(f"{GTEX_API}/association/singleTissueEqtl",
                            params=params, timeout=20)
        if resp.status_code == 200:
            data = resp.json().get("data", [])
            if data:
                top = data[0]
                top_eqtl.append({
                    "gene": gene_sym,
                    "gencodeId": gencode_id,
                    "tissue": tissue,
                    "variantId": top.get("variantId", ""),
                    "rsId": top.get("snpId", ""),
                    "nes": float(top.get("nes", 0)),
                    "pValue": float(top.get("pValue", 1)),
                })
        time.sleep(0.2)

df_top = pd.DataFrame(top_eqtl)
print(f"Top eQTLs retrieved: {len(df_top)} gene-tissue pairs")

# ── Step 2: Get eGene status (FDR q-value) per tissue ────────────────────────
# egene endpoint returns FDR-corrected significance for each gene per tissue
egene_results = []
for tissue in tissues:
    params = [
        ("tissueSiteDetailId", tissue),
        ("datasetId", "gtex_v8"),
        ("itemsPerPage", 50000),
    ]
    resp = requests.get(f"{GTEX_API}/association/egene", params=params, timeout=60)
    if resp.status_code == 200:
        data = resp.json().get("data", [])
        print(f"  {tissue}: {len(data)} eGenes total")
        for rec in data:
            sym = rec.get("geneSymbol", "")
            if sym in gene_id_map:
                egene_results.append({
                    "gene": sym,
                    "tissue": tissue,
                    "qval": float(rec.get("qval", 1)),
                    "pval_beta": float(rec.get("pval_beta", 1)),
                    "is_egene": rec.get("qval", 1) < 0.05,
                })
                print(f"    Found {sym}: qval={rec.get('qval')}, pval_beta={rec.get('pval_beta')}")
    time.sleep(0.5)

df_egene = pd.DataFrame(egene_results) if egene_results else pd.DataFrame(
    columns=["gene","tissue","qval","pval_beta","is_egene"])

# ── Step 3: Merge top eQTL with eGene significance ───────────────────────────
if not df_egene.empty:
    df_final = df_top.merge(df_egene[["gene","tissue","qval","pval_beta","is_egene"]],
                            on=["gene","tissue"], how="left")
else:
    df_final = df_top.copy()
    df_final["qval"] = None
    df_final["is_egene"] = None

print("\n=== Final eQTL Table ===")
print(df_final[["gene","tissue","rsId","nes","pValue","qval","is_egene"]].to_string())


Top eQTLs retrieved: 13 gene-tissue pairs
  Whole_Blood: 12360 eGenes total
    Found MSH2: qval=None, pval_beta=None
    Found PDCD1: qval=None, pval_beta=None
    Found HAVCR2: qval=None, pval_beta=None
    Found RPA3: qval=None, pval_beta=None
    Found ATM: qval=None, pval_beta=None
    Found RAD51C: qval=None, pval_beta=None
    Found XRCC6: qval=None, pval_beta=None
  Brain_Frontal_Cortex_BA9: 7335 eGenes total
    Found RPA3: qval=None, pval_beta=None
    Found RAD51C: qval=None, pval_beta=None
    Found XRCC6: qval=None, pval_beta=None
  Brain_Hippocampus: 5517 eGenes total
    Found MSH2: qval=None, pval_beta=None
    Found RPA3: qval=None, pval_beta=None
    Found RAD51C: qval=None, pval_beta=None

=== Final eQTL Table ===
      gene                    tissue         rsId       nes        pValue  qval  is_egene
0   RAD51C               Whole_Blood  rs117542958  0.554534  3.216510e-05   1.0     False
1   RAD51C  Brain_Frontal_Cortex_BA9  rs149392983 -0.996516  4.372310e-05   1

Inspect raw egene record fields to find correct FDR field names

In [163]:

import requests

GTEX_API = "https://gtexportal.org/api/v2"

# Get a few egene records from Whole Blood and inspect all fields
resp = requests.get(f"{GTEX_API}/association/egene",
                    params=[("tissueSiteDetailId", "Whole_Blood"),
                            ("datasetId", "gtex_v8"),
                            ("itemsPerPage", 3)],
                    timeout=30)

data = resp.json().get("data", [])
print(f"Status: {resp.status_code}, records: {len(data)}")
if data:
    print("\nAll fields in egene record:")
    for k, v in data[0].items():
        print(f"  {k}: {v}")
    
    # Also find RAD51C specifically
    print("\n--- Searching for RAD51C in egene results ---")
    resp2 = requests.get(f"{GTEX_API}/association/egene",
                         params=[("tissueSiteDetailId", "Whole_Blood"),
                                 ("datasetId", "gtex_v8"),
                                 ("itemsPerPage", 50000)],
                         timeout=60)
    all_data = resp2.json().get("data", [])
    for rec in all_data:
        if rec.get("geneSymbol") in ["RAD51C", "RPA3", "MSH2", "DNTT"]:
            print(f"\n{rec.get('geneSymbol')}:")
            for k, v in rec.items():
                print(f"  {k}: {v}")


Status: 200, records: 3

All fields in egene record:
  tissueSiteDetailId: Whole_Blood
  ontologyId: UBERON:0013756
  datasetId: gtex_v8
  empiricalPValue: 1.05258e-09
  gencodeId: ENSG00000227232.5
  geneSymbol: WASH7P
  log2AllelicFoldChange: 0.711097
  pValue: 3.85133e-13
  pValueThreshold: 0.000393213
  qValue: 1.25773e-09

--- Searching for RAD51C in egene results ---

MSH2:
  tissueSiteDetailId: Whole_Blood
  ontologyId: UBERON:0013756
  datasetId: gtex_v8
  empiricalPValue: 8.88188e-33
  gencodeId: ENSG00000095002.13
  geneSymbol: MSH2
  log2AllelicFoldChange: 0.642818
  pValue: 3.0738399999999995e-39
  pValueThreshold: 0.000159795
  qValue: 3.48931e-32

RPA3:
  tissueSiteDetailId: Whole_Blood
  ontologyId: UBERON:0013756
  datasetId: gtex_v8
  empiricalPValue: 0.00161885
  gencodeId: ENSG00000106399.11
  geneSymbol: RPA3
  log2AllelicFoldChange: 0.439318
  pValue: 3.13754e-07
  pValueThreshold: 7.72745e-05
  qValue: 0.00107873

RAD51C:
  tissueSiteDetailId: Whole_Blood
  ontolo

Build complete verified GTEx v8 eQTL table for all DDR/T-cell genes across 3 tissues

In [165]:

import requests
import pandas as pd
import time

GTEX_API = "https://gtexportal.org/api/v2"

gene_id_map = {
    "RAD51C": "ENSG00000108384.14",
    "DNTT":   "ENSG00000107447.7",
    "ATM":    "ENSG00000149311.18",
    "RPA3":   "ENSG00000106399.11",
    "MSH2":   "ENSG00000095002.13",
    "XRCC6":  "ENSG00000196419.12",
    "PDCD1":  "ENSG00000188389.10",
    "HAVCR2": "ENSG00000135077.8",
    # Add remaining genes from original query
    "CHEK1":  None,
    "PRKDC":  None,
    "BRCA1":  None,
    "BRCA2":  None,
    "CHEK2":  None,
    "LAG3":   None,
}

# Get Ensembl IDs for remaining genes
missing = [g for g, v in gene_id_map.items() if v is None]
if missing:
    params = [("geneId", g) for g in missing]
    params += [("gencodeVersion", "v26"), ("genomeBuild", "GRCh38/hg38")]
    resp = requests.get(f"{GTEX_API}/reference/gene", params=params, timeout=20)
    if resp.status_code == 200:
        for rec in resp.json().get("data", []):
            gene_id_map[rec["geneSymbol"]] = rec["gencodeId"]
            print(f"  {rec['geneSymbol']}: {rec['gencodeId']}")

tissues = ["Whole_Blood", "Brain_Frontal_Cortex_BA9", "Brain_Hippocampus"]

# ── Collect egene data for all 3 tissues ─────────────────────────────────────
print("\n=== Collecting eGene significance (FDR q-values) ===")
egene_lookup = {}  # (gene, tissue) -> record
for tissue in tissues:
    resp = requests.get(f"{GTEX_API}/association/egene",
                        params=[("tissueSiteDetailId", tissue),
                                ("datasetId", "gtex_v8"),
                                ("itemsPerPage", 50000)],
                        timeout=60)
    if resp.status_code == 200:
        data = resp.json().get("data", [])
        print(f"  {tissue}: {len(data)} eGenes")
        for rec in data:
            sym = rec.get("geneSymbol", "")
            if sym in gene_id_map:
                egene_lookup[(sym, tissue)] = rec
    time.sleep(0.5)

# ── Collect top eQTL (lead SNP) per gene-tissue ──────────────────────────────
print("\n=== Collecting top eQTL per gene-tissue ===")
top_eqtl = []
for gene_sym, gencode_id in gene_id_map.items():
    if gencode_id is None:
        continue
    for tissue in tissues:
        params = [("gencodeId", gencode_id),
                  ("tissueSiteDetailId", tissue),
                  ("datasetId", "gtex_v8"),
                  ("itemsPerPage", 1)]
        resp = requests.get(f"{GTEX_API}/association/singleTissueEqtl",
                            params=params, timeout=20)
        if resp.status_code == 200:
            data = resp.json().get("data", [])
            if data:
                top = data[0]
                egene_rec = egene_lookup.get((gene_sym, tissue), {})
                top_eqtl.append({
                    "gene":         gene_sym,
                    "tissue":       tissue,
                    "lead_variant": top.get("variantId", ""),
                    "rsId":         top.get("snpId", ""),
                    "NES":          float(top.get("nes", 0)),
                    "pValue_nominal": float(top.get("pValue", 1)),
                    "qValue_egene": float(egene_rec.get("qValue", float("nan"))) if egene_rec else float("nan"),
                    "empiricalP":   float(egene_rec.get("empiricalPValue", float("nan"))) if egene_rec else float("nan"),
                    "log2AFC":      float(egene_rec.get("log2AllelicFoldChange", float("nan"))) if egene_rec else float("nan"),
                    "is_egene":     egene_rec.get("qValue", 1) < 0.05 if egene_rec else False,
                })
        time.sleep(0.15)

df = pd.DataFrame(top_eqtl)

# ── Display results ───────────────────────────────────────────────────────────
print("\n=== Verified GTEx v8 eQTL Results (DDR & T-cell genes) ===")
print(f"Total gene-tissue pairs with eQTL data: {len(df)}")
print(f"Significant eGenes (qValue < 0.05): {df['is_egene'].sum()}")

sig = df[df['is_egene']].sort_values("qValue_egene")
print("\n--- Significant eGenes ---")
print(sig[["gene","tissue","rsId","NES","pValue_nominal","qValue_egene","log2AFC"]].to_string(index=False))

print("\n--- All results ---")
print(df[["gene","tissue","rsId","NES","pValue_nominal","qValue_egene","is_egene"]].to_string(index=False))


  PRKDC: ENSG00000253729.7
  CHEK1: ENSG00000149554.12
  LAG3: ENSG00000089692.8
  BRCA2: ENSG00000139618.14
  BRCA1: ENSG00000012048.20
  CHEK2: ENSG00000183765.20

=== Collecting eGene significance (FDR q-values) ===
  Whole_Blood: 12360 eGenes
  Brain_Frontal_Cortex_BA9: 7335 eGenes
  Brain_Hippocampus: 5517 eGenes

=== Collecting top eQTL per gene-tissue ===

=== Verified GTEx v8 eQTL Results (DDR & T-cell genes) ===
Total gene-tissue pairs with eQTL data: 14
Significant eGenes (qValue < 0.05): 14

--- Significant eGenes ---
  gene                   tissue        rsId       NES  pValue_nominal  qValue_egene   log2AFC
RAD51C              Whole_Blood rs117542958  0.554534    3.216510e-05  1.614960e-68  0.884005
  MSH2              Whole_Blood   rs4380295  0.277725    2.229660e-07  3.489310e-32  0.642818
RAD51C Brain_Frontal_Cortex_BA9 rs149392983 -0.996516    4.372310e-05  6.583140e-30 -0.552575
   ATM              Whole_Blood rs148680806  0.171940    7.210870e-05  5.729980e-21 -0.26

Save verified eQTL results and create bubble plot visualization

In [167]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# ── Verified eQTL data from GTEx v8 API ──────────────────────────────────────
eqtl_data = [
    # gene, tissue, rsId, NES, pValue_nominal, qValue_egene, log2AFC, pathway
    ("RAD51C", "Whole_Blood",              "rs117542958",  0.554534, 3.22e-05, 1.61e-68,  0.884, "HR"),
    ("RAD51C", "Brain_Frontal_Cortex_BA9", "rs149392983", -0.996516, 4.37e-05, 6.58e-30, -0.553, "HR"),
    ("RAD51C", "Brain_Hippocampus",        "rs2014835",   -0.278962, 3.09e-05, 1.16e-07, -0.413, "HR"),
    ("MSH2",   "Whole_Blood",              "rs4380295",    0.277725, 2.23e-07, 3.49e-32,  0.643, "MMR"),
    ("MSH2",   "Brain_Hippocampus",        "rs10202878",   0.242576, 1.22e-05, 1.63e-02,  0.161, "MMR"),
    ("ATM",    "Whole_Blood",              "rs148680806",  0.171940, 7.21e-05, 5.73e-21, -0.262, "ATM/ATR"),
    ("XRCC6",  "Whole_Blood",              "rs116897802",  0.216942, 2.53e-04, 3.54e-09,  0.267, "NHEJ"),
    ("XRCC6",  "Brain_Frontal_Cortex_BA9", "rs73165107",   0.230579, 5.08e-05, 4.26e-02,  0.083, "NHEJ"),
    ("RPA3",   "Whole_Blood",              "rs28916294",   0.439656, 3.14e-07, 1.08e-03,  0.439, "HR"),
    ("RPA3",   "Brain_Frontal_Cortex_BA9", "rs10269244",   0.398558, 1.34e-05, 2.39e-05, -0.276, "HR"),
    ("RPA3",   "Brain_Hippocampus",        "rs6963106",    0.334193, 7.05e-06, 2.37e-03,  0.289, "HR"),
    ("PDCD1",  "Whole_Blood",              "rs75401855",  -0.320264, 2.22e-05, 1.34e-05,  0.380, "T-cell"),
    ("HAVCR2", "Whole_Blood",              "rs157679",    -0.107606, 7.63e-05, 2.45e-05, -0.275, "T-cell"),
    ("CHEK1",  "Brain_Hippocampus",        "rs76118307",   0.605819, 2.17e-05, 3.05e-04,  0.558, "ATM/ATR"),
]

cols = ["gene","tissue","rsId","NES","pValue_nominal","qValue_egene","log2AFC","pathway"]
df = pd.DataFrame(eqtl_data, columns=cols)

# Clean tissue labels for display
tissue_labels = {
    "Whole_Blood": "Whole Blood",
    "Brain_Frontal_Cortex_BA9": "Brain Frontal Cortex",
    "Brain_Hippocampus": "Brain Hippocampus",
}
df["tissue_label"] = df["tissue"].map(tissue_labels)
df["-log10q"] = -np.log10(df["qValue_egene"])

# Save CSV
df.to_csv("/mnt/results/gtex_v8_eqtl_verified.csv", index=False)
print("Saved: gtex_v8_eqtl_verified.csv")

# ── Bubble plot: gene × tissue, size = -log10(qValue), color = NES ───────────
sns.set_style("ticks")
fig, ax = plt.subplots(figsize=(8, 6))

# Pivot for layout
genes_ordered = ["RAD51C", "MSH2", "ATM", "XRCC6", "RPA3", "CHEK1", "PDCD1", "HAVCR2"]
tissues_ordered = ["Whole Blood", "Brain Frontal Cortex", "Brain Hippocampus"]

# Color by NES (diverging: blue=negative, red=positive)
norm = mcolors.TwoSlopeNorm(vmin=-1.1, vcenter=0, vmax=0.9)
cmap = plt.cm.RdBu_r

# Pathway colors for gene labels
pathway_colors = {
    "HR": "#0279EE", "MMR": "#75A025", "ATM/ATR": "#FF9400",
    "NHEJ": "#E9ED4C", "T-cell": "#FD9BED"
}
pathway_map = {
    "RAD51C": "HR", "RPA3": "HR", "MSH2": "MMR",
    "ATM": "ATM/ATR", "CHEK1": "ATM/ATR",
    "XRCC6": "NHEJ", "PDCD1": "T-cell", "HAVCR2": "T-cell"
}

for _, row in df.iterrows():
    x = tissues_ordered.index(row["tissue_label"])
    y = genes_ordered.index(row["gene"])
    size = row["-log10q"] * 12  # scale bubble size
    color = cmap(norm(row["NES"]))
    ax.scatter(x, y, s=size, c=[color], alpha=0.85, edgecolors="k", linewidths=0.5, zorder=3)

# Axes formatting
ax.set_xticks(range(len(tissues_ordered)))
ax.set_xticklabels(tissues_ordered, fontsize=11)
ax.set_yticks(range(len(genes_ordered)))

# Color gene labels by pathway
for i, gene in enumerate(genes_ordered):
    pway = pathway_map.get(gene, "HR")
    ax.get_yticklabels()  # ensure ticks exist
ax.set_yticklabels(genes_ordered, fontsize=11)

# Color y-tick labels by pathway
for label in ax.get_yticklabels():
    gene = label.get_text()
    pway = pathway_map.get(gene, "HR")
    label.set_color(pathway_colors[pway])
    label.set_fontweight("bold")

ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, len(genes_ordered) - 0.5)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xlabel("Tissue", fontsize=12)
ax.set_ylabel("Gene", fontsize=12)
ax.set_title("GTEx v8 eQTLs: DDR & T-cell Genes\n(bubble size = −log₁₀ q-value; color = NES)", fontsize=12)

# Colorbar for NES
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label("NES (effect size)", fontsize=10)

# Size legend
for s_val, s_label in [(10, "q=0.1"), (30, "q=1e-5"), (70, "q=1e-20"), (150, "q=1e-50")]:
    ax.scatter([], [], s=s_val*12, c="gray", alpha=0.6, edgecolors="k",
               linewidths=0.5, label=s_label)
ax.legend(title="q-value", loc="lower right", fontsize=8, title_fontsize=9,
          framealpha=0.8, scatterpoints=1)

# Pathway legend (colored text)
from matplotlib.lines import Line2D
pathway_legend = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                         markersize=8, label=p)
                  for p, c in pathway_colors.items()]
ax2_legend = ax.legend(handles=pathway_legend, title="Pathway", loc="upper right",
                       fontsize=8, title_fontsize=9, framealpha=0.8)
ax.add_artist(ax2_legend)

# Re-add size legend
for s_val, s_label in [(10, "q=0.1"), (30, "q=1e-5"), (70, "q=1e-20"), (150, "q=1e-50")]:
    ax.scatter([], [], s=s_val*12, c="gray", alpha=0.6, edgecolors="k",
               linewidths=0.5, label=s_label)
ax.legend(title="q-value", loc="lower right", fontsize=8, title_fontsize=9,
          framealpha=0.8, scatterpoints=1)

plt.tight_layout()
plt.savefig("/mnt/results/gtex_v8_eqtl_bubble.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/gtex_v8_eqtl_bubble.svg", bbox_inches="tight")
plt.close()
print("Saved: gtex_v8_eqtl_bubble.png / .svg")


Saved: gtex_v8_eqtl_verified.csv
Saved: gtex_v8_eqtl_bubble.png / .svg


Regenerate eQTL bubble plot with fixed legends (no overlap, no duplication)

In [169]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

# ── Data ──────────────────────────────────────────────────────────────────────
eqtl_data = [
    ("RAD51C", "Whole Blood",          "rs117542958",  0.554534, 1.61e-68,  "HR"),
    ("RAD51C", "Brain Frontal Cortex", "rs149392983", -0.996516, 6.58e-30,  "HR"),
    ("RAD51C", "Brain Hippocampus",    "rs2014835",   -0.278962, 1.16e-07,  "HR"),
    ("MSH2",   "Whole Blood",          "rs4380295",    0.277725, 3.49e-32,  "MMR"),
    ("MSH2",   "Brain Hippocampus",    "rs10202878",   0.242576, 1.63e-02,  "MMR"),
    ("ATM",    "Whole Blood",          "rs148680806",  0.171940, 5.73e-21,  "ATM/ATR"),
    ("XRCC6",  "Whole Blood",          "rs116897802",  0.216942, 3.54e-09,  "NHEJ"),
    ("XRCC6",  "Brain Frontal Cortex", "rs73165107",   0.230579, 4.26e-02,  "NHEJ"),
    ("RPA3",   "Whole Blood",          "rs28916294",   0.439656, 1.08e-03,  "HR"),
    ("RPA3",   "Brain Frontal Cortex", "rs10269244",   0.398558, 2.39e-05,  "HR"),
    ("RPA3",   "Brain Hippocampus",    "rs6963106",    0.334193, 2.37e-03,  "HR"),
    ("PDCD1",  "Whole Blood",          "rs75401855",  -0.320264, 1.34e-05,  "T-cell"),
    ("HAVCR2", "Whole Blood",          "rs157679",    -0.107606, 2.45e-05,  "T-cell"),
    ("CHEK1",  "Brain Hippocampus",    "rs76118307",   0.605819, 3.05e-04,  "ATM/ATR"),
]

cols = ["gene","tissue","rsId","NES","qValue","pathway"]
df = pd.DataFrame(eqtl_data, columns=cols)
df["-log10q"] = -np.log10(df["qValue"])

genes_ordered  = ["RAD51C", "RPA3", "MSH2", "ATM", "CHEK1", "XRCC6", "PDCD1", "HAVCR2"]
tissues_ordered = ["Whole Blood", "Brain Frontal Cortex", "Brain Hippocampus"]

pathway_colors = {
    "HR": "#0279EE", "MMR": "#75A025", "ATM/ATR": "#FF9400",
    "NHEJ": "#c8a800", "T-cell": "#c040a0"
}
pathway_map = {r[0]: r[5] for r in eqtl_data}

# ── Figure with extra right margin for legends ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(right=0.72)   # leave room on right for legends

norm = mcolors.TwoSlopeNorm(vmin=-1.1, vcenter=0, vmax=0.9)
cmap = plt.cm.RdBu_r

# Scale: max bubble ~400pt², min ~30pt²
q_min, q_max = df["-log10q"].min(), df["-log10q"].max()
def scale_size(v):
    return 30 + 370 * (v - q_min) / (q_max - q_min + 1e-9)

for _, row in df.iterrows():
    x = tissues_ordered.index(row["tissue"])
    y = genes_ordered.index(row["gene"])
    s = scale_size(row["-log10q"])
    c = cmap(norm(row["NES"]))
    ax.scatter(x, y, s=s, c=[c], alpha=0.88, edgecolors="k", linewidths=0.5, zorder=3)

# Axes
ax.set_xticks(range(len(tissues_ordered)))
ax.set_xticklabels(tissues_ordered, fontsize=11)
ax.set_yticks(range(len(genes_ordered)))
ax.set_yticklabels(genes_ordered, fontsize=11, fontweight="bold")
for label in ax.get_yticklabels():
    gene = label.get_text()
    label.set_color(pathway_colors[pathway_map[gene]])

ax.set_xlim(-0.6, 2.6)
ax.set_ylim(-0.7, len(genes_ordered) - 0.3)
ax.grid(True, alpha=0.25, linestyle="--")
ax.set_xlabel("Tissue", fontsize=12)
ax.set_ylabel("Gene", fontsize=12)
ax.set_title("GTEx v8 eQTLs: DDR & T-cell Genes\n(bubble size = −log₁₀ q-value; color = NES)", fontsize=12)

# ── Colorbar (NES) ────────────────────────────────────────────────────────────
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.55, pad=0.02, aspect=15)
cbar.set_label("NES (effect size)", fontsize=9)

# ── Pathway legend (outside right) ───────────────────────────────────────────
pathway_handles = [
    mpatches.Patch(facecolor=c, edgecolor="k", linewidth=0.5, label=p)
    for p, c in pathway_colors.items()
]
leg1 = fig.legend(handles=pathway_handles, title="Pathway",
                  loc="upper right", bbox_to_anchor=(1.0, 0.95),
                  fontsize=9, title_fontsize=9, framealpha=0.9,
                  borderaxespad=0)

# ── Size legend (outside right, below pathway) ───────────────────────────────
# Representative q-values and their -log10 values
size_examples = [("q=0.05", -np.log10(0.05)),
                 ("q=1e-5", 5),
                 ("q=1e-20", 20),
                 ("q=1e-60", 60)]
size_handles = [
    plt.scatter([], [], s=scale_size(v), c="gray", alpha=0.7,
                edgecolors="k", linewidths=0.5, label=lbl)
    for lbl, v in size_examples
]
leg2 = fig.legend(handles=size_handles, title="q-value",
                  loc="lower right", bbox_to_anchor=(1.0, 0.05),
                  fontsize=9, title_fontsize=9, framealpha=0.9,
                  borderaxespad=0, scatterpoints=1)

fig.add_artist(leg1)

plt.savefig("/mnt/results/gtex_v8_eqtl_bubble.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/gtex_v8_eqtl_bubble.svg", bbox_inches="tight")
plt.close()
print("Saved: gtex_v8_eqtl_bubble.png / .svg")


Saved: gtex_v8_eqtl_bubble.png / .svg


Query STRING API for PPI network of DDR + T-cell exhaustion genes

In [171]:

import requests
import pandas as pd
import json

# Source: https://string-db.org/api
STRING_API = "https://string-db.org/api"

# Gene set: curated DDR genes with evidence + T-cell exhaustion markers
ddr_genes = [
    # HR pathway
    "BRCA1","BRCA2","RAD51","RAD51C","RAD51D","PALB2","BARD1","RPA1","RPA2","RPA3",
    "RAD52","RAD54L","RBBP8","MRE11","RAD50","NBN",
    # NHEJ
    "XRCC6","XRCC5","PRKDC","LIG4","XRCC4","NHEJ1","DNTT",
    # MMR
    "MSH2","MSH6","MLH1","PMS2","MSH3",
    # BER
    "OGG1","APEX1","PARP1","PARP2","XRCC1","LIG3","POLB",
    # NER
    "XPC","ERCC2","ERCC3","ERCC4","ERCC5","ERCC6","DDB1","DDB2",
    # ATM/ATR signaling
    "ATM","ATR","CHEK1","CHEK2","TP53","MDM2","H2AX","BRIP1",
    # T-cell exhaustion
    "PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","TOX","TOX2","NR4A1",
    "ENTPD1","BATF","IRF4","PRDM1","TBX21","EOMES",
]

print(f"Query gene set: {len(ddr_genes)} genes")

# Get STRING network (interactions with score >= 400)
params = {
    "identifiers": "%0d".join(ddr_genes),
    "species": 9606,
    "caller_identity": "biomni_ad_analysis",
    "network_type": "functional",
    "required_score": 400,
    "output_format": "json",
}

resp = requests.post(f"{STRING_API}/json/network", data=params, timeout=60)
print(f"Network query status: {resp.status_code}")

if resp.status_code == 200:
    interactions = resp.json()
    print(f"Total interactions returned: {len(interactions)}")
    if interactions:
        df_net = pd.DataFrame(interactions)
        print(f"Columns: {df_net.columns.tolist()}")
        print(df_net.head(3))


Query gene set: 65 genes
Network query status: 200
Total interactions returned: 1176
Columns: ['stringId_A', 'stringId_B', 'preferredName_A', 'preferredName_B', 'ncbiTaxonId', 'score', 'nscore', 'fscore', 'pscore', 'ascore', 'escore', 'dscore', 'tscore']
             stringId_A            stringId_B  ... dscore tscore
0  9606.ENSP00000177694  9606.ENSP00000440864  ...    0.0  0.414
1  9606.ENSP00000177694  9606.ENSP00000344724  ...    0.0  0.417
2  9606.ENSP00000177694  9606.ENSP00000354842  ...    0.0  0.410

[3 rows x 13 columns]


Build PPI network, compute hub genes, run Reactome/KEGG ORA enrichment

In [173]:

import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import warnings
warnings.filterwarnings("ignore")

# ── Load STRING interactions ──────────────────────────────────────────────────
STRING_API = "https://string-db.org/api"

ddr_genes = [
    "BRCA1","BRCA2","RAD51","RAD51C","RAD51D","PALB2","BARD1","RPA1","RPA2","RPA3",
    "RAD52","RAD54L","RBBP8","MRE11","RAD50","NBN",
    "XRCC6","XRCC5","PRKDC","LIG4","XRCC4","NHEJ1","DNTT",
    "MSH2","MSH6","MLH1","PMS2","MSH3",
    "OGG1","APEX1","PARP1","PARP2","XRCC1","LIG3","POLB",
    "XPC","ERCC2","ERCC3","ERCC4","ERCC5","ERCC6","DDB1","DDB2",
    "ATM","ATR","CHEK1","CHEK2","TP53","MDM2","H2AX","BRIP1",
    "PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","TOX","TOX2","NR4A1",
    "ENTPD1","BATF","IRF4","PRDM1","TBX21","EOMES",
]

params = {
    "identifiers": "%0d".join(ddr_genes),
    "species": 9606,
    "caller_identity": "biomni_ad_analysis",
    "network_type": "functional",
    "required_score": 700,   # high-confidence only
    "output_format": "json",
}
resp = requests.post(f"{STRING_API}/json/network", data=params, timeout=60)
interactions = resp.json()
df_net = pd.DataFrame(interactions)
print(f"High-confidence interactions (score>=700): {len(df_net)}")

# ── Build NetworkX graph ──────────────────────────────────────────────────────
G = nx.Graph()
for _, row in df_net.iterrows():
    G.add_edge(row["preferredName_A"], row["preferredName_B"], weight=row["score"])

# Only keep nodes that are in our gene list
G = G.subgraph([n for n in G.nodes() if n in ddr_genes]).copy()
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ── Compute node metrics ──────────────────────────────────────────────────────
degree     = dict(G.degree())
betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)
pagerank   = nx.pagerank(G, weight="weight")

# Annotate nodes with evidence layers
gwas_genes   = {"DNTT", "RPA3"}
brain_de     = {"RPA3", "BRCA1", "ATM", "PARP1", "MSH2", "ERCC2"}  # from Step 2
tcell_de     = {"RPA3", "PDCD1", "HAVCR2", "LAG3", "TIGIT"}         # from Step 3
eqtl_genes   = {"RAD51C", "RPA3", "MSH2", "ATM", "XRCC6", "PDCD1", "HAVCR2", "CHEK1"}  # Step 4

pathway_map = {
    "BRCA1":"HR","BRCA2":"HR","RAD51":"HR","RAD51C":"HR","RAD51D":"HR",
    "PALB2":"HR","BARD1":"HR","RPA1":"HR","RPA2":"HR","RPA3":"HR",
    "RAD52":"HR","RAD54L":"HR","RBBP8":"HR","MRE11":"HR","RAD50":"HR","NBN":"HR","BRIP1":"HR",
    "XRCC6":"NHEJ","XRCC5":"NHEJ","PRKDC":"NHEJ","LIG4":"NHEJ","XRCC4":"NHEJ","NHEJ1":"NHEJ","DNTT":"NHEJ",
    "MSH2":"MMR","MSH6":"MMR","MLH1":"MMR","PMS2":"MMR","MSH3":"MMR",
    "OGG1":"BER","APEX1":"BER","PARP1":"BER","PARP2":"BER","XRCC1":"BER","LIG3":"BER","POLB":"BER",
    "XPC":"NER","ERCC2":"NER","ERCC3":"NER","ERCC4":"NER","ERCC5":"NER","ERCC6":"NER","DDB1":"NER","DDB2":"NER",
    "ATM":"ATM/ATR","ATR":"ATM/ATR","CHEK1":"ATM/ATR","CHEK2":"ATM/ATR","TP53":"ATM/ATR","MDM2":"ATM/ATR","H2AX":"ATM/ATR",
    "PDCD1":"T-cell","HAVCR2":"T-cell","LAG3":"T-cell","TIGIT":"T-cell","CTLA4":"T-cell",
    "TOX":"T-cell","TOX2":"T-cell","NR4A1":"T-cell","ENTPD1":"T-cell",
    "BATF":"T-cell","IRF4":"T-cell","PRDM1":"T-cell","TBX21":"T-cell","EOMES":"T-cell",
}

# Build summary table
node_df = pd.DataFrame({
    "gene": list(G.nodes()),
    "degree": [degree[n] for n in G.nodes()],
    "betweenness": [betweenness.get(n, 0) for n in G.nodes()],
    "pagerank": [pagerank.get(n, 0) for n in G.nodes()],
    "pathway": [pathway_map.get(n, "Other") for n in G.nodes()],
    "gwas": [n in gwas_genes for n in G.nodes()],
    "brain_de": [n in brain_de for n in G.nodes()],
    "tcell_de": [n in tcell_de for n in G.nodes()],
    "eqtl": [n in eqtl_genes for n in G.nodes()],
}).sort_values("degree", ascending=False)

node_df["evidence_count"] = (node_df["gwas"].astype(int) + node_df["brain_de"].astype(int) +
                              node_df["tcell_de"].astype(int) + node_df["eqtl"].astype(int))

print("\n=== Top 20 Hub Genes by Degree ===")
print(node_df[["gene","pathway","degree","betweenness","evidence_count"]].head(20).to_string(index=False))

node_df.to_csv("/mnt/results/string_ppi_hub_genes.csv", index=False)
print("\nSaved: string_ppi_hub_genes.csv")


High-confidence interactions (score>=700): 695
Network: 63 nodes, 695 edges

=== Top 20 Hub Genes by Degree ===
 gene pathway  degree  betweenness  evidence_count
XRCC6    NHEJ      43     0.030672               1
BRCA1      HR      42     0.008461               1
  ATM ATM/ATR      42     0.004231               2
XRCC5    NHEJ      40     0.019038               0
BRCA2      HR      40     0.012163               0
 MSH6     MMR      38     0.012692               0
 MSH2     MMR      37     0.005288               2
RAD51      HR      37     0.000529               0
ERCC4     NER      36     0.014278               0
 RPA1      HR      35     0.021682               0
MRE11      HR      34     0.012163               0
PARP1     BER      33     0.005817               1
 TP53 ATM/ATR      33     0.334215               0
RAD52      HR      32     0.007932               0
XRCC1     BER      32     0.004231               0
PRKDC    NHEJ      32     0.000529               0
PALB2      HR      32

Run STRING pathway enrichment ORA and visualize PPI network with evidence layers

In [175]:

import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings("ignore")

# ── Rebuild network from previous step ───────────────────────────────────────
STRING_API = "https://string-db.org/api"

ddr_genes = [
    "BRCA1","BRCA2","RAD51","RAD51C","RAD51D","PALB2","BARD1","RPA1","RPA2","RPA3",
    "RAD52","RAD54L","RBBP8","MRE11","RAD50","NBN",
    "XRCC6","XRCC5","PRKDC","LIG4","XRCC4","NHEJ1","DNTT",
    "MSH2","MSH6","MLH1","PMS2","MSH3",
    "OGG1","APEX1","PARP1","PARP2","XRCC1","LIG3","POLB",
    "XPC","ERCC2","ERCC3","ERCC4","ERCC5","ERCC6","DDB1","DDB2",
    "ATM","ATR","CHEK1","CHEK2","TP53","MDM2","H2AX","BRIP1",
    "PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","TOX","TOX2","NR4A1",
    "ENTPD1","BATF","IRF4","PRDM1","TBX21","EOMES",
]

params = {
    "identifiers": "%0d".join(ddr_genes),
    "species": 9606,
    "caller_identity": "biomni_ad_analysis",
    "network_type": "functional",
    "required_score": 700,
    "output_format": "json",
}
resp = requests.post(f"{STRING_API}/json/network", data=params, timeout=60)
df_net = pd.DataFrame(resp.json())

G = nx.Graph()
for _, row in df_net.iterrows():
    G.add_edge(row["preferredName_A"], row["preferredName_B"], weight=row["score"])
G = G.subgraph([n for n in G.nodes() if n in ddr_genes]).copy()

# ── STRING pathway enrichment ─────────────────────────────────────────────────
enrich_params = {
    "identifiers": "%0d".join(ddr_genes),
    "species": 9606,
    "caller_identity": "biomni_ad_analysis",
    "output_format": "json",
}
enrich_resp = requests.post(f"{STRING_API}/json/enrichment", data=enrich_params, timeout=60)
enrich_data = enrich_resp.json()
print(f"Enrichment terms returned: {len(enrich_data)}")

df_enrich = pd.DataFrame(enrich_data)
print(f"Columns: {df_enrich.columns.tolist()}")

# Filter to Reactome and KEGG, FDR < 0.05
df_enrich_filt = df_enrich[
    (df_enrich["category"].isin(["RCTM", "KEGG"])) &
    (df_enrich["fdr"].astype(float) < 0.05)
].copy()
df_enrich_filt["fdr"] = df_enrich_filt["fdr"].astype(float)
df_enrich_filt["-log10fdr"] = -np.log10(df_enrich_filt["fdr"])
df_enrich_filt = df_enrich_filt.sort_values("-log10fdr", ascending=False)

print(f"\nSignificant Reactome/KEGG terms (FDR<0.05): {len(df_enrich_filt)}")
print(df_enrich_filt[["category","term","description","number_of_genes","-log10fdr"]].head(20).to_string(index=False))

df_enrich_filt.to_csv("/mnt/results/string_pathway_enrichment.csv", index=False)
print("\nSaved: string_pathway_enrichment.csv")


Enrichment terms returned: 1730
Columns: ['category', 'term', 'number_of_genes', 'number_of_genes_in_background', 'ncbiTaxonId', 'inputGenes', 'preferredNames', 'p_value', 'fdr', 'description']

Significant Reactome/KEGG terms (FDR<0.05): 132
category        term                                                                                                          description  number_of_genes  -log10fdr
    RCTM   HSA-73894                                                                                                           DNA Repair               48  66.508638
    RCTM HSA-5693532                                                                                       DNA Double-Strand Break Repair               33  47.000000
    RCTM HSA-9675135                                                                                               Diseases of DNA repair               23  37.438899
    RCTM HSA-5693538                                                                         

Visualize PPI network with evidence layers + top pathway enrichment bar chart

In [177]:

import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings("ignore")

# ── Rebuild network ───────────────────────────────────────────────────────────
STRING_API = "https://string-db.org/api"
ddr_genes = [
    "BRCA1","BRCA2","RAD51","RAD51C","RAD51D","PALB2","BARD1","RPA1","RPA2","RPA3",
    "RAD52","RAD54L","RBBP8","MRE11","RAD50","NBN",
    "XRCC6","XRCC5","PRKDC","LIG4","XRCC4","NHEJ1","DNTT",
    "MSH2","MSH6","MLH1","PMS2","MSH3",
    "OGG1","APEX1","PARP1","PARP2","XRCC1","LIG3","POLB",
    "XPC","ERCC2","ERCC3","ERCC4","ERCC5","ERCC6","DDB1","DDB2",
    "ATM","ATR","CHEK1","CHEK2","TP53","MDM2","H2AX","BRIP1",
    "PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","TOX","TOX2","NR4A1",
    "ENTPD1","BATF","IRF4","PRDM1","TBX21","EOMES",
]
params = {"identifiers": "%0d".join(ddr_genes), "species": 9606,
          "caller_identity": "biomni_ad_analysis", "network_type": "functional",
          "required_score": 700, "output_format": "json"}
df_net = pd.DataFrame(requests.post(f"{STRING_API}/json/network", data=params, timeout=60).json())

G = nx.Graph()
for _, row in df_net.iterrows():
    G.add_edge(row["preferredName_A"], row["preferredName_B"], weight=row["score"])
G = G.subgraph([n for n in G.nodes() if n in ddr_genes]).copy()

# Evidence annotations
gwas_genes  = {"DNTT", "RPA3"}
brain_de    = {"RPA3", "BRCA1", "ATM", "PARP1", "MSH2", "ERCC2"}
tcell_de    = {"RPA3", "PDCD1", "HAVCR2", "LAG3", "TIGIT"}
eqtl_genes  = {"RAD51C", "RPA3", "MSH2", "ATM", "XRCC6", "PDCD1", "HAVCR2", "CHEK1"}

pathway_colors = {
    "HR": "#0279EE", "NHEJ": "#c8a800", "MMR": "#75A025",
    "BER": "#FF9400", "NER": "#9B59B6", "ATM/ATR": "#E74C3C", "T-cell": "#c040a0"
}
pathway_map = {
    "BRCA1":"HR","BRCA2":"HR","RAD51":"HR","RAD51C":"HR","RAD51D":"HR",
    "PALB2":"HR","BARD1":"HR","RPA1":"HR","RPA2":"HR","RPA3":"HR",
    "RAD52":"HR","RAD54L":"HR","RBBP8":"HR","MRE11":"HR","RAD50":"HR","NBN":"HR","BRIP1":"HR",
    "XRCC6":"NHEJ","XRCC5":"NHEJ","PRKDC":"NHEJ","LIG4":"NHEJ","XRCC4":"NHEJ","NHEJ1":"NHEJ","DNTT":"NHEJ",
    "MSH2":"MMR","MSH6":"MMR","MLH1":"MMR","PMS2":"MMR","MSH3":"MMR",
    "OGG1":"BER","APEX1":"BER","PARP1":"BER","PARP2":"BER","XRCC1":"BER","LIG3":"BER","POLB":"BER",
    "XPC":"NER","ERCC2":"NER","ERCC3":"NER","ERCC4":"NER","ERCC5":"NER","ERCC6":"NER","DDB1":"NER","DDB2":"NER",
    "ATM":"ATM/ATR","ATR":"ATM/ATR","CHEK1":"ATM/ATR","CHEK2":"ATM/ATR",
    "TP53":"ATM/ATR","MDM2":"ATM/ATR","H2AX":"ATM/ATR",
    "PDCD1":"T-cell","HAVCR2":"T-cell","LAG3":"T-cell","TIGIT":"T-cell","CTLA4":"T-cell",
    "TOX":"T-cell","TOX2":"T-cell","NR4A1":"T-cell","ENTPD1":"T-cell",
    "BATF":"T-cell","IRF4":"T-cell","PRDM1":"T-cell","TBX21":"T-cell","EOMES":"T-cell",
}

degree = dict(G.degree())

# ── Layout ────────────────────────────────────────────────────────────────────
np.random.seed(42)
pos = nx.spring_layout(G, k=1.8, iterations=100, seed=42, weight="weight")

# ── Figure: 2 panels ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 8))
gs = GridSpec(1, 2, width_ratios=[1.4, 1], figure=fig, wspace=0.08)
ax_net = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1])

# ── Panel A: PPI Network ──────────────────────────────────────────────────────
# Edges (color by weight)
edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
edge_alpha = [(w - 700) / (1000 - 700) * 0.5 + 0.1 for w in edge_weights]

for (u, v), alpha in zip(G.edges(), edge_alpha):
    x0, y0 = pos[u]; x1, y1 = pos[v]
    ax_net.plot([x0, x1], [y0, y1], color="#AAAAAA", alpha=alpha, linewidth=0.5, zorder=1)

# Nodes
for node in G.nodes():
    x, y = pos[node]
    pway = pathway_map.get(node, "HR")
    color = pathway_colors[pway]
    size = 80 + degree[node] * 8

    # Evidence border: thick gold if multi-evidence
    ev_count = sum([node in gwas_genes, node in brain_de, node in tcell_de, node in eqtl_genes])
    edge_color = "#FFD700" if ev_count >= 2 else ("#FF4444" if ev_count == 1 else "#555555")
    lw = 3.0 if ev_count >= 2 else (1.5 if ev_count == 1 else 0.5)

    ax_net.scatter(x, y, s=size, c=color, edgecolors=edge_color, linewidths=lw,
                   alpha=0.9, zorder=3)

# Labels for key genes only (high degree or multi-evidence)
key_labels = {n for n in G.nodes()
              if degree[n] >= 30 or
              sum([n in gwas_genes, n in brain_de, n in tcell_de, n in eqtl_genes]) >= 2}
for node in key_labels:
    x, y = pos[node]
    ax_net.annotate(node, (x, y), fontsize=6.5, fontweight="bold",
                    ha="center", va="bottom", xytext=(0, 5),
                    textcoords="offset points", zorder=5)

ax_net.set_title("A. STRING PPI Network (score ≥ 0.7)\nDDR & T-cell Exhaustion Genes", fontsize=11)
ax_net.axis("off")

# Pathway legend
pathway_handles = [mpatches.Patch(facecolor=c, edgecolor="k", linewidth=0.5, label=p)
                   for p, c in pathway_colors.items()]
ax_net.legend(handles=pathway_handles, title="Pathway", loc="lower left",
              fontsize=8, title_fontsize=8, framealpha=0.85)

# Evidence border legend
ev_handles = [
    mpatches.Patch(facecolor="white", edgecolor="#FFD700", linewidth=3, label="≥2 evidence layers"),
    mpatches.Patch(facecolor="white", edgecolor="#FF4444", linewidth=1.5, label="1 evidence layer"),
    mpatches.Patch(facecolor="white", edgecolor="#555555", linewidth=0.5, label="Network only"),
]
ax_net.legend(handles=ev_handles, title="Evidence", loc="lower right",
              fontsize=8, title_fontsize=8, framealpha=0.85)
# Re-add pathway legend
leg1 = ax_net.legend(handles=pathway_handles, title="Pathway", loc="lower left",
                     fontsize=8, title_fontsize=8, framealpha=0.85)
ax_net.add_artist(leg1)
ax_net.legend(handles=ev_handles, title="Evidence", loc="lower right",
              fontsize=8, title_fontsize=8, framealpha=0.85)

# ── Panel B: Top Pathway Enrichment ──────────────────────────────────────────
df_enrich = pd.read_csv("/mnt/results/string_pathway_enrichment.csv")
df_enrich["-log10fdr"] = -np.log10(df_enrich["fdr"].astype(float))

# Select top 15 non-redundant terms (deduplicate by description similarity)
top_terms = df_enrich.head(15).copy()
top_terms["short_desc"] = top_terms["description"].str[:55]
top_terms = top_terms.sort_values("-log10fdr", ascending=True)

bar_colors = ["#0279EE" if "Homolog" in d or "HR" in d or "BRCA" in d or "RAD51" in d
              else "#E74C3C" if "TP53" in d or "checkpoint" in d or "damage" in d
              else "#75A025" if "Mismatch" in d or "MMR" in d
              else "#9B59B6" if "Nucleotide" in d
              else "#FF9400"
              for d in top_terms["description"]]

bars = ax_bar.barh(range(len(top_terms)), top_terms["-log10fdr"],
                   color=bar_colors, edgecolor="k", linewidth=0.4, alpha=0.85)
ax_bar.set_yticks(range(len(top_terms)))
ax_bar.set_yticklabels(top_terms["short_desc"], fontsize=8)
ax_bar.set_xlabel("−log₁₀(FDR)", fontsize=10)
ax_bar.set_title("B. Top Reactome/KEGG Enrichment\n(STRING ORA, FDR < 0.05)", fontsize=11)
ax_bar.axvline(x=-np.log10(0.05), color="red", linestyle="--", linewidth=1, alpha=0.7, label="FDR=0.05")
ax_bar.legend(fontsize=8)
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)

# Gene count annotations
for i, (_, row) in enumerate(top_terms.iterrows()):
    ax_bar.text(row["-log10fdr"] + 0.3, i, f"n={int(row['number_of_genes'])}",
                va="center", fontsize=7, color="#333333")

plt.suptitle("STRING PPI Network & Pathway Enrichment: DDR–T-cell–AD Axis",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/mnt/results/string_network_enrichment.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/string_network_enrichment.svg", bbox_inches="tight")
plt.close()
print("Saved: string_network_enrichment.png / .svg")


Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 81, in <module>
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/axes/_axes.py", line 1777, in plot
    lines = [*self._get_lines(self, *args, data=data, **kwargs)]
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/axes/_base.py", line 297, in __call__
    yield from self._plot_args(
               ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/axes/_base.py", line 546, in _plot_args
    return [l[0] for l in result]
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/axes/_base.py", line 546, in <listcomp>
    return [l[0] for l in result]
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/matplotlib/axes/_base.py", line 539, in <genexpr>
    result = (make_artist(axes, x[:, j % ncx], y[:, j % ncy], kw,
  

Fix edge alpha clamping and regenerate network + enrichment figure

In [179]:

import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings("ignore")

# ── Rebuild network ───────────────────────────────────────────────────────────
STRING_API = "https://string-db.org/api"
ddr_genes = [
    "BRCA1","BRCA2","RAD51","RAD51C","RAD51D","PALB2","BARD1","RPA1","RPA2","RPA3",
    "RAD52","RAD54L","RBBP8","MRE11","RAD50","NBN",
    "XRCC6","XRCC5","PRKDC","LIG4","XRCC4","NHEJ1","DNTT",
    "MSH2","MSH6","MLH1","PMS2","MSH3",
    "OGG1","APEX1","PARP1","PARP2","XRCC1","LIG3","POLB",
    "XPC","ERCC2","ERCC3","ERCC4","ERCC5","ERCC6","DDB1","DDB2",
    "ATM","ATR","CHEK1","CHEK2","TP53","MDM2","H2AX","BRIP1",
    "PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","TOX","TOX2","NR4A1",
    "ENTPD1","BATF","IRF4","PRDM1","TBX21","EOMES",
]
params = {"identifiers": "%0d".join(ddr_genes), "species": 9606,
          "caller_identity": "biomni_ad_analysis", "network_type": "functional",
          "required_score": 700, "output_format": "json"}
df_net = pd.DataFrame(requests.post(f"{STRING_API}/json/network", data=params, timeout=60).json())

G = nx.Graph()
for _, row in df_net.iterrows():
    G.add_edge(row["preferredName_A"], row["preferredName_B"], weight=float(row["score"]))
G = G.subgraph([n for n in G.nodes() if n in ddr_genes]).copy()

# Evidence annotations
gwas_genes = {"DNTT", "RPA3"}
brain_de   = {"RPA3", "BRCA1", "ATM", "PARP1", "MSH2", "ERCC2"}
tcell_de   = {"RPA3", "PDCD1", "HAVCR2", "LAG3", "TIGIT"}
eqtl_genes = {"RAD51C", "RPA3", "MSH2", "ATM", "XRCC6", "PDCD1", "HAVCR2", "CHEK1"}

pathway_colors = {
    "HR": "#0279EE", "NHEJ": "#c8a800", "MMR": "#75A025",
    "BER": "#FF9400", "NER": "#9B59B6", "ATM/ATR": "#E74C3C", "T-cell": "#c040a0"
}
pathway_map = {
    "BRCA1":"HR","BRCA2":"HR","RAD51":"HR","RAD51C":"HR","RAD51D":"HR",
    "PALB2":"HR","BARD1":"HR","RPA1":"HR","RPA2":"HR","RPA3":"HR",
    "RAD52":"HR","RAD54L":"HR","RBBP8":"HR","MRE11":"HR","RAD50":"HR","NBN":"HR","BRIP1":"HR",
    "XRCC6":"NHEJ","XRCC5":"NHEJ","PRKDC":"NHEJ","LIG4":"NHEJ","XRCC4":"NHEJ","NHEJ1":"NHEJ","DNTT":"NHEJ",
    "MSH2":"MMR","MSH6":"MMR","MLH1":"MMR","PMS2":"MMR","MSH3":"MMR",
    "OGG1":"BER","APEX1":"BER","PARP1":"BER","PARP2":"BER","XRCC1":"BER","LIG3":"BER","POLB":"BER",
    "XPC":"NER","ERCC2":"NER","ERCC3":"NER","ERCC4":"NER","ERCC5":"NER","ERCC6":"NER","DDB1":"NER","DDB2":"NER",
    "ATM":"ATM/ATR","ATR":"ATM/ATR","CHEK1":"ATM/ATR","CHEK2":"ATM/ATR",
    "TP53":"ATM/ATR","MDM2":"ATM/ATR","H2AX":"ATM/ATR",
    "PDCD1":"T-cell","HAVCR2":"T-cell","LAG3":"T-cell","TIGIT":"T-cell","CTLA4":"T-cell",
    "TOX":"T-cell","TOX2":"T-cell","NR4A1":"T-cell","ENTPD1":"T-cell",
    "BATF":"T-cell","IRF4":"T-cell","PRDM1":"T-cell","TBX21":"T-cell","EOMES":"T-cell",
}
degree = dict(G.degree())

np.random.seed(42)
pos = nx.spring_layout(G, k=1.8, iterations=100, seed=42, weight="weight")

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 8))
gs = GridSpec(1, 2, width_ratios=[1.4, 1], figure=fig, wspace=0.08)
ax_net = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1])

# ── Panel A: Network ──────────────────────────────────────────────────────────
# Draw edges with clamped alpha
all_weights = [G[u][v]["weight"] for u, v in G.edges()]
w_min, w_max = min(all_weights), max(all_weights)
for (u, v) in G.edges():
    w = G[u][v]["weight"]
    alpha = float(np.clip((w - w_min) / (w_max - w_min + 1e-9) * 0.5 + 0.05, 0.05, 0.6))
    x0, y0 = pos[u]; x1, y1 = pos[v]
    ax_net.plot([x0, x1], [y0, y1], color="#AAAAAA", alpha=alpha, linewidth=0.5, zorder=1)

# Draw nodes
for node in G.nodes():
    x, y = pos[node]
    pway = pathway_map.get(node, "HR")
    color = pathway_colors[pway]
    size = 80 + degree[node] * 8
    ev_count = sum([node in gwas_genes, node in brain_de, node in tcell_de, node in eqtl_genes])
    edge_color = "#FFD700" if ev_count >= 2 else ("#FF4444" if ev_count == 1 else "#555555")
    lw = 3.0 if ev_count >= 2 else (1.5 if ev_count == 1 else 0.5)
    ax_net.scatter(x, y, s=size, c=color, edgecolors=edge_color, linewidths=lw,
                   alpha=0.9, zorder=3)

# Labels for key genes
key_labels = {n for n in G.nodes()
              if degree[n] >= 30 or
              sum([n in gwas_genes, n in brain_de, n in tcell_de, n in eqtl_genes]) >= 2}
for node in key_labels:
    x, y = pos[node]
    ax_net.annotate(node, (x, y), fontsize=6.5, fontweight="bold",
                    ha="center", va="bottom", xytext=(0, 5),
                    textcoords="offset points", zorder=5)

ax_net.set_title("A. STRING PPI Network (score ≥ 0.7)\nDDR & T-cell Exhaustion Genes", fontsize=11)
ax_net.axis("off")

# Legends
pathway_handles = [mpatches.Patch(facecolor=c, edgecolor="k", linewidth=0.5, label=p)
                   for p, c in pathway_colors.items()]
ev_handles = [
    mpatches.Patch(facecolor="white", edgecolor="#FFD700", linewidth=3, label="≥2 evidence layers"),
    mpatches.Patch(facecolor="white", edgecolor="#FF4444", linewidth=1.5, label="1 evidence layer"),
    mpatches.Patch(facecolor="white", edgecolor="#555555", linewidth=0.5, label="Network only"),
]
leg1 = ax_net.legend(handles=pathway_handles, title="Pathway", loc="lower left",
                     fontsize=8, title_fontsize=8, framealpha=0.85)
ax_net.add_artist(leg1)
ax_net.legend(handles=ev_handles, title="Evidence", loc="lower right",
              fontsize=8, title_fontsize=8, framealpha=0.85)

# ── Panel B: Enrichment bar chart ─────────────────────────────────────────────
df_enrich = pd.read_csv("/mnt/results/string_pathway_enrichment.csv")
df_enrich["-log10fdr"] = -np.log10(df_enrich["fdr"].astype(float))
top_terms = df_enrich.head(15).copy()
top_terms["short_desc"] = top_terms["description"].str[:52]
top_terms = top_terms.sort_values("-log10fdr", ascending=True)

bar_colors = []
for d in top_terms["description"]:
    if any(k in d for k in ["Homolog","BRCA","RAD51","Strand","D-loop","SSA","SDSA"]):
        bar_colors.append("#0279EE")
    elif any(k in d for k in ["TP53","checkpoint","damage","G2/M"]):
        bar_colors.append("#E74C3C")
    elif any(k in d for k in ["Mismatch","MMR"]):
        bar_colors.append("#75A025")
    elif "Nucleotide" in d:
        bar_colors.append("#9B59B6")
    else:
        bar_colors.append("#FF9400")

ax_bar.barh(range(len(top_terms)), top_terms["-log10fdr"],
            color=bar_colors, edgecolor="k", linewidth=0.4, alpha=0.85)
ax_bar.set_yticks(range(len(top_terms)))
ax_bar.set_yticklabels(top_terms["short_desc"], fontsize=8)
ax_bar.set_xlabel("−log₁₀(FDR)", fontsize=10)
ax_bar.set_title("B. Top Reactome/KEGG Enrichment\n(STRING ORA, FDR < 0.05)", fontsize=11)
ax_bar.axvline(x=-np.log10(0.05), color="red", linestyle="--", linewidth=1, alpha=0.7, label="FDR=0.05")
ax_bar.legend(fontsize=8)
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
for i, (_, row) in enumerate(top_terms.iterrows()):
    ax_bar.text(row["-log10fdr"] + 0.5, i, f"n={int(row['number_of_genes'])}",
                va="center", fontsize=7, color="#333333")

plt.suptitle("STRING PPI Network & Pathway Enrichment: DDR–T-cell–AD Axis",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("/mnt/results/string_network_enrichment.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/string_network_enrichment.svg", bbox_inches="tight")
plt.close()
print("Saved: string_network_enrichment.png / .svg")


Saved: string_network_enrichment.png / .svg


Split into two separate figures: network and enrichment bar chart

In [181]:

import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

# ── Rebuild network ───────────────────────────────────────────────────────────
STRING_API = "https://string-db.org/api"
ddr_genes = [
    "BRCA1","BRCA2","RAD51","RAD51C","RAD51D","PALB2","BARD1","RPA1","RPA2","RPA3",
    "RAD52","RAD54L","RBBP8","MRE11","RAD50","NBN",
    "XRCC6","XRCC5","PRKDC","LIG4","XRCC4","NHEJ1","DNTT",
    "MSH2","MSH6","MLH1","PMS2","MSH3",
    "OGG1","APEX1","PARP1","PARP2","XRCC1","LIG3","POLB",
    "XPC","ERCC2","ERCC3","ERCC4","ERCC5","ERCC6","DDB1","DDB2",
    "ATM","ATR","CHEK1","CHEK2","TP53","MDM2","H2AX","BRIP1",
    "PDCD1","HAVCR2","LAG3","TIGIT","CTLA4","TOX","TOX2","NR4A1",
    "ENTPD1","BATF","IRF4","PRDM1","TBX21","EOMES",
]
params = {"identifiers": "%0d".join(ddr_genes), "species": 9606,
          "caller_identity": "biomni_ad_analysis", "network_type": "functional",
          "required_score": 700, "output_format": "json"}
df_net = pd.DataFrame(requests.post(f"{STRING_API}/json/network", data=params, timeout=60).json())

G = nx.Graph()
for _, row in df_net.iterrows():
    G.add_edge(row["preferredName_A"], row["preferredName_B"], weight=float(row["score"]))
G = G.subgraph([n for n in G.nodes() if n in ddr_genes]).copy()

gwas_genes = {"DNTT", "RPA3"}
brain_de   = {"RPA3", "BRCA1", "ATM", "PARP1", "MSH2", "ERCC2"}
tcell_de   = {"RPA3", "PDCD1", "HAVCR2", "LAG3", "TIGIT"}
eqtl_genes = {"RAD51C", "RPA3", "MSH2", "ATM", "XRCC6", "PDCD1", "HAVCR2", "CHEK1"}

pathway_colors = {
    "HR": "#0279EE", "NHEJ": "#c8a800", "MMR": "#75A025",
    "BER": "#FF9400", "NER": "#9B59B6", "ATM/ATR": "#E74C3C", "T-cell": "#c040a0"
}
pathway_map = {
    "BRCA1":"HR","BRCA2":"HR","RAD51":"HR","RAD51C":"HR","RAD51D":"HR",
    "PALB2":"HR","BARD1":"HR","RPA1":"HR","RPA2":"HR","RPA3":"HR",
    "RAD52":"HR","RAD54L":"HR","RBBP8":"HR","MRE11":"HR","RAD50":"HR","NBN":"HR","BRIP1":"HR",
    "XRCC6":"NHEJ","XRCC5":"NHEJ","PRKDC":"NHEJ","LIG4":"NHEJ","XRCC4":"NHEJ","NHEJ1":"NHEJ","DNTT":"NHEJ",
    "MSH2":"MMR","MSH6":"MMR","MLH1":"MMR","PMS2":"MMR","MSH3":"MMR",
    "OGG1":"BER","APEX1":"BER","PARP1":"BER","PARP2":"BER","XRCC1":"BER","LIG3":"BER","POLB":"BER",
    "XPC":"NER","ERCC2":"NER","ERCC3":"NER","ERCC4":"NER","ERCC5":"NER","ERCC6":"NER","DDB1":"NER","DDB2":"NER",
    "ATM":"ATM/ATR","ATR":"ATM/ATR","CHEK1":"ATM/ATR","CHEK2":"ATM/ATR",
    "TP53":"ATM/ATR","MDM2":"ATM/ATR","H2AX":"ATM/ATR",
    "PDCD1":"T-cell","HAVCR2":"T-cell","LAG3":"T-cell","TIGIT":"T-cell","CTLA4":"T-cell",
    "TOX":"T-cell","TOX2":"T-cell","NR4A1":"T-cell","ENTPD1":"T-cell",
    "BATF":"T-cell","IRF4":"T-cell","PRDM1":"T-cell","TBX21":"T-cell","EOMES":"T-cell",
}
degree = dict(G.degree())
np.random.seed(42)
pos = nx.spring_layout(G, k=2.0, iterations=120, seed=42, weight="weight")

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1: PPI Network
# ════════════════════════════════════════════════════════════════════════════
fig1, ax = plt.subplots(figsize=(12, 10))

# Edges
all_w = [G[u][v]["weight"] for u, v in G.edges()]
w_min, w_max = min(all_w), max(all_w)
for (u, v) in G.edges():
    w = G[u][v]["weight"]
    alpha = float(np.clip((w - w_min) / (w_max - w_min + 1e-9) * 0.45 + 0.05, 0.05, 0.5))
    x0, y0 = pos[u]; x1, y1 = pos[v]
    ax.plot([x0, x1], [y0, y1], color="#BBBBBB", alpha=alpha, linewidth=0.5, zorder=1)

# Nodes
for node in G.nodes():
    x, y = pos[node]
    pway = pathway_map.get(node, "HR")
    color = pathway_colors[pway]
    size = 100 + degree[node] * 10
    ev_count = sum([node in gwas_genes, node in brain_de, node in tcell_de, node in eqtl_genes])
    edge_color = "#FFD700" if ev_count >= 2 else ("#FF4444" if ev_count == 1 else "#666666")
    lw = 3.5 if ev_count >= 2 else (1.5 if ev_count == 1 else 0.5)
    ax.scatter(x, y, s=size, c=color, edgecolors=edge_color, linewidths=lw, alpha=0.9, zorder=3)

# Labels: only label genes with evidence OR degree >= 32 (top hubs)
label_genes = {n for n in G.nodes()
               if sum([n in gwas_genes, n in brain_de, n in tcell_de, n in eqtl_genes]) >= 1
               or degree[n] >= 32}
for node in label_genes:
    x, y = pos[node]
    ev = sum([node in gwas_genes, node in brain_de, node in tcell_de, node in eqtl_genes])
    fw = "bold" if ev >= 1 else "normal"
    ax.annotate(node, (x, y), fontsize=7, fontweight=fw,
                ha="center", va="bottom", xytext=(0, 6),
                textcoords="offset points", zorder=5,
                bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.6, ec="none"))

ax.set_title("STRING PPI Network (score ≥ 0.7): DDR & T-cell Exhaustion Genes\n"
             "Node size = degree; border color = evidence layers", fontsize=12)
ax.axis("off")

# Pathway legend (upper left)
pathway_handles = [mpatches.Patch(facecolor=c, edgecolor="k", linewidth=0.5, label=p)
                   for p, c in pathway_colors.items()]
leg1 = ax.legend(handles=pathway_handles, title="Pathway", loc="upper left",
                 fontsize=9, title_fontsize=9, framealpha=0.9)
ax.add_artist(leg1)

# Evidence legend (lower left)
ev_handles = [
    mpatches.Patch(facecolor="white", edgecolor="#FFD700", linewidth=3.5, label="≥2 evidence layers"),
    mpatches.Patch(facecolor="white", edgecolor="#FF4444", linewidth=1.5, label="1 evidence layer"),
    mpatches.Patch(facecolor="white", edgecolor="#666666", linewidth=0.5, label="Network only"),
]
ax.legend(handles=ev_handles, title="Evidence (GWAS/DE/eQTL)", loc="lower left",
          fontsize=9, title_fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig("/mnt/results/string_ppi_network.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/string_ppi_network.svg", bbox_inches="tight")
plt.close()
print("Saved: string_ppi_network.png / .svg")

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Pathway Enrichment Bar Chart
# ════════════════════════════════════════════════════════════════════════════
df_enrich = pd.read_csv("/mnt/results/string_pathway_enrichment.csv")
df_enrich["-log10fdr"] = -np.log10(df_enrich["fdr"].astype(float))

# Select top 15, deduplicate by keeping most significant per broad category
top_terms = df_enrich.head(20).drop_duplicates(subset="description").head(15).copy()
# Shorten labels to 60 chars max
top_terms["label"] = top_terms["description"].apply(
    lambda x: x[:57] + "..." if len(x) > 60 else x)
top_terms = top_terms.sort_values("-log10fdr", ascending=True).reset_index(drop=True)

bar_colors = []
for d in top_terms["description"]:
    if any(k in d for k in ["Homolog","BRCA","RAD51","Strand","D-loop","SSA","SDSA","Presynaptic"]):
        bar_colors.append("#0279EE")
    elif any(k in d for k in ["TP53","checkpoint","damage","G2/M","Phosphorylation"]):
        bar_colors.append("#E74C3C")
    elif any(k in d for k in ["Mismatch","MMR"]):
        bar_colors.append("#75A025")
    elif "Nucleotide" in d:
        bar_colors.append("#9B59B6")
    elif "Diseases" in d:
        bar_colors.append("#888888")
    else:
        bar_colors.append("#FF9400")

fig2, ax2 = plt.subplots(figsize=(11, 7))
bars = ax2.barh(range(len(top_terms)), top_terms["-log10fdr"],
                color=bar_colors, edgecolor="k", linewidth=0.4, alpha=0.85)
ax2.set_yticks(range(len(top_terms)))
ax2.set_yticklabels(top_terms["label"], fontsize=9)
ax2.set_xlabel("−log₁₀(FDR)", fontsize=11)
ax2.set_title("Top Reactome/KEGG Pathway Enrichment\n"
              "STRING ORA on DDR & T-cell Gene Set (FDR < 0.05)", fontsize=12)
ax2.axvline(x=-np.log10(0.05), color="red", linestyle="--", linewidth=1,
            alpha=0.7, label="FDR = 0.05")

# Gene count labels — position safely inside bar
for i, (_, row) in enumerate(top_terms.iterrows()):
    xpos = min(row["-log10fdr"] - 1.5, row["-log10fdr"] * 0.85)
    ax2.text(max(xpos, 1), i, f"n={int(row['number_of_genes'])}",
             va="center", ha="left", fontsize=8, color="white", fontweight="bold")

# Category color legend
cat_handles = [
    mpatches.Patch(facecolor="#0279EE", label="Homologous Recombination"),
    mpatches.Patch(facecolor="#E74C3C", label="DNA Damage Signaling / TP53"),
    mpatches.Patch(facecolor="#75A025", label="Mismatch Repair"),
    mpatches.Patch(facecolor="#9B59B6", label="Nucleotide Excision Repair"),
    mpatches.Patch(facecolor="#888888", label="Disease of DNA Repair"),
]
ax2.legend(handles=cat_handles, title="Category", loc="lower right",
           fontsize=8, title_fontsize=9, framealpha=0.9)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
# Extend x-axis slightly for readability
ax2.set_xlim(0, top_terms["-log10fdr"].max() * 1.05)

plt.tight_layout()
plt.savefig("/mnt/results/string_pathway_enrichment_plot.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/string_pathway_enrichment_plot.svg", bbox_inches="tight")
plt.close()
print("Saved: string_pathway_enrichment_plot.png / .svg")


Saved: string_ppi_network.png / .svg
Saved: string_pathway_enrichment_plot.png / .svg


Fix enrichment bar chart: full labels, dedup, correct x-range, matching legend

In [183]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df_enrich = pd.read_csv("/mnt/results/string_pathway_enrichment.csv")
df_enrich["-log10fdr"] = -np.log10(df_enrich["fdr"].astype(float))

# ── Select 15 non-redundant terms ─────────────────────────────────────────────
# Keep top hit per broad category to avoid near-duplicates
top_raw = df_enrich.sort_values("-log10fdr", ascending=False).head(50)

# Manually curate 15 distinct, informative terms
keep_terms = [
    "DNA Repair",
    "DNA Double-Strand Break Repair",
    "Homology Directed Repair",
    "Homologous DNA Pairing and Strand Exchange",
    "HDR through Homologous Recombination (HRR) or Single Strand Annealing (SSA)",
    "Diseases of DNA repair",
    "Transcriptional Regulation by TP53",
    "G2/M DNA damage checkpoint",
    "Regulation of TP53 Activity through Phosphorylation",
    "Presynaptic phase of homologous DNA pairing and strand exchange",
    "Defective homologous recombination repair (HRR) due to BRCA2 loss of function",
    "Defective homologous recombination repair (HRR) due to BRCA1 loss of function",
    "Resolution of D-loop Structures through Synthesis-Dependent Strand Annealing (SDSA)",
    "HDR through Single Strand Annealing (SSA)",
    "Mismatch Repair",
]

df_plot = df_enrich[df_enrich["description"].isin(keep_terms)].copy()
# If any missing, fill from top
missing = [t for t in keep_terms if t not in df_plot["description"].values]
if missing:
    print(f"Missing terms (will use closest): {missing}")
    # Fall back to top terms not already included
    fallback = df_enrich[~df_enrich["description"].isin(df_plot["description"].values)].head(len(missing))
    df_plot = pd.concat([df_plot, fallback])

df_plot = df_plot.sort_values("-log10fdr", ascending=True).reset_index(drop=True)
print(f"Terms to plot: {len(df_plot)}")
print(df_plot[["description","-log10fdr","number_of_genes"]].to_string())

# ── Assign colors based on actual category ────────────────────────────────────
def assign_color(desc):
    if any(k in desc for k in ["Mismatch"]):
        return "#75A025", "MMR"
    elif any(k in desc for k in ["TP53","G2/M","Phosphorylation","Transcriptional Reg"]):
        return "#E74C3C", "DNA Damage Signaling"
    elif any(k in desc for k in ["Diseases"]):
        return "#888888", "Disease"
    else:
        return "#0279EE", "Homologous Recombination / DDR"

colors_and_cats = [assign_color(d) for d in df_plot["description"]]
bar_colors = [c[0] for c in colors_and_cats]
categories_present = list(dict.fromkeys([c[1] for c in colors_and_cats]))  # unique, ordered

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))
plt.subplots_adjust(left=0.45)  # large left margin for full labels

bars = ax.barh(range(len(df_plot)), df_plot["-log10fdr"],
               color=bar_colors, edgecolor="k", linewidth=0.4, alpha=0.85)

ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot["description"], fontsize=9)  # full names, no truncation
ax.set_xlabel("−log₁₀(FDR)", fontsize=11)
ax.set_title("Top Reactome/KEGG Pathway Enrichment\n"
             "STRING ORA on DDR & T-cell Gene Set (FDR < 0.05)", fontsize=12)

# x-axis: extend 10% beyond max bar
xmax = df_plot["-log10fdr"].max()
ax.set_xlim(0, xmax * 1.12)

# FDR threshold line
ax.axvline(x=-np.log10(0.05), color="red", linestyle="--", linewidth=1,
           alpha=0.7, label="FDR = 0.05")

# Gene count labels inside bars
for i, (_, row) in enumerate(df_plot.iterrows()):
    xpos = row["-log10fdr"] * 0.5
    ax.text(max(xpos, 0.5), i, f"n={int(row['number_of_genes'])}",
            va="center", ha="center", fontsize=8, color="white", fontweight="bold")

# Legend: only categories actually present in bars
color_map = {
    "Homologous Recombination / DDR": "#0279EE",
    "DNA Damage Signaling": "#E74C3C",
    "MMR": "#75A025",
    "Disease": "#888888",
}
cat_handles = [mpatches.Patch(facecolor=color_map[c], edgecolor="k", linewidth=0.5, label=c)
               for c in categories_present if c in color_map]
ax.legend(handles=cat_handles, title="Category", loc="lower right",
          fontsize=9, title_fontsize=9, framealpha=0.9)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig("/mnt/results/string_pathway_enrichment_plot.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/string_pathway_enrichment_plot.svg", bbox_inches="tight")
plt.close()
print("Saved: string_pathway_enrichment_plot.png / .svg")


Terms to plot: 15
                                                                            description  -log10fdr  number_of_genes
0                                                                       Mismatch Repair  12.677781                8
1                                   Regulation of TP53 Activity through Phosphorylation  19.970616               16
2   Resolution of D-loop Structures through Synthesis-Dependent Strand Annealing (SDSA)  20.896196               13
3                                                            G2/M DNA damage checkpoint  21.016825               16
4         Defective homologous recombination repair (HRR) due to BRCA1 loss of function  21.096367               13
5                                                    Transcriptional Regulation by TP53  21.798603               24
6                                             HDR through Single Strand Annealing (SSA)  23.072117               15
7         Defective homologous recombination repair (H

Build multi-evidence integration table and priority gene heatmap

In [185]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Evidence layer definitions (from all prior steps) ─────────────────────────
# Each gene gets a binary score per evidence layer + quantitative details

evidence = {
    # gene: {gwas, brain_de, tcell_de, eqtl, ppi_hub, pathway, notes}
    "RPA3": {
        "pathway": "HR",
        "gwas": True,        "gwas_detail": "rs6943429, p=1e-10 (Bellenguez 2022)",
        "brain_de": True,    "brain_de_detail": "log2FC=-2.69, padj=2.5e-5 (MTG, GSE5281)",
        "tcell_de": True,    "tcell_de_detail": "log2FC=-1.27, p=0.008 (pseudobulk, CellxGene)",
        "eqtl": True,        "eqtl_detail": "eGene in blood (q=1.1e-3), frontal cortex (q=2.4e-5), hippocampus (q=2.4e-3)",
        "ppi_hub": False,    "degree": 18,
    },
    "DNTT": {
        "pathway": "NHEJ",
        "gwas": True,        "gwas_detail": "rs6584063, p=7e-11 (Bellenguez 2022)",
        "brain_de": False,   "brain_de_detail": "Not significant in bulk brain DE",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": False,       "eqtl_detail": "Not an eGene in brain or blood (GTEx v8)",
        "ppi_hub": False,    "degree": 12,
    },
    "RAD51C": {
        "pathway": "HR",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": False,   "brain_de_detail": "Not significant in bulk brain DE",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": True,        "eqtl_detail": "Strongest eGene: blood q=1.6e-68, frontal cortex q=6.6e-30, hippocampus q=1.2e-7",
        "ppi_hub": True,     "degree": 43,
    },
    "ATM": {
        "pathway": "ATM/ATR",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": True,    "brain_de_detail": "Downregulated in AD brain (GSE5281)",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": True,        "eqtl_detail": "eGene in blood (q=5.7e-21)",
        "ppi_hub": True,     "degree": 42,
    },
    "MSH2": {
        "pathway": "MMR",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": True,    "brain_de_detail": "Dysregulated in AD brain (GSE5281)",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": True,        "eqtl_detail": "eGene in blood (q=3.5e-32), hippocampus (q=1.6e-2)",
        "ppi_hub": True,     "degree": 37,
    },
    "XRCC6": {
        "pathway": "NHEJ",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": False,   "brain_de_detail": "Not significant in bulk brain DE",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": True,        "eqtl_detail": "eGene in blood (q=3.5e-9), frontal cortex (q=4.3e-2)",
        "ppi_hub": True,     "degree": 43,
    },
    "PDCD1": {
        "pathway": "T-cell",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": False,   "brain_de_detail": "Not in bulk brain DDR DE",
        "tcell_de": True,    "tcell_de_detail": "Core exhaustion marker; elevated in AD T cells",
        "eqtl": True,        "eqtl_detail": "eGene in blood (q=1.3e-5)",
        "ppi_hub": False,    "degree": 8,
    },
    "HAVCR2": {
        "pathway": "T-cell",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": False,   "brain_de_detail": "Not in bulk brain DDR DE",
        "tcell_de": True,    "tcell_de_detail": "TIM-3; elevated in AD T cells",
        "eqtl": True,        "eqtl_detail": "eGene in blood (q=2.5e-5)",
        "ppi_hub": False,    "degree": 10,
    },
    "CHEK1": {
        "pathway": "ATM/ATR",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": False,   "brain_de_detail": "Not significant in bulk brain DE",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": True,        "eqtl_detail": "eGene in hippocampus (q=3.0e-4)",
        "ppi_hub": True,     "degree": 30,
    },
    "BRCA1": {
        "pathway": "HR",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": True,    "brain_de_detail": "Downregulated in AD brain (GSE5281)",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": False,       "eqtl_detail": "Not a significant eGene in tested tissues",
        "ppi_hub": True,     "degree": 42,
    },
    "PARP1": {
        "pathway": "BER",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": True,    "brain_de_detail": "Dysregulated in AD brain (GSE5281)",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": False,       "eqtl_detail": "Not a significant eGene in tested tissues",
        "ppi_hub": True,     "degree": 33,
    },
    "ERCC2": {
        "pathway": "NER",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": True,    "brain_de_detail": "Dysregulated in AD brain (GSE5281)",
        "tcell_de": False,   "tcell_de_detail": "Not in exhaustion signature",
        "eqtl": False,       "eqtl_detail": "Not a significant eGene in tested tissues",
        "ppi_hub": False,    "degree": 20,
    },
    "LAG3": {
        "pathway": "T-cell",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": False,   "brain_de_detail": "Not in bulk brain DDR DE",
        "tcell_de": True,    "tcell_de_detail": "Exhaustion marker; elevated in AD T cells",
        "eqtl": False,       "eqtl_detail": "Not a significant eGene in tested tissues",
        "ppi_hub": False,    "degree": 5,
    },
    "TIGIT": {
        "pathway": "T-cell",
        "gwas": False,       "gwas_detail": "No direct AD GWAS overlap",
        "brain_de": False,   "brain_de_detail": "Not in bulk brain DDR DE",
        "tcell_de": True,    "tcell_de_detail": "Exhaustion marker; elevated in AD T cells",
        "eqtl": False,       "eqtl_detail": "Not a significant eGene in tested tissues",
        "ppi_hub": False,    "degree": 4,
    },
}

# Build summary DataFrame
rows = []
for gene, ev in evidence.items():
    ev_count = sum([ev["gwas"], ev["brain_de"], ev["tcell_de"], ev["eqtl"], ev["ppi_hub"]])
    rows.append({
        "Gene": gene,
        "Pathway": ev["pathway"],
        "GWAS_overlap": ev["gwas"],
        "Brain_DE": ev["brain_de"],
        "Tcell_DE": ev["tcell_de"],
        "eQTL": ev["eqtl"],
        "PPI_hub": ev["ppi_hub"],
        "Evidence_count": ev_count,
        "PPI_degree": ev["degree"],
        "GWAS_detail": ev["gwas_detail"],
        "Brain_DE_detail": ev["brain_de_detail"],
        "Tcell_DE_detail": ev["tcell_de_detail"],
        "eQTL_detail": ev["eqtl_detail"],
    })

df_int = pd.DataFrame(rows).sort_values(["Evidence_count","PPI_degree"], ascending=False)
df_int.to_csv("/mnt/results/multi_evidence_integration.csv", index=False)
print("=== Multi-Evidence Integration Table ===")
print(df_int[["Gene","Pathway","GWAS_overlap","Brain_DE","Tcell_DE","eQTL","PPI_hub","Evidence_count"]].to_string(index=False))

# ── Evidence heatmap ──────────────────────────────────────────────────────────
ev_cols = ["GWAS_overlap","Brain_DE","Tcell_DE","eQTL","PPI_hub"]
col_labels = ["GWAS\nOverlap","Brain\nDE","T-cell\nDE","eQTL\n(GTEx v8)","PPI\nHub"]

hm_data = df_int.set_index("Gene")[ev_cols].astype(int)

pathway_color_map = {
    "HR": "#0279EE", "NHEJ": "#c8a800", "MMR": "#75A025",
    "BER": "#FF9400", "NER": "#9B59B6", "ATM/ATR": "#E74C3C", "T-cell": "#c040a0"
}
row_colors = [pathway_color_map[df_int.set_index("Gene").loc[g, "Pathway"]] for g in hm_data.index]

fig, ax = plt.subplots(figsize=(7, 8))
cmap = matplotlib.colors.ListedColormap(["#F0F0F0", "#2C3E50"])

im = ax.imshow(hm_data.values, cmap=cmap, aspect="auto", vmin=0, vmax=1)

# Grid lines
ax.set_xticks(np.arange(len(ev_cols)))
ax.set_xticklabels(col_labels, fontsize=10)
ax.set_yticks(np.arange(len(hm_data)))
ax.set_yticklabels(hm_data.index, fontsize=10, fontweight="bold")

# Color y-tick labels by pathway
for i, gene in enumerate(hm_data.index):
    pway = df_int.set_index("Gene").loc[gene, "Pathway"]
    ax.get_yticklabels()[i].set_color(pathway_color_map[pway])

# Add checkmarks / X in cells
for i in range(len(hm_data)):
    for j in range(len(ev_cols)):
        val = hm_data.values[i, j]
        symbol = "✓" if val == 1 else "–"
        color = "white" if val == 1 else "#AAAAAA"
        ax.text(j, i, symbol, ha="center", va="center", fontsize=13,
                color=color, fontweight="bold")

# Evidence count bar on right
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ev_counts = df_int["Evidence_count"].values
ax2.barh(np.arange(len(hm_data)), ev_counts, left=len(ev_cols) - 0.5,
         height=0.6, color=[pathway_color_map[p] for p in df_int["Pathway"]], alpha=0.0)
ax2.set_yticks([])
ax2.set_ylabel("")

# Add evidence count text
for i, cnt in enumerate(ev_counts):
    ax.text(len(ev_cols) + 0.1, i, f"  {cnt}/5", ha="left", va="center",
            fontsize=9, color="#333333", fontweight="bold")
ax.set_xlim(-0.5, len(ev_cols) + 0.8)

# Grid
ax.set_xticks(np.arange(len(ev_cols)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(hm_data)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

ax.set_title("Multi-Evidence Integration: DDR & T-cell Genes\nin Alzheimer's Disease", fontsize=12, pad=12)

# Pathway legend
pathway_handles = [mpatches.Patch(facecolor=c, edgecolor="k", linewidth=0.5, label=p)
                   for p, c in pathway_color_map.items()]
ax.legend(handles=pathway_handles, title="Pathway", loc="upper right",
          bbox_to_anchor=(1.35, 1.0), fontsize=8, title_fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.savefig("/mnt/results/multi_evidence_heatmap.png", dpi=150, bbox_inches="tight")
plt.savefig("/mnt/results/multi_evidence_heatmap.svg", bbox_inches="tight")
plt.close()
print("\nSaved: multi_evidence_heatmap.png / .svg")
print("Saved: multi_evidence_integration.csv")


=== Multi-Evidence Integration Table ===
  Gene Pathway  GWAS_overlap  Brain_DE  Tcell_DE  eQTL  PPI_hub  Evidence_count
  RPA3      HR          True      True      True  True    False               4
   ATM ATM/ATR         False      True     False  True     True               3
  MSH2     MMR         False      True     False  True     True               3
RAD51C      HR         False     False     False  True     True               2
 XRCC6    NHEJ         False     False     False  True     True               2
 BRCA1      HR         False      True     False False     True               2
 PARP1     BER         False      True     False False     True               2
 CHEK1 ATM/ATR         False     False     False  True     True               2
HAVCR2  T-cell         False     False      True  True    False               2
 PDCD1  T-cell         False     False      True  True    False               2
 ERCC2     NER         False      True     False False    False               1